# Diagnóstico de consumo de nitrógeno — comparación LAB004–012 y LAB019–021

**Objetivo:** distinguir consumo prematuro, incertidumbre de biomasa, restricciones del crecimiento y diferencias entre YAN analítico y el único estado N del modelo. Las secciones 1–22 comparan doce labs con figuras de varios paneles, sin utilizar mediciones de CO₂. La sección 23 añade la primera comparación gaseosa para LAB019–021 desde actividad detectada, conservando las trayectorias químicas. No se recalibran parámetros ni se introduce EKF.

Todas las simulaciones recorren **el experimento completo, con todos los pulsos registrados**. La división pre/post pulso corresponde sólo a métricas. Se mantienen las condiciones iniciales y relojes de los diagnósticos anteriores; no se incorpora un corrimiento de t₀ ni una latencia.

- **Históricos LAB004–012:** reproducción de datos que participaron en la calibración; no son un holdout. La biomasa histórica se usa para contraste, condicionada a su conversión de células a masa y a la interpretación de X como biomasa efectiva.
- **Recientes LAB019–021:** química Y15/Alcolyzer, temperatura y nutrición registradas. Biomasa Oculyze como proxy incierto, excluida de las métricas de validación. NUT con sus intervalos provisionales, sin inventar horas exactas.
- **theta_full vs incompleto + defaults:** referencias congeladas. Los escenarios indican explícitamente qué cambia; nunca se reajusta un parámetro para cada lab.
- **Remanente constante:** hipótesis exploratoria YAN_pred=N_disponible+B, con B común de 0/10/20/30 mg/L y N_disponible(0)=YAN(0)−B. No afirma que PAN residual sea no asimilable ni sustituye un modelo de dos pools.

Las figuras mantienen colores, unidades y leyendas consistentes. En las bandas/tablas de sensibilidad no se atribuye una incertidumbre estadística a factores elegidos manualmente. Las calibraciones y datos originales se protegen por hashes.

**GF final de LAB019:** se incluye 0 g/L a las 166 h como aproximación explícita autorizada por el usuario al resultado original negativo (−0,03 g/L en GLUCOSE-FRUCTOSE). El flag `GF_zero_substitution` y los valores originales quedan en la auditoría. El punto participa en figuras, errores, balances y barridos; no se modifican los archivos del Y15.

In [ ]:
from pathlib import Path
from dataclasses import replace
from datetime import datetime
import csv,hashlib,io,json,math,re,sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import openpyxl
from IPython import get_ipython
from IPython.display import display,Markdown,Image
ip=get_ipython()
if ip is not None: ip.run_line_magic('matplotlib','inline')
assert Path(sys.prefix).name=='fermentation',sys.executable
ROOT=next(p for p in [Path.cwd().resolve(),*Path.cwd().resolve().parents] if (p/'fermentation_model').exists())
FM=ROOT/'fermentation_model'; LAB2026=FM/'laboratory_2026'
sys.path.insert(0,str(FM));sys.path.insert(0,str(LAB2026/'notebooks/diagnostics'))
from shared import run_new_must_glycerol_estimability_doe as base
from laboratory_2026 import run_estimability_historical_by_medium as historical
import _nitrogen_diagnosis as diagnosis
OUT=LAB2026/'results/nitrogen_consumption_diagnosis'
FIGURES=OUT/'figures';FIGURES.mkdir(parents=True,exist_ok=True)
DATA_DIR=FM/'data/mem2026/LAB019-021'
LABS=('LAB019','LAB020','LAB021')
HIST_LABS=tuple(f'LAB{i:03d}' for i in range(4,13))
ALL_LABS=(*HIST_LABS,*LABS)
GROUPS=[HIST_LABS[i:i+3] for i in range(0,9,3)]+[LABS]
T0=pd.Timestamp('2026-09-21 17:00:00')
FERMENTER={'LAB019':'F1','LAB020':'F2','LAB021':'F3'}
CAL=LAB2026/'results/estimability_historical_natural'
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'font.family':'DejaVu Sans','font.size':9,'figure.dpi':105})
pd.set_option('display.max_rows',45)
def sha256(path):
    digest=hashlib.sha256()
    with open(path,'rb') as handle:
        for block in iter(lambda:handle.read(1024*1024),b''): digest.update(block)
    return digest.hexdigest()
data_source=historical.SECONDARY_DATA_PATH if historical.SECONDARY_DATA_PATH.exists() else base.NORMALIZED_DATA_PATH
PROTECTED=[data_source,CAL/'process_temperature_30min.csv',CAL/'theta_natural_full.csv',
    CAL/'theta.csv',CAL/'core_fit_summary.csv',CAL/'core_fit_metrics_by_batch_state.csv',
    FM/'shared/results/new_must_glycerol_overnight_validation/theta_multistart_07.csv',
    FM/'shared/run_new_must_glycerol_estimability_doe.py',
    LAB2026/'run_estimability_historical_by_medium.py',LAB2026/'nut_sample_audit.py',
    LAB2026/'notebooks/theta_full_vs_incomplete_defaults_natural_review.ipynb',
    LAB2026/'notebooks/EKF/post_activity_prediction_comparison.source.ipynb',
    LAB2026/'notebooks/diagnostics/_nitrogen_diagnosis.py',
    DATA_DIR/'Mediciones-Offline-LAB019-021.xlsx',DATA_DIR/'backup_global_temperatura.csv']
PROTECTED+=sorted(DATA_DIR.glob('EXPAuto*.txt'))+sorted(DATA_DIR.glob('ONLINE*.txt'))+sorted(DATA_DIR.glob('Oculyze*.csv'))
assert all(p.exists() for p in PROTECTED)
HASH_BEFORE={str(p.relative_to(ROOT)):sha256(p) for p in PROTECTED}
print('Python:',sys.executable)
print('Datos históricos:',data_source)
print('No se abre backup_global_co2.csv; no se ejecuta optimización.')

## 1. Qué impone la ecuación y qué está congelado

La RHS oficial utiliza
\[
\dot N=-q_N a_\mu(T)\frac{N}{N+K_N(T)}X,\quad
\mu=\mu_0 a_\mu(T)\frac{N}{N+K_N(T)},\quad K_N(T)=\frac{\mu_0}{s_N}a_K(T).
\]
Así, entre aportes de N, exige
\[
N+\frac{q_N}{\mu_0}(X+X_d)-N_{\rm añadido}=\mathrm{constante}.
\]
La rapidez de captación, la biomasa efectiva y el rendimiento están ligados. Reducir qN cambia también el rendimiento μ₀/qN. El escenario de rapidez escala **μ₀, qN y sN juntos**, conservando ese rendimiento y K_N de referencia. Cambiar sN modifica la saturación; cambiar T modifica todas las tasas térmicas, no sólo N.

El estado N no contiene ecuaciones separadas de PAN y amonio ni un término de liberación. Un aumento medido sin entrada registrada no puede reproducirse exactamente con un consumo puro. Se identificará, sin asumir que la causa sea un error analítico.

theta_full reconstruye el vector usado en la calibración: once valores naturales reestimados y seis heredados que se mantuvieron fijos; no corresponde a un reajuste conjunto de diecisiete parámetros sobre estos doce labs. Completar el CSV incompleto con defaults produce una variante distinta, no recupera necesariamente la calibración original. La capa de CO₂ y sus parámetros no intervienen en este análisis químico.

In [ ]:
full_table=pd.read_csv(CAL/'theta_natural_full.csv')
THETA_FULL=full_table.set_index('parameter').theta.astype(float).to_dict()
partial=pd.read_csv(CAL/'theta.csv').set_index('parameter').theta.astype(float).to_dict()
THETA_DEFAULTS={**base.DEFAULT_THETA,**partial}
THETA_DEFAULTS={s:float(THETA_DEFAULTS[s]) for s in base.FULL17}
THETA_SETS={'theta_full':THETA_FULL,'theta_defaults':THETA_DEFAULTS}
assert set(THETA_FULL)==set(base.FULL17)
DIFFERENT=('sN','qXG','qXF','sG','sF','m0')
assert {k for k in base.FULL17 if not np.isclose(THETA_FULL[k],THETA_DEFAULTS[k],rtol=1e-12)}==set(DIFFERENT)
prior=pd.read_csv(FM/'shared/results/new_must_glycerol_overnight_validation/theta_multistart_07.csv',index_col=0).iloc[:,0].to_dict()
assert all(np.isclose(THETA_FULL[k],prior[k]) for k in DIFFERENT)
PARAMETERS=full_table.copy()
PARAMETERS['theta_defaults']=[THETA_DEFAULTS[k] for k in PARAMETERS.parameter]
display(PARAMETERS)
display(pd.DataFrame([{'theta':name,'qN':th['qN'],'mu0':th['mu0'],
    'qN_over_mu0_gN_gDW':th['qN']/th['mu0'],'biomass_yield_gDW_gN':th['mu0']/th['qN']}
    for name,th in THETA_SETS.items()]))
display(pd.DataFrame([{'scenario':name,'kind':kind,'value':value}
    for name,(kind,value) in diagnosis.SCENARIOS.items()]))

## 2. Preparación de química reciente y NUT
Se reproduce visiblemente la preparación ya auditada: horas de extracción del Excel, exports Y15, corrección de etanol diluido m12, conversiones y biomasa Oculyze. **Excepción explícita solicitada por el usuario:** el GF final de LAB019-12 (166 h) se representa y puntúa como **0 g/L**, aproximando su resultado GLUCOSE-FRUCTOSE de −0,03 g/L; el otro ensayo dio −1,85 g/L. Ambos resultados originales se conservan junto a un flag de sustitución. Es un valor de trabajo, no un cero exacto medido ni un límite de detección conocido. Los demás resultados ausentes/fuera de rango conservan su tratamiento anterior. Se conserva el mapeo histórico de m1 a t=0, aunque su extracción ocurrió 1,17 h antes de inocular. Las muestras NUT no se convierten en mediciones puntuales con hora cierta.

In [ ]:
from shared.new_must_data_loader import ETHANOL_G_L_PER_PERCENT_VV


def numeric(value):
    try:
        out = float(value)
        return out if np.isfinite(out) else np.nan
    except (TypeError, ValueError):
        return np.nan


def load_offline_samples():
    path = DATA_DIR / "Mediciones-Offline-LAB019-021.xlsx"
    wb = openpyxl.load_workbook(path, read_only=True, data_only=True)
    rows = []
    for excel_row, row in enumerate(wb["MedicionesOffline"].iter_rows(min_row=2, values_only=True), start=2):
        fecha, hora, lab, brix, density = row[0], row[1], row[2], row[5], row[6]
        if lab is None and brix is None and density is None:
            continue
        if lab not in LABS or fecha is None or hora is None or row[4] is None:
            raise ValueError(f"MedicionesOffline fila {excel_row}: metadatos de muestra incompletos")
        hh = hora.time() if isinstance(hora, datetime) else hora
        rows.append({
            "lab": str(lab), "sample": str(row[4]).strip(),
            "sample_timestamp": datetime.combine(fecha.date() if hasattr(fecha, "date") else fecha, hh),
            "Brix": numeric(brix), "density": numeric(density),
            "Etanol_pct_vv": numeric(row[10]), "metadata_fallback": False,
        })
    wb.close()
    result = pd.DataFrame(rows)
    assert not result.duplicated(["lab", "sample"]).any(), "muestras offline duplicadas"
    return result

def load_y15():
    rows = []
    paths = sorted(DATA_DIR.glob("EXPAuto*.txt")) + sorted(DATA_DIR.glob("ONLINE*.txt"))
    for path in paths:
        for line in open(path, encoding="utf-8-sig", errors="replace"):
            fields = line.rstrip("\n").split("\t")
            if len(fields) < 6:
                continue
            raw = fields[3].strip()
            value = float(raw.replace(",", ".")) if re.fullmatch(r"-?\d+(,\d+)?", raw) else np.nan
            rows.append({"sample_id": fields[0], "analyte": fields[1], "value": value, "raw_value": raw, "analysis_timestamp": fields[5], "source_file": path.name})
    long = pd.DataFrame(rows).drop_duplicates(["sample_id", "analyte"], keep="first")
    wide = long.pivot(index="sample_id", columns="analyte", values="value")
    raw_wide = long.pivot(index="sample_id", columns="analyte", values="raw_value")
    return long, wide, raw_wide


offline = load_offline_samples()
# Dilución Alcolyzer m12, distinta por lote (cuaderno de laboratorio): LAB020 27.5 mL de vino + 10 mL de agua;
# LAB021 32.5 mL de vino + 5 mL de agua (ambas a 37.5 mL). Sólo corrige etanol (medido en la mezcla);
# Brix y densidad se midieron antes de diluir.
DILUTION_FACTORS_M12 = {"LAB020": 37.5 / 27.5, "LAB021": 37.5 / 32.5}
for _lab_m12, _factor in DILUTION_FACTORS_M12.items():
    _sel = offline["lab"].eq(_lab_m12) & offline["sample"].eq("12")
    offline.loc[_sel, "Etanol_pct_vv"] = offline.loc[_sel, "Etanol_pct_vv"] * _factor
assert np.isclose(offline.loc[(offline["lab"] == "LAB020") & (offline["sample"] == "12"), "Etanol_pct_vv"].iloc[0], 7.33 * 15 / 11)
assert np.isclose(offline.loc[(offline["lab"] == "LAB021") & (offline["sample"] == "12"), "Etanol_pct_vv"].iloc[0], 8.66 * 15 / 13)
y15_long, y15, y15_raw = load_y15()

chem_rows = []
for lab in LABS:
    samples = offline[offline["lab"].eq(lab)].copy().set_index("sample")
    numbered_samples = sorted((item for item in samples.index if str(item).isdigit()), key=int)
    assert "1" in numbered_samples, f"{lab}: falta muestra inicial"
    for sample in numbered_samples:
        sid = f"{lab}-{sample}"
        meta = samples.loc[sample]
        stamp = pd.Timestamp(meta["sample_timestamp"])
        sample_t_h = (stamp - T0).total_seconds() / 3600.0
        t_h = 0.0 if sample == "1" else sample_t_h
        get = lambda name: float(y15.loc[sid, name]) if sid in y15.index and name in y15.columns and pd.notna(y15.loc[sid, name]) else np.nan
        get_raw = lambda name: str(y15_raw.loc[sid, name]) if sid in y15_raw.index and name in y15_raw.columns and pd.notna(y15_raw.loc[sid, name]) else ""
        gf_320_raw = get("GLU-FRU-320")
        gf_low_raw = get("GLUCOSE-FRUCTOSE")
        if np.isfinite(gf_320_raw) and gf_320_raw >= 0.0:
            gf, gf_source = gf_320_raw, "GLU-FRU-320"
        elif np.isfinite(gf_low_raw) and gf_low_raw >= 0.0:
            gf, gf_source = gf_low_raw, "GLUCOSE-FRUCTOSE"
        else:
            gf, gf_source = np.nan, ""
        gf_zero_substitution = bool(lab == "LAB019" and str(sample) == "12"
            and not np.isfinite(gf) and np.isfinite(gf_low_raw) and gf_low_raw < 0.0)
        if gf_zero_substitution:
            # User-authorized working value; retain both negative assay results below.
            gf, gf_source = 0.0, "GLUCOSE-FRUCTOSE: resultado final negativo aproximado a 0"
        glucose_320_raw = get("GLUCOSE-320")
        glucose_low_raw = get("GLUCOSE")
        glucose_raw = glucose_320_raw if np.isfinite(glucose_320_raw) else glucose_low_raw
        glucose = glucose_raw if np.isfinite(glucose_raw) and glucose_raw >= 0.0 else np.nan
        fructose = gf - glucose if np.isfinite(gf) and np.isfinite(glucose) and gf >= glucose else np.nan
        pan, ammonia_raw = get("PAN"), get("AMMONIA")
        ammonia = max(ammonia_raw, 0.0) if np.isfinite(ammonia_raw) else np.nan
        yan = pan + 0.823 * ammonia if np.isfinite(pan) and np.isfinite(ammonia) else np.nan
        chem_rows.append({
            "lab": lab, "sample": sample, "sample_timestamp": stamp, "sample_t_h": sample_t_h, "t_h": t_h,
            "metadata_fallback": bool(meta["metadata_fallback"]), "density": float(meta["density"]), "Brix": float(meta["Brix"]),
            "GF_g_l": gf, "GF_source": gf_source, "GF_zero_substitution": gf_zero_substitution,
            "GF_320_raw_g_l": gf_320_raw, "GF_low_raw_g_l": gf_low_raw,
            "GF_320_below_zero": bool(np.isfinite(gf_320_raw) and gf_320_raw < 0.0),
            "G_g_l": glucose, "G_raw_g_l": glucose_raw, "F_g_l": fructose,
            "PAN_mg_l": pan, "NH3_raw_mg_l": ammonia_raw, "NH3_censored_mg_l": ammonia,
            "YAN_mg_l": yan, "N_kg_m3": yan / 1000.0 if np.isfinite(yan) else np.nan,
            "Gly_g_l": get("GLYCEROL"), "Pyr_mg_l": get("PYRUVIC ACID"),
            "Etanol_pct_vv": float(meta["Etanol_pct_vv"]),
            "E_g_l": meta["Etanol_pct_vv"] * ETHANOL_G_L_PER_PERCENT_VV if np.isfinite(meta["Etanol_pct_vv"]) else np.nan,
            "AcAld_mg_l": get("ACETALDEHID-1000"), "Acetate_g_l": get("ACETIC ACID"),
            "G_raw_text": get_raw("GLUCOSE-320") or get_raw("GLUCOSE"),
            "GF_raw_text": get_raw("GLU-FRU-320"),
            "GF_low_raw_text": get_raw("GLUCOSE-FRUCTOSE"),
            "NH3_below_zero": bool(np.isfinite(ammonia_raw) and ammonia_raw < 0.0),
        })
from laboratory_2026.nut_sample_audit import load_nut_samples
NUT_SAMPLE_TIMES = {}  # completar sólo con horas de extracción confirmadas
NUT_SAMPLES = load_nut_samples(DATA_DIR, {lab:T0 for lab in LABS}, NUT_SAMPLE_TIMES)
NUT_SAMPLES.to_csv(OUT/'nut_sample_audit.csv', index=False)
for nut in NUT_SAMPLES.itertuples():
    chem_rows.append(dict(lab=nut.lab, sample='NUT', sample_timestamp=nut.sample_timestamp,
        sample_t_h=nut.plot_t_h, t_h=nut.plot_t_h, metadata_fallback=nut.time_provisional,
        PAN_mg_l=nut.PAN_mg_l, NH3_raw_mg_l=nut.NH3_raw_mg_l,
        NH3_censored_mg_l=nut.NH3_censored_mg_l, YAN_mg_l=nut.YAN_mg_l, N_kg_m3=nut.N_kg_m3,
        YAN_reported_mg_l=nut.YAN_reported_mg_l, NH3_below_zero=False,
        time_lower_h=nut.time_lower_h, time_upper_h=nut.time_upper_h,
        time_provisional=nut.time_provisional, time_source=nut.time_source))
chem = pd.DataFrame(chem_rows).sort_values(["lab", "t_h"]).reset_index(drop=True)
chem['is_NUT'] = chem['sample'].eq('NUT')
chem['GF_zero_substitution'] = chem['GF_zero_substitution'].fillna(False).astype(bool)
chem['time_provisional'] = chem['time_provisional'].fillna(False).astype(bool)
chem['time_lower_h'] = chem['time_lower_h'].fillna(chem.t_h)
chem['time_upper_h'] = chem['time_upper_h'].fillna(chem.t_h)
chem['time_source'] = chem['time_source'].fillna('offline extraction timestamp; m1 mapped to t0')
chem["is_initial"] = chem["sample"].eq("1")
chem["soft_flag_GF"] = chem["lab"].eq("LAB019") & chem["sample"].eq("4")
chem["soft_flag_metadata"] = chem["metadata_fallback"]
chem["y15_available"] = chem[["GF_g_l", "YAN_mg_l", "Gly_g_l", "Pyr_mg_l", "AcAld_mg_l"]].notna().any(axis=1)
assert chem[chem["sample"].eq("1")]["E_g_l"].notna().all(), "falta Etanol m1 (inicializa E0)"

assert np.allclose(chem[chem["sample"].eq("1")]["t_h"], 0.0)
assert chem.query("lab == 'LAB020' and sample == '4'")["G_g_l"].isna().all()
assert chem.query("lab == 'LAB020' and sample == '4'")["G_raw_g_l"].iloc[0] == -7.28
assert chem[chem["sample"].eq("12")]["y15_available"].all(), "falta exportación Y15 m12"
assert chem.loc[chem["sample"].eq("12"), "GF_320_below_zero"].all(), "revisar QA de GF m12"
display(chem[["lab", "sample", "sample_t_h", "t_h", "GF_g_l", "GF_source", "GF_zero_substitution", "G_g_l", "F_g_l", "YAN_mg_l", "Gly_g_l", "E_g_l", "Pyr_mg_l", "AcAld_mg_l", "y15_available", "NH3_below_zero", "metadata_fallback"]])
print("Muestras sin resultado Y15:", chem.loc[~chem["y15_available"], ["lab", "sample"]].to_dict("records"))

# Todas las mediciones Oculyze: concentración total en millones de células/mL,
# conversión de 30 pg/célula y viabilidad porcentual para comparar con X viable.
from shared.new_must_data_loader import MILLION_CELLS_ML_TO_KG_M3
biomass_rows=[]
for lab in LABS:
    report_path=DATA_DIR/f'Oculyze_report_{lab}.csv'
    with open(report_path,encoding='utf-8-sig') as handle:
        report_rows=list(csv.DictReader(handle))
    for row in report_rows:
        sample_id=row['Name'].strip().upper()
        if sample_id=='CERO':
            continue  # Se conserva la convención primaria de concentración cruda.
        match=re.fullmatch(rf'{lab}-(\d+)',sample_id)
        if match is None:
            raise ValueError(f'{report_path.name}: identificador inesperado {sample_id}')
        sample=str(int(match.group(1)))
        meta=chem[chem.lab.eq(lab)&chem['sample'].eq(sample)]
        assert len(meta)==1,(lab,sample)
        meta=meta.iloc[0]
        concentration=float(row['Concentration'])
        viability=float(row['Viability'])
        assert np.isfinite(concentration) and concentration>=0
        assert np.isfinite(viability) and 0<=viability<=100
        total=concentration*MILLION_CELLS_ML_TO_KG_M3
        viable=total*viability/100.
        biomass_rows.append(dict(lab=lab,sample=sample,sample_id=sample_id,
            sample_timestamp=meta.sample_timestamp,sample_t_h=float(meta.sample_t_h),
            t_h=float(meta.t_h),concentration_mcells_ml=concentration,viability_pct=viability,
            X_total_g_l=total,X_viable_g_l=viable,Xd_g_l=total-viable,
            source_file=report_path.name))
BIOMASS_OBSERVATIONS=pd.DataFrame(biomass_rows).sort_values(['lab','t_h']).reset_index(drop=True)
assert not BIOMASS_OBSERVATIONS.duplicated(['lab','sample']).any()
assert BIOMASS_OBSERVATIONS.groupby('lab').size().to_dict()=={lab:11 for lab in LABS}
assert np.allclose(BIOMASS_OBSERVATIONS.X_total_g_l,
                   BIOMASS_OBSERVATIONS.X_viable_g_l+BIOMASS_OBSERVATIONS.Xd_g_l)
chem=chem.merge(BIOMASS_OBSERVATIONS[['lab','sample','X_total_g_l','X_viable_g_l','Xd_g_l']],
                on=['lab','sample'],how='left',validate='one_to_one')
display(BIOMASS_OBSERVATIONS[['lab','sample','sample_t_h','t_h','concentration_mcells_ml',
                             'viability_pct','X_total_g_l','X_viable_g_l','Xd_g_l']].round(5))
print('Mediciones Oculyze por lote:',BIOMASS_OBSERVATIONS.groupby('lab').size().to_dict())

assert chem[chem.is_NUT].lab.nunique() == 3
assert chem.loc[chem.is_NUT, ['GF_g_l','G_g_l','F_g_l','Gly_g_l','E_g_l','X_viable_g_l']].isna().all().all()
chem.to_csv(OUT/'chemistry_samples_with_NUT.csv', index=False)
display(NUT_SAMPLES[['lab','YAN_reported_mg_l','YAN_mg_l','pulse_timestamp',
                     'analysis_PAN','analysis_YAN','sample_timestamp','time_lower_h','plot_t_h','time_upper_h']])
def draw_nut_state(ax, lab, state):
    if state != 'N': return
    nut = NUT_SAMPLES[NUT_SAMPLES.lab.eq(lab)].iloc[0]
    x = float(nut.plot_t_h)
    ax.errorbar([x],[nut.N_kg_m3],
        xerr=[[x-nut.time_lower_h],[nut.time_upper_h-x]],fmt='*',color='#8e24aa',
        ms=8,capsize=3,zorder=8,label='NUT · tiempo provisional' if nut.time_provisional else 'NUT · hora confirmada')

GF_ZERO_AUDIT=chem.loc[chem.GF_zero_substitution,
    ['lab','sample','t_h','GF_g_l','GF_source','GF_320_raw_g_l','GF_low_raw_g_l','GF_raw_text','GF_low_raw_text']].copy()
assert len(GF_ZERO_AUDIT)==1
assert GF_ZERO_AUDIT.iloc[0].lab=='LAB019' and GF_ZERO_AUDIT.iloc[0]['sample']=='12'
assert GF_ZERO_AUDIT.iloc[0].t_h==166. and GF_ZERO_AUDIT.iloc[0].GF_g_l==0.
assert GF_ZERO_AUDIT.iloc[0].GF_320_raw_g_l==-1.85 and GF_ZERO_AUDIT.iloc[0].GF_low_raw_g_l==-.03
GF_ZERO_POLICY=dict(lab='LAB019',sample='12',time_h=166.,working_GF_g_l=0.,
    raw_GLUC_FRU_320_g_l=-1.85,raw_GLUCOSE_FRUCTOSE_g_l=-.03,
    status='User-authorized zero approximation from negative final assay; not an exact measured zero',
    scope='GF only in this notebook; raw G/F and original files preserved',
    included_in_metrics_balances_and_grids=True)
GF_ZERO_AUDIT.to_csv(OUT/'GF_final_zero_audit.csv',index=False)
(OUT/'GF_final_zero_policy.json').write_text(json.dumps(GF_ZERO_POLICY,ensure_ascii=False,indent=2),encoding='utf-8')
display(GF_ZERO_AUDIT)


## 3. Grupo histórico, temperatura y eventos
Los históricos usan el mismo CSV normalizado y la temperatura archivada de la calibración. LAB019–021 usan temperatura registrada y el pulso 2 de la hoja Nutricion; la nutrición inicial ya está contenida en las condiciones iniciales. En los históricos se conserva el libro de eventos de la calibración, cuya suficiencia se comprueba con los balances observados. No se corrigen dosis para hacer coincidir las curvas.

Las simulaciones van de t=0 al último tiempo químico de cada lab. La malla incorpora cada muestra, cada pulso y un punto inmediatamente posterior para dibujar los saltos y evaluar NUT sin interpolar una rampa artificial.

In [ ]:
historical_data=historical.load_historical_medium_data('natural')
historical_batches,temperature_alignment=historical.augment_natural_batches_with_temperature(
    base.make_batches(historical_data),pd.read_csv(CAL/'process_temperature_30min.csv'))
BATCHES={b.batch:b for b in historical_batches if b.batch in HIST_LABS}
assert tuple(BATCHES)==HIST_LABS
wb=openpyxl.load_workbook(DATA_DIR/'Mediciones-Offline-LAB019-021.xlsx',read_only=True,data_only=True)
nutrition=[r for r in wb['Nutricion'].iter_rows(min_row=2,values_only=True) if r[0] in LABS and r[2]=='Pulso 2']
wb.close()
PULSES={r[0]:(((pd.Timestamp(r[3])-T0).total_seconds()/3600.,float(r[4])/1000.),) for r in nutrition}
assert set(PULSES)==set(LABS)
new_batches,recent_audit=diagnosis.load_recent_batches(chem,DATA_DIR/'backup_global_temperatura.csv',T0,FERMENTER,PULSES)
BATCHES.update(new_batches)
POINTS=diagnosis.observation_table(BATCHES,chem)
POINTS['zero_substitution']=POINTS.lab.eq('LAB019')&POINTS.state.eq('GF')&POINTS.t_h.eq(166.)
assert POINTS.zero_substitution.sum()==1
assert POINTS.loc[POINTS.zero_substitution,'observed'].iloc[0]==0.
assert len(POINTS.loc[POINTS.lab.eq('LAB019')&POINTS.state.eq('GF')&~POINTS.initial_anchor])==11
COMPONENTS=diagnosis.analytical_components(historical_data,chem)
assert COMPONENTS.sum_check.abs().max()<1e-8
INPUT_ROWS=[]
for lab,b in BATCHES.items():
    INPUT_ROWS.append(dict(lab=lab,cohort='historico' if lab in HIST_LABS else 'reciente',
        start_h=float(b.time[0]),end_h=float(b.time[-1]),N_pulses=repr(b.pulses['N']),
        biomass_validation=lab in HIST_LABS,temperature_min=float(b.temperature_c.min()),
        temperature_max=float(b.temperature_c.max()),**b.initials))
INPUTS=pd.DataFrame(INPUT_ROWS)
OBSERVED_BALANCE=diagnosis.observed_balance(POINTS,BATCHES)
BIOMASS_BALANCE=diagnosis.biomass_balance(POINTS,BATCHES,THETA_FULL)
display(INPUTS)
display(recent_audit)
display(OBSERVED_BALANCE.query('excess_mg_l>20').round(3))
print('Conversión histórica: PAN + 0.82×NH informado; reciente: PAN + 0.823×NH no negativo.')
print('Diferencia máxima por 0.82→0.823:',COMPONENTS.factor_change_082_to_0823_mg_l.abs().max(),'mg/L')
print('Esa diferencia no explica discrepancias de decenas o cientos de mg/L.')

## 4. Escenarios completos y verificaciones
Se integra la RHS oficial con tolerancias estrictas y saltos exactos. Para cada lab y theta se calculan catorce escenarios: referencia, biomasa, captación, rapidez con rendimiento fijo, saturación, temperatura y remanente. Las tablas puntúan mediciones posteriores al ancla inicial, sobre el mismo soporte en todos los escenarios. Se reproduce el recorte a límites de la salida oficial, pero se conservan mínimos/máximos internos: un escenario que requiere recortar concentraciones fuera de rango se marca como físicamente no admisible. Sus errores son descriptivos, no una solución aceptable.

GF se representa para facilitar la lectura, pero no se cuenta como otra observación independiente de G y F en la auditoría de calibración. En el resumen de otras variables de los recientes se utiliza GF/E/Gly, excluyendo X/Xd y evitando contar G/F/GF simultáneamente. Su escala GF=5 g/L es exploratoria; las escalas históricas son pesos del ajuste, no precisiones certificadas del Y15.

In [ ]:
CURVES={}; METRIC_ROWS=[]; CHECK_ROWS=[]; FEATURES=[]; MECHANISMS=[]
for theta_name,th in THETA_SETS.items():
    for lab,batch in BATCHES.items():
        points=POINTS[POINTS.lab.eq(lab)]
        pulse=min((t for t,d in batch.pulses['N']),default=np.inf)
        obs_cross=diagnosis.observed_crossing(points,pulse)
        for name in diagnosis.SCENARIOS:
            curve=diagnosis.run(batch,th,name)
            CURVES[(lab,theta_name,name)]=curve
            for row in diagnosis.score(curve,points,pulse):
                METRIC_ROWS.append(dict(lab=lab,cohort='historico' if lab in HIST_LABS else 'reciente',
                    theta=theta_name,scenario=name,physically_admissible=curve.attrs['physically_admissible'],**row))
            pre=curve.loc[curve.index<=pulse]
            FEATURES.append(dict(lab=lab,theta=theta_name,scenario=name,
                first_available_N_le_5_h=diagnosis.first_crossing(pre.index,pre.N),
                first_predicted_YAN_le_5_h=diagnosis.first_crossing(pre.index,pre.N_pred),
                background_mg_l=1000*curve.attrs['background'],
                n_YAN_below_background=int((points.loc[points.state.eq('N'),'observed']<curve.attrs['background']).sum()),
                final_YAN_pred_mg_l=1000*curve.N_pred.iloc[-1],**obs_cross))
            CHECK_ROWS.append(dict(lab=lab,theta=theta_name,scenario=name,
                inventory_drift_mg_l=1000*curve.attrs['inventory_drift'],
                jump_error=curve.attrs['jump_error'],final_h=float(curve.index[-1]),
                initial_YAN_error_mg_l=1000*abs(curve.N_pred.iloc[0]-batch.initials['N']),
                min_state=float(curve[list(base.STATE_NAMES)].min().min())))
            CHECK_ROWS[-1].update(physically_admissible=curve.attrs['physically_admissible'],
                **{'raw_min_'+s:v for s,v in curve.attrs['raw_min'].items()},
                **{'raw_max_'+s:v for s,v in curve.attrs['raw_max'].items()})
        ref=CURVES[(lab,theta_name,'referencia')]
        official=base.simulate(batch,th,ref.index.to_numpy(float))
        assert official is not None
        delta=float(np.max(np.abs(official.to_numpy()-ref[list(base.STATE_NAMES)].to_numpy())))
        assert delta<.01,(lab,theta_name,delta)
        CHECK_ROWS.append(dict(lab=lab,theta=theta_name,scenario='official_parity',
                              official_max_error_g_l=delta,final_h=float(ref.index[-1])))
        MECHANISMS.append(dict(lab=lab,theta=theta_name,X0=batch.initials['X'],
            N0_mg_l=1000*batch.initials['N'],T0=ref.T_C.iloc[0],KN0_mg_l=ref.KN_mg_l.iloc[0],
            n_lim0=ref.n_lim.iloc[0],initial_uptake_mg_l_h=ref.uptake_mg_l_h.iloc[0],
            initial_mu_h=ref.mu_h.iloc[0],new_biomass_budget_g_l=batch.initials['N']*th['mu0']/th['qN']))
        print(theta_name,lab,'14 escenarios hasta',batch.time[-1],'h',flush=True)
METRICS=pd.DataFrame(METRIC_ROWS);CHECKS=pd.DataFrame(CHECK_ROWS)
FEATURES=pd.DataFrame(FEATURES);MECHANISMS=pd.DataFrame(MECHANISMS)
assert len(CURVES)==336
assert CHECKS.initial_YAN_error_mg_l.dropna().max()<1e-8
assert CHECKS.inventory_drift_mg_l.dropna().max()<.002
display(MECHANISMS.round(5))
display(CHECKS.groupby(['theta','scenario'])[['inventory_drift_mg_l','jump_error','official_max_error_g_l']].max())
display(CHECKS.loc[CHECKS.physically_admissible.eq(False),
    ['lab','theta','scenario','raw_min_G','raw_min_F','raw_min_N']])

## 5. Convenciones de las figuras
Negro: química medida. Azul: referencia theta_full. Naranja discontinuo: referencia incompleta con defaults. Los escenarios de reducción y aumento aparecen en verde y rojo. Morado: nutrición registrada y muestras NUT; NUT se muestra con un punto morado; su intervalo temporal se documenta en la preparación de datos. Las muestras iniciales son anclas y no se incluyen en RMSE posterior.

Cada figura contiene varios paneles. Todas las curvas se muestran hasta el final de su lab. Las columnas de PAN y amonio muestran datos, no predicciones de componentes que el modelo no tiene. **†** en una curva indica que el integrador tuvo estados internos fuera de límites físicos y la salida oficial necesitó recorte: no se considera una solución físicamente admisible.

In [ ]:
THETA_COLOR={'theta_full':'#2166ac','theta_defaults':'#d95f02'}
THETA_STYLE={'theta_full':'-','theta_defaults':'--'}
SC_COLOR={'referencia':'#2166ac','X0_x0.5':'#00897b','X0_x2':'#c62828',
    'qN_x0.5':'#ef6c00','qN_x2':'#c62828','crecimiento_x0.8':'#7b1fa2','crecimiento_x1.2':'#c62828',
    'sN_x0.5':'#00897b','sN_x2':'#c62828','T_menos2C':'#00897b','T_mas2C':'#c62828',
    'remanente_10':'#00897b','remanente_20':'#795548','remanente_30':'#7b1fa2'}
FIGURE_LOG=[]
def save(fig,name):
    path=FIGURES/(name+'.png')
    fig.savefig(path,dpi=125,bbox_inches='tight',facecolor='white')
    buffer=io.BytesIO();fig.savefig(buffer,format='png',dpi=105,bbox_inches='tight',facecolor='white')
    plt.close(fig);display(Image(data=buffer.getvalue()));buffer.close()
    FIGURE_LOG.append(dict(name=name,path=str(path.relative_to(ROOT))))
def common_legend(fig,axes,ncol=5):
    unique={}
    for ax in np.asarray(axes).flat:
        h,l=ax.get_legend_handles_labels()
        for handle,label in zip(h,l): unique.setdefault(label,handle)
    fig.legend(unique.values(),unique.keys(),loc='outside upper center',ncol=ncol,fontsize=8)
def mark_events(ax,lab):
    for i,(t,d) in enumerate(BATCHES[lab].pulses['N']):
        ax.axvline(t,color='#ad1457',ls=':',lw=1,label='Pulso N registrado' if i==0 else None)
def draw_nut(ax,lab,scale=1000.):
    if lab not in LABS: return
    nut=NUT_SAMPLES[NUT_SAMPLES.lab.eq(lab)].iloc[0]
    ax.scatter([nut.plot_t_h],[nut.N_kg_m3*scale],s=20,color='#8e24aa',
        marker='o',zorder=8,label='NUT')

def draw_nut_component(ax,lab,column):
    if lab not in LABS: return
    nut=NUT_SAMPLES[NUT_SAMPLES.lab.eq(lab)].iloc[0]
    value=nut.PAN_mg_l if column=='PAN' else .823*nut.NH3_censored_mg_l
    ax.scatter([nut.plot_t_h],[value],s=20,color='#8e24aa',marker='o',zorder=8,label='NUT')

def draw_observations(ax,lab,state,scale=1.):
    pts=POINTS.loc[POINTS.lab.eq(lab)&POINTS.state.eq(state)]
    proxy=lab in LABS and state in ('X','Xd')
    ax.scatter(pts.t_h,pts.observed*scale,s=20,marker='o',
        facecolors='none' if proxy else 'black',edgecolors='gray' if proxy else 'black',
        zorder=6,clip_on=False,label='Biomasa' if proxy else 'Medición')
    if state=='N': draw_nut(ax,lab,scale)

def group_name(group): return group[0]+'_'+group[-1]
def case_label(lab,theta,name):
    return name+(' †' if not CURVES[(lab,theta,name)].attrs['physically_admissible'] else '')

## 6. Panorama: ¿en cuáles labs se agota N prematuramente?
Las líneas muestran ambos theta sin perturbaciones. El umbral de 5 mg/L define un agotamiento operativo del modelo; no se confunde con un cero matemático. La biomasa inicial y la temperatura difieren entre labs, incluso si el YAN inicial es parecido.

In [ ]:
for labs,nrows,ncols,name in [(HIST_LABS,3,3,'01_panorama_historico'),(LABS,1,3,'02_panorama_reciente')]:
    fig,axes=plt.subplots(nrows,ncols,figsize=(14,3.2*nrows),squeeze=False,layout='constrained')
    for ax,lab in zip(axes.flat,labs):
        for theta in THETA_SETS:
            c=CURVES[(lab,theta,'referencia')]
            ax.plot(c.index,c.N_pred*1000,color=THETA_COLOR[theta],ls=THETA_STYLE[theta],
                label=theta+(' †' if not c.attrs['physically_admissible'] else ''))
        draw_observations(ax,lab,'N',1000.);mark_events(ax,lab)
        ax.axhline(5,color='gray',ls=':',lw=.8,label='Umbral 5 mg/L')
        ax.set(title=lab,xlabel='t [h]',ylabel='N / YAN [mg N/L]')
    common_legend(fig,axes);save(fig,name)

## 7. PAN, componente amoniacal y YAN total
Para comparar unidades, el componente amoniacal se expresa como aporte de N a YAN usando la convención de cada campaña: 0,82×NH informado en históricos y 0,823×NH no negativo en recientes. El valor bruto y la conversión se conservan en el CSV. Una diferencia en el factor de conversión no se atribuye automáticamente al ensayo.

Un remanente dominado por PAN justifica revisar el modelo de observación, pero no demuestra que ese PAN sea no asimilable. En los primeros dos paneles no se trazan predicciones ficticias de PAN o amonio.

In [ ]:
for group in GROUPS:
    fig,axes=plt.subplots(3,3,figsize=(14,9),layout='constrained')
    for i,lab in enumerate(group):
        data=COMPONENTS.loc[COMPONENTS.lab.eq(lab)&~COMPONENTS.is_NUT].sort_values('t_h')
        for j,(column,title,color) in enumerate([('PAN','PAN','#00897b'),
            ('ammonia_N','Componente amoniacal como N','#7b1fa2'),('YAN','YAN total','black')]):
            ax=axes[i,j]
            ax.plot(data.t_h,data[column],'o-',ms=3,color=color,label=title+' medido')
            if column=='YAN':
                c=CURVES[(lab,'theta_full','referencia')]
                ax.plot(c.index,c.N_pred*1000,color='#2166ac',label='N modelo full')
                draw_nut(ax,lab)
            else:
                draw_nut_component(ax,lab,column)
            mark_events(ax,lab);ax.set(title=f'{lab} · {title}',xlabel='t [h]',ylabel='mg N/L')
    common_legend(fig,axes);save(fig,'03_componentes_'+group_name(group))

## 8. Sensibilidad del consumo: X inicial, qN y rapidez
Cada panel cambia un mecanismo, manteniendo los restantes parámetros. Se muestra el horizonte completo, incluidos los pulsos. La escala de rapidez modifica μ₀/qN/sN conjuntamente para conservar el rendimiento y la constante Monod; no se confunde con reducir sólo μ₀. Son escenarios exploratorios, no intervalos de confianza.

In [ ]:
SENS_GROUPS=[('Biomasa inicial',['X0_x0.5','X0_x2']),('Captación qN',['qN_x0.5','qN_x2']),
             ('Rapidez; rendimiento fijo',['crecimiento_x0.8','crecimiento_x1.2'])]
for group in GROUPS:
    fig,axes=plt.subplots(3,3,figsize=(14,9),layout='constrained')
    for i,lab in enumerate(group):
        for j,(title,names) in enumerate(SENS_GROUPS):
            ax=axes[i,j]
            for name in ['referencia',*names]:
                c=CURVES[(lab,'theta_full',name)]
                ax.plot(c.index,c.N_pred*1000,color=SC_COLOR[name],label=case_label(lab,'theta_full',name))
            draw_observations(ax,lab,'N',1000.);mark_events(ax,lab)
            ax.set(title=f'{lab} · {title}',xlabel='t [h]',ylabel='N / YAN [mg N/L]')
    common_legend(fig,axes,ncol=5);save(fig,'04_sensibilidad_'+group_name(group))

### Saturación y temperatura: controles complementarios
Las columnas comparan sN×0,5/×2 y T±2 °C. El segundo caso altera todas las tasas dependientes de temperatura y es una perturbación hipotética de la entrada, no una afirmación de error del sensor. Se muestran tres casos históricos representativos y los tres recientes; la tabla contiene estos escenarios para los doce labs y ambos theta.

In [ ]:
for group in [('LAB006','LAB011','LAB012'),LABS]:
    fig,axes=plt.subplots(3,2,figsize=(12,9),layout='constrained')
    for i,lab in enumerate(group):
        for j,(title,names) in enumerate([('Saturación sN',['sN_x0.5','sN_x2']),
                                         ('Temperatura ±2 °C',['T_menos2C','T_mas2C'])]):
            ax=axes[i,j]
            for name in ['referencia',*names]:
                c=CURVES[(lab,'theta_full',name)]
                ax.plot(c.index,c.N_pred*1000,color=SC_COLOR[name],label=case_label(lab,'theta_full',name))
            draw_observations(ax,lab,'N',1000.);mark_events(ax,lab)
            ax.set(title=f'{lab} · {title}',xlabel='t [h]',ylabel='mg N/L')
    common_legend(fig,axes);save(fig,'05_saturacion_temperatura_'+group_name(group))

## 9. Qué pasa con las otras variables al mejorar N
Se comparan referencia, X0×0,5, qN×0,5 y rapidez×0,8. Mirando sólo N, dos mecanismos pueden dar curvas muy parecidas, pero predecir distinta biomasa, azúcar y etanol. X reciente se muestra como proxy hueco y no entra en las métricas de validación. La incertidumbre de actividad frente a viabilidad no se elimina convirtiendo células a gramos.

In [ ]:
for group in GROUPS:
    fig,axes=plt.subplots(3,4,figsize=(17,9),layout='constrained')
    for i,lab in enumerate(group):
        for j,state in enumerate(['N','X','GF','E']):
            ax=axes[i,j];scale=1000. if state=='N' else 1.
            for name in ['referencia','X0_x0.5','qN_x0.5','crecimiento_x0.8']:
                c=CURVES[(lab,'theta_full',name)]
                ax.plot(c.index,(c.N_pred if state=='N' else c[state])*scale,
                    color=SC_COLOR[name],label=case_label(lab,'theta_full',name))
            draw_observations(ax,lab,state,scale);mark_events(ax,lab)
            ax.set(title=f'{lab} · {state}',xlabel='t [h]',ylabel='mg N/L' if state=='N' else 'g/L')
    common_legend(fig,axes,ncol=6);save(fig,'06_consecuencias_'+group_name(group))
COMPENSATION=[]
for lab,batch in BATCHES.items():
    first=min((t for t,d in batch.pulses['N']),default=np.inf)
    a=CURVES[(lab,'theta_full','X0_x0.5')];b=CURVES[(lab,'theta_full','qN_x0.5')]
    pre=a.index<=first
    COMPENSATION.append(dict(lab=lab,max_N_difference_before_pulse_mg_l=1000*np.max(np.abs(a.N[pre]-b.N[pre])),
        max_X_difference_full_g_l=float(np.max(np.abs(a.X-b.X))),
        max_E_difference_full_g_l=float(np.max(np.abs(a.E-b.E)))))
COMPENSATION=pd.DataFrame(COMPENSATION)
display(COMPENSATION.round(5))

## 10. Consumo de N y crecimiento: una restricción verificable
Cada panel utiliza únicamente muestras simultáneas N/X/Xd. El eje x es Δ(X+Xd); el eje y es N inicial + dosis − N medido. La línea representa la relación impuesta por qN/μ₀. Si los puntos no siguen esa relación, el modelo, la correspondencia de estados, la conversión de biomasa o el registro de entradas deben revisarse: el gráfico no identifica por sí solo cuál falla.

Las líneas que unen puntos muestran el orden temporal. En los recientes la biomasa es exploratoria; en los históricos sigue existiendo incertidumbre por masa celular y actividad. No se interpola biomasa para fabricar pares simultáneos.

In [ ]:
for labs,nrows,ncols,name in [(HIST_LABS,3,3,'07_balance_historico'),(LABS,1,3,'08_balance_reciente_proxy')]:
    fig,axes=plt.subplots(nrows,ncols,figsize=(14,3.3*nrows),squeeze=False,layout='constrained')
    for ax,lab in zip(axes.flat,labs):
        p=BIOMASS_BALANCE.loc[BIOMASS_BALANCE.lab.eq(lab)].sort_values('t_h')
        if p.empty: ax.set_title(lab+' · sin pares');continue
        ax.plot(p.delta_total_biomass_g_l,p.net_N_consumed_mg_l,'o-',ms=4,color='gray' if lab in LABS else 'black',
                label='Proxy reciente; no validación' if lab in LABS else 'Pares simultáneos')
        span=np.linspace(min(0.,p.delta_total_biomass_g_l.min()),max(.1,p.delta_total_biomass_g_l.max()),50)
        ax.plot(span,1000*THETA_FULL['qN']/THETA_FULL['mu0']*span,color='#2166ac',label='Restricción del modelo')
        ax.axhline(0,color='gray',lw=.6);ax.axvline(0,color='gray',lw=.6)
        ax.set(title=lab,xlabel='Δ biomasa total [g/L]',ylabel='N consumido neto [mg N/L]')
    common_legend(fig,axes);save(fig,name)
display(BIOMASS_BALANCE.groupby(['cohort','lab']).agg(n_pairs=('t_h','size'),
    max_abs_closure_mg_l=('closure_mg_l',lambda s:s.abs().max())).round(3))

## 11. Hipótesis explícita de remanente
Se prueba YAN_pred=N_disponible+B con B=0/10/20/30 mg/L **idéntico entre labs**. Se descuenta B de N inicial antes de simular; la dosis registrada se añade a N_disponible. Esto cambia también crecimiento y fermentación, por lo que no es subir una curva ya calculada.

Izquierda: predicción de YAN frente a mediciones. Derecha: error predicción−YAN, excluyendo el ancla inicial. Un remanente que ayuda al final puede no explicar el consumo prematuro y puede contradecir mediciones por debajo de B. Se informa ese conteo, sin eliminar datos ni estimar B usando el último punto de cada lab. Un B constante tampoco puede explicar aumentos de YAN entre muestras sin una fuente registrada.

In [ ]:
for group in GROUPS:
    fig,axes=plt.subplots(3,2,figsize=(12,9),layout='constrained')
    for i,lab in enumerate(group):
        pts=POINTS.loc[POINTS.lab.eq(lab)&POINTS.state.eq('N')&~POINTS.initial_anchor]
        for name in ['referencia','remanente_10','remanente_20','remanente_30']:
            c=CURVES[(lab,'theta_full',name)];color=SC_COLOR[name]
            axes[i,0].plot(c.index,c.N_pred*1000,color=color,label=case_label(lab,'theta_full',name))
            pred=np.interp(pts.t_h,c.index,c.N_pred)
            axes[i,1].plot(pts.t_h,1000*(pred-pts.observed),'o-',ms=3,color=color,label=case_label(lab,'theta_full',name))
        draw_observations(axes[i,0],lab,'N',1000.)
        for j in range(2):
            mark_events(axes[i,j],lab)
            axes[i,j].set(title=f'{lab} · '+('YAN predicho' if j==0 else 'Error posterior'),
                xlabel='t [h]',ylabel='mg N/L')
        axes[i,1].axhline(0,color='black',lw=1)
    common_legend(fig,axes);save(fig,'09_remanente_'+group_name(group))
display(FEATURES.query('theta=="theta_full" and scenario in ["referencia","remanente_10","remanente_20","remanente_30"]')
    [['lab','scenario','background_mg_l','n_YAN_below_background','first_available_N_le_5_h',
      'first_predicted_YAN_le_5_h','final_YAN_pred_mg_l']].round(3))

## 12. Métricas comparables, cruces observados y NUT reservadas
El agotamiento observado se informa como intervalo entre muestras; si no hay una muestra ≤5 mg/L antes del pulso se declara que no se observó ese cruce. No se interpreta la última muestra positiva como hora de agotamiento.

El diagrama de compromiso usa ratios de RMSE respecto a la referencia del mismo lab: izquierda/abajo de 1 es mejora. El resumen no-N usa G/F/E/Gly/X/Xd históricos, y GF/E/Gly recientes. Son pesos exploratorios y se evita contar GF junto a sus componentes. No se interpreta un cambio de esos pesos como una nueva simulación ni como validación estadística.

In [ ]:
complete=METRICS.query('period=="completo"').copy()
OTHER_ROWS=[]
for (lab,theta,scenario),g in complete.groupby(['lab','theta','scenario']):
    channels=['G','F','E','Gly','X','Xd'] if lab in HIST_LABS else ['GF','E','Gly']
    other=g[g.state.isin(channels)&g.eligible]
    OTHER_ROWS.append(dict(lab=lab,theta=theta,scenario=scenario,n_other=int(other.n.sum()),
        other_weighted_RMSE=float(np.sqrt(other.WSSE.sum()/other.n.sum()))))
OTHER=pd.DataFrame(OTHER_ROWS)
N_METRICS=complete[complete.state.eq('N')].merge(OTHER,on=['lab','theta','scenario'],validate='one_to_one')
reference=N_METRICS.query('scenario=="referencia"')[['lab','theta','RMSE','other_weighted_RMSE']]
N_METRICS=N_METRICS.merge(reference,on=['lab','theta'],suffixes=('','_reference'),validate='many_to_one')
N_METRICS['N_RMSE_ratio']=N_METRICS.RMSE/N_METRICS.RMSE_reference
N_METRICS['other_RMSE_ratio']=N_METRICS.other_weighted_RMSE/N_METRICS.other_weighted_RMSE_reference
fig,axes=plt.subplots(2,2,figsize=(15,10),layout='constrained')
x=np.arange(len(ALL_LABS));width=.36
for theta,offset in [('theta_full',-.18),('theta_defaults',.18)]:
    p=N_METRICS.query('scenario=="referencia" and theta==@theta').set_index('lab').loc[list(ALL_LABS)]
    axes[0,0].bar(x+offset,p.RMSE*1000,width,color=THETA_COLOR[theta],label=theta)
    axes[0,1].bar(x+offset,p.bias*1000,width,color=THETA_COLOR[theta],label=theta)
for ax,title in [(axes[0,0],'Error N: RMSE completo'),(axes[0,1],'Sesgo: predicción − YAN')]:
    ax.set_xticks(x,ALL_LABS,rotation=55);ax.set(title=title,ylabel='mg N/L');ax.legend(fontsize=8)
f=FEATURES.query('theta=="theta_full" and scenario=="referencia"').set_index('lab')
for i,lab in enumerate(ALL_LABS):
    row=f.loc[lab];ax=axes[1,0]
    if np.isfinite(row.first_available_N_le_5_h):
        ax.scatter(i,row.first_available_N_le_5_h,color='#2166ac',marker='o',label='Cruce predicho' if i==0 else None)
    if np.isfinite(row.obs_upper_h):
        lower=row.obs_lower_h if np.isfinite(row.obs_lower_h) else row.obs_upper_h
        ax.plot([i,i],[lower,row.obs_upper_h],color='black',lw=3)
        ax.scatter(i,row.obs_upper_h,color='black',marker='_',label='Intervalo entre muestras' if i==0 else None)
    else:
        pts=POINTS.loc[POINTS.lab.eq(lab)&POINTS.state.eq('N')]
        pulse=min((t for t,d in BATCHES[lab].pulses['N']),default=np.inf)
        pos=pts.loc[(pts.t_h<=pulse)&(pts.observed>.005)]
        if len(pos): ax.scatter(i,pos.t_h.max(),marker='^',facecolors='none',edgecolors='#ef6c00',
            label='Última muestra >5; cruce no observado')
axes[1,0].set_xticks(x,ALL_LABS,rotation=55)
axes[1,0].set(title='Agotamiento antes del primer pulso',ylabel='t [h]')
hh,ll=axes[1,0].get_legend_handles_labels();dd=dict(zip(ll,hh));axes[1,0].legend(dd.values(),dd.keys(),fontsize=7)
for name in ['X0_x0.5','qN_x0.5','crecimiento_x0.8','remanente_20']:
    for cohort,marker in [('historico','o'),('reciente','^')]:
        p=N_METRICS.query('theta=="theta_full" and scenario==@name and cohort==@cohort')
        admissible=p.physically_admissible.astype(bool)
        axes[1,1].scatter(p.loc[admissible,'N_RMSE_ratio'],p.loc[admissible,'other_RMSE_ratio'],color=SC_COLOR[name],marker=marker,s=35,
            label=name+' · '+cohort,alpha=.8)
        if (~admissible).any():
            axes[1,1].scatter(p.loc[~admissible,'N_RMSE_ratio'],p.loc[~admissible,'other_RMSE_ratio'],
                marker='x',color=SC_COLOR[name],s=50,label='Recorte interno; no admisible')
axes[1,1].axvline(1,color='gray',ls=':');axes[1,1].axhline(1,color='gray',ls=':')
axes[1,1].set(title='Mejorar N vs otras variables',xlabel='RMSE N / referencia',ylabel='Error no-N / referencia')
axes[1,1].legend(fontsize=7,ncol=2)
save(fig,'10_resumen_comparativo')
NUT_ROWS=[]
for (lab,theta,scenario),curve in CURVES.items():
    if lab not in LABS: continue
    nut=NUT_SAMPLES[NUT_SAMPLES.lab.eq(lab)].iloc[0]
    support=np.unique(np.r_[nut.time_lower_h,nut.time_upper_h,
        curve.index[(curve.index>=nut.time_lower_h)&(curve.index<=nut.time_upper_h)]])
    pred=diagnosis.interpolate_post_action(curve,support,BATCHES[lab].pulses['N'])*1000.
    NUT_ROWS.append(dict(lab=lab,theta=theta,scenario=scenario,observed_mg_l=nut.YAN_mg_l,
        lower_h=nut.time_lower_h,upper_h=nut.time_upper_h,pred_min_mg_l=float(pred.min()),
        pred_max_mg_l=float(pred.max()),time_provisional=nut.time_provisional))
NUT_VALIDATION=pd.DataFrame(NUT_ROWS)
display(NUT_VALIDATION.query('theta=="theta_full" and scenario in ["referencia","qN_x0.5","X0_x0.5","remanente_20"]').round(3))
display(N_METRICS.query('theta=="theta_full" and scenario in ["referencia","qN_x0.5","X0_x0.5","remanente_20"]')
    [['lab','scenario','n','RMSE','bias','N_RMSE_ratio','other_RMSE_ratio']].round(4))

## 13. Auditoría de la calibración histórica
Se reconstruye el objetivo histórico con theta_full (once valores naturales y seis heredados fijos) y el integrador oficial. Incluye las anclas iniciales para reproducir sus 608 residuos; las métricas diagnósticas anteriores las excluyen. Se comprueba el objetivo contra el CSV publicado y se informa también qué ocurre al completar el CSV incompleto con defaults. Esta segunda variante no reproduce el vector completo del ajuste original.

La descomposición muestra cuánto aportó cada estado al objetivo. El cambio hipotético del peso de N sólo recalcula el objetivo de las mismas curvas: **no se ejecuta un reajuste**. Las escalas σ son pesos históricos, no incertidumbres certificadas de los ensayos. La etiqueta success del optimizador no demuestra buen ajuste por estado ni optimalidad global.

In [ ]:
AUDIT_POINTS=[]
hist_batches=[BATCHES[lab] for lab in HIST_LABS]
for batch in hist_batches:
    official=base.simulate(batch,THETA_FULL,batch.time)
    assert official is not None
    for s in base.STATE_NAMES:
        obs=batch.observations[s];keep=np.isfinite(obs)
        pred=official.loc[batch.time,s].to_numpy()[keep];yy=obs[keep]
        sigma=base.sigma_for_state(s,yy)
        for t,y,p,ss in zip(batch.time[keep],yy,pred,sigma):
            AUDIT_POINTS.append(dict(lab=batch.batch,state=s,t_h=float(t),observed=float(y),
                predicted=float(p),sigma=float(ss),residual_scaled=float((p-y)/ss),
                initial_anchor=bool(t==batch.time[0])))
AUDIT_POINTS=pd.DataFrame(AUDIT_POINTS)
AUDIT_POINTS['WSSE']=AUDIT_POINTS.residual_scaled**2
CALIBRATION_AUDIT=AUDIT_POINTS.groupby('state').agg(n=('WSSE','size'),WSSE=('WSSE','sum')).reset_index()
published=pd.read_csv(CAL/'core_fit_summary.csv').iloc[0]
assert len(AUDIT_POINTS)==int(published.n_residuals)==608
reconstructed=float(CALIBRATION_AUDIT.WSSE.sum())
assert np.isclose(reconstructed,published.final_wsse,rtol=1e-8,atol=1e-5),(reconstructed,published.final_wsse)
default_residuals=base.residual_vector(THETA_DEFAULTS,hist_batches)
default_objective=float(default_residuals@default_residuals)
CALIBRATION_AUDIT['fraction_objective']=CALIBRATION_AUDIT.WSSE/reconstructed
display(CALIBRATION_AUDIT.round(4))
display(pd.DataFrame([{'objective_reconstructed':reconstructed,'objective_published':published.final_wsse,
                       'objective_defaults_variant':default_objective,
                       'n_residuals':len(AUDIT_POINTS),'historical_fit_success':published.success}]))
fig,axes=plt.subplots(2,2,figsize=(14,9),layout='constrained')
axes[0,0].bar(CALIBRATION_AUDIT.state,CALIBRATION_AUDIT.WSSE,color='#2166ac')
axes[0,0].set(title='Aporte de cada estado al objetivo',ylabel='Suma de residuos escalados²')
axes[0,1].bar(CALIBRATION_AUDIT.state,CALIBRATION_AUDIT.n,color='#00897b')
axes[0,1].set(title='Observaciones utilizadas',ylabel='Número de residuos')
npoints=AUDIT_POINTS[AUDIT_POINTS.state.eq('N')]
axes[1,0].scatter(npoints.observed*1000,npoints.residual_scaled,s=14,color='black')
axes[1,0].axhline(0,color='gray');axes[1,0].set(title='Residuos de N en la calibración',
    xlabel='YAN observado [mg N/L]',ylabel='(predicción − medición) / escala de ajuste')
nwsse=float(CALIBRATION_AUDIT.loc[CALIBRATION_AUDIT.state.eq('N'),'WSSE'].iloc[0])
weights=np.array([.5,1.,2.,4.]);nv=nwsse*weights**2;other=reconstructed-nwsse
axes[1,1].bar(np.arange(4),nv,label='N',color='#2166ac')
axes[1,1].bar(np.arange(4),np.full(4,other),bottom=nv,label='Otros estados',color='#bdbdbd')
axes[1,1].set_xticks(np.arange(4),['×0.5','×1','×2','×4'])
axes[1,1].set(title='Peso del residuo N; mismas curvas, sin reajuste',xlabel='Multiplicador del residuo escalado',ylabel='Objetivo')
axes[1,1].legend();save(fig,'11_auditoria_calibracion')

## 14. Resultados, límites y decisiones
Las comparaciones siguientes no eligen automáticamente un ganador. Una mejora de N que empeora fuertemente azúcares, etanol o biomasa identifica un compromiso, no una solución completa. Un fondo constante no explica aumentos de YAN sin fuentes y debe respetar las mediciones por debajo de él. LAB010–012 tienen química escasa/anómala y se conservan con sus flags; no se eliminan para mejorar el ajuste.

El experimento químico completo, la correspondencia YAN–estado N y los componentes analíticos se evalúan antes de introducir CO₂ o EKF. Esta prueba no estima por sí sola el estado fisiológico del inóculo ni demuestra que todo PAN residual sea no disponible.

In [ ]:
summary=N_METRICS.query('theta=="theta_full"').groupby(['cohort','scenario']).agg(
    median_N_RMSE_ratio=('N_RMSE_ratio','median'),median_other_RMSE_ratio=('other_RMSE_ratio','median'),
    n_labs=('lab','nunique'),n_physically_admissible=('physically_admissible','sum')).reset_index()
display(summary.round(4))
lines=['**Lectura calculada de los escenarios:**']
for cohort in ['historico','reciente']:
    lines.append('**'+cohort+'**')
    for name in ['X0_x0.5','qN_x0.5','crecimiento_x0.8','remanente_20']:
        row=summary.query('cohort==@cohort and scenario==@name').iloc[0]
        lines.append(f'- {name}: ratio mediano de error N {row.median_N_RMSE_ratio:.2f}; '
                     f'ratio no-N {row.median_other_RMSE_ratio:.2f}; '
                     f'{int(row.n_physically_admissible)}/{row.n_labs} labs sin excursiones internas fuera de límites.')
fraction=float(CALIBRATION_AUDIT.loc[CALIBRATION_AUDIT.state.eq('N'),'fraction_objective'].iloc[0])
lines.append(f'N representó {100*fraction:.1f}% del objetivo histórico reconstruido. '
    'Esto describe el peso efectivo del error, no demuestra que una simple reponderación solucione el modelo.')
lines.append(f'Se detectan {(OBSERVED_BALANCE.excess_mg_l>20).sum()} intervalos con aumentos mayores a 20 mg/L '
    'por encima de las entradas registradas; revisar fuentes, muestreo y representación antes de exigir un ajuste exacto.')
display(Markdown('\n\n'.join(lines)))
TABLES=dict(parameters=PARAMETERS,inputs=INPUTS,recent_initialization_audit=recent_audit,
    observations=POINTS,components=COMPONENTS,observed_nitrogen_balance=OBSERVED_BALANCE,
    biomass_nitrogen_balance=BIOMASS_BALANCE,metrics=METRICS,mechanisms=MECHANISMS,
    depletion_features=FEATURES,checks=CHECKS,nitrogen_vs_other_metrics=N_METRICS,
    scenario_summary=summary,compensation=COMPENSATION,nut_reserved_validation=NUT_VALIDATION,
    calibration_audit=CALIBRATION_AUDIT,calibration_residuals=AUDIT_POINTS)
for name,table in TABLES.items(): table.to_csv(OUT/(name+'.csv'),index=False)
arrays={};array_manifest=[]
for i,(key,curve) in enumerate(CURVES.items()):
    array_name=f'case_{i}';arrays[array_name]=np.column_stack([curve.index,curve.to_numpy()])
    array_manifest.append(dict(array=array_name,key=list(key),columns=['t_h']+list(curve.columns)))
np.savez_compressed(OUT/'trajectories.npz',**arrays)
pd.DataFrame(FIGURE_LOG).to_csv(OUT/'figure_index.csv',index=False)
HASH_AFTER={str(p.relative_to(ROOT)):sha256(p) for p in PROTECTED}
assert HASH_BEFORE==HASH_AFTER,'Cambió un archivo protegido'
manifest=dict(observation_policy=GF_ZERO_POLICY,created_at=datetime.now().astimezone().isoformat(),python=sys.executable,
    input_sha256=HASH_BEFORE,theta=THETA_SETS,scenarios=diagnosis.SCENARIOS,
    config=diagnosis.CONFIG,arrays=array_manifest,figures=FIGURE_LOG,
    horizons={lab:float(b.time[-1]) for lab,b in BATCHES.items()},
    gas_data_loaded=False,optimizer_calls=0,kalman_updates=0,scenario_count=len(CURVES),
    calibration_objective_reconstructed=reconstructed,calibration_objective_published=float(published.final_wsse),
    calibration_objective_defaults_variant=default_objective,
    n_physically_admissible=sum(c.attrs['physically_admissible'] for c in CURVES.values()),
    recent_biomass_validation=False,constant_background='exploratory observation hypothesis; shared grid; not fitted',
    source_code_sha256=sha256(LAB2026/'notebooks/diagnostics/nitrogen_consumption_diagnosis.source.ipynb'))
(OUT/'manifest.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding='utf-8')
print('Integridad verificada:',len(PROTECTED),'inputs; escenarios:',len(CURVES),'figuras:',len(FIGURE_LOG))
print('Datos, calibraciones y notebooks anteriores intactos; sin commit ni push.')

## 15. Conclusiones de la configuración congelada

**La rapidez inicial y el remanente final son dos problemas diferentes.** Un fondo de 20 mg/L reduce el error mediano de N aproximadamente 16% en históricos y 9% en recientes; mejora parte de la cola, pero no explica las grandes diferencias tempranas. No se ha demostrado que todo PAN residual sea no asimilable.

**La biomasa efectiva inicial es una hipótesis prioritaria en LAB019–021.** X0×0,5 reduce el RMSE mediano de N alrededor de 52% y el resumen de otras variables químicas alrededor de 56%. qN×0,5 da prácticamente la misma mejora de N, pero aumenta el error no-N alrededor de 65%. Una curva de N aislada no permite distinguir estos mecanismos. No se afirma que X0×0,5 sea la biomasa verdadera: es un escenario manual y la conversión de recuento a biomasa activa sigue siendo incierta.

**La corrección no se transfiere sin más a todos los históricos.** Allí X0×0,5 mejora N alrededor de 25%, pero empeora el resumen no-N alrededor de 34%. Reducir la rapidez conjuntamente a ×0,8 mejora N aproximadamente 27% con un cambio mediano no-N cercano a −2%; es una dirección que merece estudiar mediante un ajuste comparable, no una calibración nueva ya validada.

**La auditoría reproduce la calibración original.** theta_full recupera el objetivo publicado (11672,084819; 608 residuos). Completar el CSV incompleto con defaults da 17248,638426. N aporta 13,0% al objetivo original y X 30,5%: N sí participó, pero el ajuste se compromete con otros estados. Reponderar N requiere después reajustar y validar; cambiar el peso de curvas fijas no demuestra que el problema se resuelva.

**Los controles detectan cinco escenarios de los 336 con glucosa interna negativa.** Ocurren en theta_full al duplicar X0 en LAB010–012 y al reducir qN a la mitad en LAB010–011. Se conservan como diagnóstico con † y no se presentan como soluciones físicamente admisibles. La salida oficial los recorta a cero; las ecuaciones originales no se han modificado. En la RHS, el consumo de glucosa asociado al crecimiento (qXG × factor de crecimiento × X) no tiene una limitación por disponibilidad de G: puede continuar cuando G llega a cero si aún hay N. Estas excursiones de hasta unas décimas de g/L no se presentan como mero redondeo numérico; requieren revisar esa restricción antes de aceptar esos escenarios.

**Cinco intervalos tienen aumentos medidos superiores a 20 mg/L por encima de las entradas registradas.** Un modelo puramente consumidor no puede reproducirlos exactamente bajo ese registro, incluso cambiando parámetros. Corresponde revisar las entradas, el muestreo y la representación del estado; no atribuir automáticamente esos aumentos a un fallo del Y15.

**Siguiente decisión científica:** estudiar por separado biomasa efectiva inicial, rapidez/rendimiento y relación YAN–N, usando la química completa para evitar compensaciones. Conservar los labs recientes como contraste condicionado y comprobar una corrección compartida en datos no utilizados para elegirla. Este notebook no recalibra, no usa CO₂ y no ejecuta EKF.

## 16. Inóculo común medido: 7,44 × 10⁶ células/mL en LAB019–021

Esta sección incorpora un dato nuevo aportado por el usuario **después** del diagnóstico anterior. El reporte del inóculo conjunto, antes de repartirlo, indica 496,77 millones de células/mL totales, 424,19 millones viables/mL y **85,39 % de viabilidad**. El usuario calcula una concentración inicial en cada reactor de **7,44 millones de células/mL**. Se acepta ese valor ya diluido: no se vuelve a aplicar la dilución analítica 5:100 ni se reconstruyen volúmenes de reparto que no se proporcionaron. El valor común presupone un reparto homogéneo; no demuestra que los tres reactores recibieran exactamente la misma concentración.

Con la conversión usada en estos notebooks, 0,03 g/L por millón de células/mL:
\[
X_{\mathrm{total},0}=7,44\times0,03=0,2232\ \mathrm{g/L}.
\]
Esta conversión es una hipótesis de masa por célula, no una medición de peso seco ni de actividad metabólica. El dato no es **7,44 células/mL**.

Se comparan tres inicializaciones con **los mismos parámetros antiguos y ambos theta**:

| Caso | Estado X inicial | Estado Xd inicial | Propósito |
|---|---|---|---|
| Referencia anterior | Oculyze total de cada reactor | Valor anterior de cada reactor | Conservar la comparación ya presentada |
| 7,44 · convención anterior | 0,2232 g/L | Valor anterior de cada reactor | Cambiar solamente X₀, igual que en los escenarios anteriores |
| 7,44 · separar viabilidad | 0,2232 × 0,8539 = 0,19059048 g/L | 0,2232 × (1−0,8539) = 0,03260952 g/L | Inicialización coherente con X viable y Xd muerta |

**La segunda fila sigue deliberadamente la convención anterior de poner recuento total en X. No debe interpretarse como una separación correcta de células vivas y muertas.** La tercera usa la viabilidad del inóculo y conserva X+Xd=0,2232 g/L. No se suma también el CERO del reactor ni se descuenta un fondo sin una correspondencia experimental confirmada.

Se mantienen química, temperatura, reloj y pulsos; se simula **hasta las 166 h**. No se introduce lag, EKF, CO₂, remanente ni reajuste. Las mediciones posteriores de química se reservan para contraste; la biomasa Oculyze posterior sigue siendo un proxy excluido de la puntuación. NUT mantiene su intervalo temporal provisional.

In [ ]:
INOC_OUT=OUT/'inoculum_common_744';INOC_OUT.mkdir(exist_ok=True)
INOC_TOTAL_MILLION_ML=7.44
INOC_VIABILITY=.8539
INOC_MASS_FACTOR=.03
INOC_TOTAL_G_L=INOC_TOTAL_MILLION_ML*INOC_MASS_FACTOR
INOC_CASES={
    'referencia_anterior':('Referencia anterior','#2166ac','-'),
    'inoculo_total_convencion':('7,44 · convención anterior','#00897b','--'),
    'inoculo_viable_consistente':('7,44 · separar viabilidad','#e66101','-.'),
}
INOC_PROVENANCE=dict(total_inoculum_million_cells_ml=496.77,
    viable_inoculum_million_cells_ml=424.19,viability_fraction=INOC_VIABILITY,
    reactor_total_million_cells_ml=INOC_TOTAL_MILLION_ML,
    reactor_concentration_source='User-calculated dilution; volumes not supplied',
    mass_factor_g_l_per_million_cells_ml=INOC_MASS_FACTOR,
    screenshot='inoculum_report.png',screenshot_sha256=sha256(INOC_OUT/'inoculum_report.png'),
    measurement_created='2026-09-21 17:08; report creation time, not assumed sampling time',
    no_second_analytical_dilution=True,old_raw_data_changed=False)
(INOC_OUT/'input_provenance.json').write_text(json.dumps(INOC_PROVENANCE,ensure_ascii=False,indent=2),encoding='utf-8')
INOC_CURVES={};INOC_INPUT_ROWS=[];INOC_SCORE_ROWS=[];INOC_CHECK_ROWS=[];INOC_FEATURE_ROWS=[];INOC_NUT_ROWS=[]
for lab in LABS:
    original=BATCHES[lab]
    common=replace(original,initials={**original.initials,'X':INOC_TOTAL_G_L})
    viable=replace(original,initials={**original.initials,
        'X':INOC_TOTAL_G_L*INOC_VIABILITY,'Xd':INOC_TOTAL_G_L*(1-INOC_VIABILITY)})
    alternatives={'referencia_anterior':original,'inoculo_total_convencion':common,
                  'inoculo_viable_consistente':viable}
    for case,batch in alternatives.items():
        assert all(batch.initials[s]==original.initials[s] for s in ['N','G','F','E','Gly'])
        assert batch.pulses==original.pulses
        assert np.array_equal(batch.temperature_c,original.temperature_c)
        assert np.array_equal(batch.time,original.time)
        if case=='inoculo_total_convencion': assert batch.initials['Xd']==original.initials['Xd']
        if case=='inoculo_viable_consistente':
            assert np.isclose(batch.initials['X']+batch.initials['Xd'],INOC_TOTAL_G_L)
        INOC_INPUT_ROWS.append(dict(lab=lab,case=case,label=INOC_CASES[case][0],
            X0_g_l=batch.initials['X'],Xd0_g_l=batch.initials['Xd'],
            initial_total_state_g_l=batch.initials['X']+batch.initials['Xd'],
            X0_ratio_to_previous=batch.initials['X']/original.initials['X'],
            N0_mg_l=1000*batch.initials['N'],end_h=float(batch.time[-1]),
            N_pulses=repr(batch.pulses['N'])))
        for theta,th in THETA_SETS.items():
            curve=CURVES[(lab,theta,'referencia')] if case=='referencia_anterior' else diagnosis.run(batch,th,'referencia')
            INOC_CURVES[(lab,theta,case)]=curve
            pulse=min(t for t,d in batch.pulses['N'])
            points=POINTS[POINTS.lab.eq(lab)]
            for row in diagnosis.score(curve,points,pulse):
                INOC_SCORE_ROWS.append(dict(lab=lab,theta=theta,case=case,**row))
            pre=curve[curve.index<=pulse]
            after=curve[curve.index>pulse]
            INOC_FEATURE_ROWS.append(dict(lab=lab,theta=theta,case=case,
                first_pre_pulse_N_le_5_h=diagnosis.first_crossing(pre.index,pre.N),
                first_post_pulse_N_le_5_h=diagnosis.first_crossing(after.index,after.N),
                first_post_pulse_N_le_5_delay_h=diagnosis.first_crossing(after.index,after.N)-pulse,
                initial_uptake_mg_l_h=float(curve.uptake_mg_l_h.iloc[0]),
                final_YAN_mg_l=1000*curve.N_pred.iloc[-1],
                **diagnosis.observed_crossing(points,pulse)))
            rawmin=curve.attrs['raw_min'];rawmax=curve.attrs['raw_max']
            INOC_CHECK_ROWS.append(dict(lab=lab,theta=theta,case=case,
                initial_X_error_g_l=abs(curve.X.iloc[0]-batch.initials['X']),
                initial_Xd_error_g_l=abs(curve.Xd.iloc[0]-batch.initials['Xd']),
                initial_N_error_g_l=abs(curve.N.iloc[0]-batch.initials['N']),
                inventory_drift_mg_l=1000*curve.attrs['inventory_drift'],
                jump_error=curve.attrs['jump_error'],
                physically_admissible=curve.attrs['physically_admissible'],
                start_h=float(curve.index[0]),end_h=float(curve.index[-1]),
                **{'raw_min_'+s:v for s,v in rawmin.items()},
                **{'raw_max_'+s:v for s,v in rawmax.items()}))
            assert curve.index[-1]==166. and curve.index[0]==0.
            assert curve.attrs['physically_admissible'],(lab,theta,case,rawmin)
            for event,dose in batch.pulses['N']:
                jump=curve.loc[event+diagnosis.CONFIG['after_event_h'],'N']-curve.loc[event,'N']
                assert abs(jump-dose)<1e-5,(lab,theta,case,'pulse')
            nut=NUT_SAMPLES[NUT_SAMPLES.lab.eq(lab)].iloc[0]
            support=np.unique(np.r_[nut.time_lower_h,nut.time_upper_h,
                curve.index[(curve.index>=nut.time_lower_h)&(curve.index<=nut.time_upper_h)]])
            pred=1000*diagnosis.interpolate_post_action(curve,support,batch.pulses['N'])
            INOC_NUT_ROWS.append(dict(lab=lab,theta=theta,case=case,
                observed_mg_l=nut.YAN_mg_l,lower_h=nut.time_lower_h,upper_h=nut.time_upper_h,
                pred_min_mg_l=float(pred.min()),pred_max_mg_l=float(pred.max()),time_provisional=True))
INOC_INPUTS=pd.DataFrame(INOC_INPUT_ROWS)
INOC_METRICS=pd.DataFrame(INOC_SCORE_ROWS)
INOC_FEATURES=pd.DataFrame(INOC_FEATURE_ROWS)
INOC_CHECKS=pd.DataFrame(INOC_CHECK_ROWS)
INOC_NUT=pd.DataFrame(INOC_NUT_ROWS)
display(INOC_INPUTS.round(6))
display(INOC_FEATURES.round(3))
print('12 simulaciones nuevas + 6 referencias conservadas; todos los horizontes completos y estados admisibles.')

### 16.1. Predicciones con el inóculo común: N, biomasa, azúcares y etanol

Cada figura contiene tres filas (labs) y cuatro columnas (N, X, GF, E), para un theta fijo. **Azul:** referencia anterior; **verde discontinuo:** 7,44 siguiendo la convención anterior; **naranja punto-raya:** 7,44 separando viabilidad. Las líneas verticales marcan los pulsos. Los puntos NUT conservan la convención temporal documentada en las tablas; los círculos huecos de biomasa no son validación fiable.

El inóculo común elimina la diferencia inicial de X entre reactores, pero las predicciones pueden seguir siendo distintas por temperatura, química y nutrición. No se obliga a que las curvas coincidan ni se mueve t₀.

Se agregaron los puntos iniciales de biomasa de los escenarios verde y naranja en t=0. Se muestran como círculos normales, sin leyendas adicionales ni recuadros dentro del panel. Los valores de X₀ están en las tablas de esta sección. La curva azul conserva la referencia anterior y los círculos huecos corresponden a las mediciones posteriores de biomasa. Las condiciones iniciales añadidas no se cuentan como mediciones independientes en RMSE.

In [ ]:
INOC_FIGURES=[]

def draw_new_biomass_initials(ax,lab,theta,curves,cases):
    pts=POINTS.loc[POINTS.lab.eq(lab)&POINTS.state.eq('X')&~POINTS.initial_anchor]
    ax.scatter(pts.t_h,pts.observed,s=20,marker='o',facecolors='none',
        edgecolors='gray',zorder=6,label='Biomasa')
    for case,(label,color,style) in cases.items():
        if case=='referencia_anterior': continue
        curve=curves[(lab,theta,case)]
        ax.scatter([float(curve.index[0])],[float(curve.X.iloc[0])],s=20,
            marker='o',color=color,zorder=8,clip_on=False,label='_nolegend_',
            gid=f'initial_X0:{lab}:{theta}:{case}')
    ax.set_xlim(-3,166)

def save_inoculum(fig,name):
    path=INOC_OUT/(name+'.png')
    fig.savefig(path,dpi=125,bbox_inches='tight',facecolor='white')
    buffer=io.BytesIO();fig.savefig(buffer,format='png',dpi=105,bbox_inches='tight',facecolor='white')
    plt.close(fig);display(Image(data=buffer.getvalue()));buffer.close()
    INOC_FIGURES.append(dict(name=name,path=str(path.relative_to(ROOT))))
for theta in THETA_SETS:
    fig,axes=plt.subplots(3,4,figsize=(17,9),layout='constrained')
    for i,lab in enumerate(LABS):
        for j,state in enumerate(['N','X','GF','E']):
            ax=axes[i,j];scale=1000. if state=='N' else 1.
            for case,(label,color,style) in INOC_CASES.items():
                curve=INOC_CURVES[(lab,theta,case)]
                ax.plot(curve.index,(curve.N_pred if state=='N' else curve[state])*scale,
                    color=color,ls=style,lw=1.7,label=label)
            if state=='X':
                draw_new_biomass_initials(ax,lab,theta,INOC_CURVES,INOC_CASES)
            else:
                draw_observations(ax,lab,state,scale)
            mark_events(ax,lab)
            ax.set(title=f'{lab} · {state}\n{theta}',xlabel='t desde inoculación [h]',
                ylabel='mg N/L' if state=='N' else 'g/L',xlim=(-3,166) if state=='X' else (0,166))
    common_legend(fig,axes,ncol=3)
    save_inoculum(fig,'01_quimica_'+theta)
fig,axes=plt.subplots(3,2,figsize=(13,9),layout='constrained')
for i,lab in enumerate(LABS):
    for j,theta in enumerate(THETA_SETS):
        ax=axes[i,j]
        pts=POINTS.loc[POINTS.lab.eq(lab)&POINTS.state.eq('N')&~POINTS.initial_anchor]
        for case,(label,color,style) in INOC_CASES.items():
            curve=INOC_CURVES[(lab,theta,case)]
            residual=1000*(np.interp(pts.t_h,curve.index,curve.N_pred)-pts.observed)
            ax.plot(pts.t_h,residual,marker='o',ms=3,color=color,ls=style,label=label)
        ax.axhline(0,color='black',lw=.8);mark_events(ax,lab)
        ax.set(title=f'{lab} · residuo N · {theta}',xlabel='t desde inoculación [h]',
            ylabel='Predicción − YAN [mg N/L]',xlim=(0,166))
common_legend(fig,axes,ncol=2)
save_inoculum(fig,'02_residuos_N_ambos_theta')

### 16.1.1. Comparación cuantitativa y límites

RMSE y sesgo se calculan sobre **las mismas muestras posteriores al ancla inicial**. Se presentan N, GF, E y Gly por separado, en sus unidades, y el resumen no-N usa sólo GF/E/Gly: no incluye biomasa incierta ni cuenta GF junto a G y F. Un ratio menor que 1 mejora frente a la referencia anterior del mismo lab y theta; no es una probabilidad ni un intervalo de confianza.

El cruce N≤5 mg/L es una definición operativa del modelo. Si no ocurre antes del pulso se muestra como ausente, no como agotamiento a la hora del pulso. La hora de agotamiento experimental sólo puede acotarse con las muestras disponibles. NUT se compara con el rango predicho sobre su intervalo temporal **después** del pulso y no entra como una observación de hora exacta en RMSE.

In [ ]:
INOC_COMPARISON_ROWS=[]
for (lab,theta,case),group in INOC_METRICS.query('period=="completo"').groupby(['lab','theta','case']):
    row=dict(lab=lab,theta=theta,case=case)
    for state in ['N','GF','E','Gly']:
        item=group[group.state.eq(state)].iloc[0]
        scale=1000. if state=='N' else 1.
        row[state+'_RMSE']=item.RMSE*scale
        row[state+'_bias']=item.bias*scale
        row[state+'_n']=int(item.n)
    other=group[group.state.isin(['GF','E','Gly'])&group.eligible]
    row['other_weighted_RMSE']=float(np.sqrt(other.WSSE.sum()/other.n.sum()))
    INOC_COMPARISON_ROWS.append(row)
INOC_COMPARISON=pd.DataFrame(INOC_COMPARISON_ROWS)
previous=INOC_COMPARISON.query('case=="referencia_anterior"')[['lab','theta','N_RMSE','other_weighted_RMSE']]
INOC_COMPARISON=INOC_COMPARISON.merge(previous,on=['lab','theta'],suffixes=('','_previous'),validate='many_to_one')
INOC_COMPARISON['N_RMSE_ratio']=INOC_COMPARISON.N_RMSE/INOC_COMPARISON.N_RMSE_previous
INOC_COMPARISON['other_RMSE_ratio']=INOC_COMPARISON.other_weighted_RMSE/INOC_COMPARISON.other_weighted_RMSE_previous
display(INOC_COMPARISON[['lab','theta','case','N_RMSE','N_bias','GF_RMSE','E_RMSE','Gly_RMSE',
    'N_RMSE_ratio','other_RMSE_ratio']].round(3))
display(INOC_NUT.round(3))
INOC_SUMMARY=INOC_COMPARISON.groupby(['theta','case']).agg(
    median_N_RMSE_ratio=('N_RMSE_ratio','median'),
    median_other_RMSE_ratio=('other_RMSE_ratio','median')).reset_index()
display(INOC_SUMMARY.round(3))
lines=['**Resultados calculados con el dato nuevo:**']
for theta in THETA_SETS:
    lines.append('**'+theta+'**')
    for case in list(INOC_CASES)[1:]:
        group=INOC_COMPARISON.query('theta==@theta and case==@case')
        med=INOC_SUMMARY.query('theta==@theta and case==@case').iloc[0]
        nbetter=int((group.N_RMSE_ratio<1).sum())
        obetter=int((group.other_RMSE_ratio<1).sum())
        lines.append(f'- {INOC_CASES[case][0]}: RMSE N mejora en {nbetter}/3 labs '
            f'(ratio mediano {med.median_N_RMSE_ratio:.3f}); error conjunto GF/E/Gly mejora en '
            f'{obetter}/3 (ratio mediano {med.median_other_RMSE_ratio:.3f}).')
for lab in LABS:
    rows=INOC_FEATURES.query('theta=="theta_full" and lab==@lab').set_index('case')
    fmt=lambda v:f'{v:.2f} h' if np.isfinite(v) else 'sin cruce antes del pulso'
    lines.append(f'- {lab}, theta_full: primer cruce N≤5 mg/L antes del pulso: '
        f'referencia {fmt(rows.loc["referencia_anterior","first_pre_pulse_N_le_5_h"])}; '
        f'7,44 convención {fmt(rows.loc["inoculo_total_convencion","first_pre_pulse_N_le_5_h"])}; '
        f'7,44 viable {fmt(rows.loc["inoculo_viable_consistente","first_pre_pulse_N_le_5_h"])}.')
lines.append('El dato independiente reduce la arbitrariedad de X₀. Una mejora no valida por sí sola '
    'el factor masa/célula, la cinética ni la equivalencia entre biomasa viable y efectiva. '
    'Si persisten diferencias de N, este cambio no las resuelve automáticamente; '
    'revisar también la química, el acoplamiento captación–crecimiento y el balance de aportes. '
    'La comparación no constituye una recalibración y no analiza la salida de CO₂.')
lines.append('La mediana no describe a los tres labs por igual: con theta_full, el nuevo inóculo '
    'mejora el resumen GF/E/Gly en LAB019 y LAB020, pero lo empeora en LAB021. '
    'Además, ambos theta siguen agotando N antes de lo que sugieren varias muestras y no reproducen '
    'el remanente posterior. Por tanto, una X₀ independiente mejora el diagnóstico, pero no resuelve '
    'por sí sola la discrepancia de nitrógeno ni justifica una misma conclusión para los tres labs.')
display(Markdown('\n\n'.join(lines)))

In [ ]:
INOC_TABLES=dict(initial_conditions=INOC_INPUTS,metrics=INOC_METRICS,
    comparison=INOC_COMPARISON,summary=INOC_SUMMARY,depletion_features=INOC_FEATURES,
    checks=INOC_CHECKS,nut_reserved_validation=INOC_NUT,figure_index=pd.DataFrame(INOC_FIGURES))
for name,table in INOC_TABLES.items(): table.to_csv(INOC_OUT/(name+'.csv'),index=False)
inoc_arrays={};inoc_array_manifest=[]
for i,(key,curve) in enumerate(INOC_CURVES.items()):
    name=f'case_{i}';inoc_arrays[name]=np.column_stack([curve.index,curve.to_numpy()])
    inoc_array_manifest.append(dict(array=name,key=list(key),columns=['t_h']+list(curve.columns)))
np.savez_compressed(INOC_OUT/'trajectories.npz',**inoc_arrays)
assert HASH_BEFORE=={str(p.relative_to(ROOT)):sha256(p) for p in PROTECTED}
assert INOC_CHECKS.physically_admissible.all()
assert INOC_CHECKS.inventory_drift_mg_l.max()<.002
assert not INOC_METRICS.loc[INOC_METRICS.state.isin(['X','Xd']),'eligible'].any()
assert INOC_CHECKS.initial_X_error_g_l.max()<1e-12
assert INOC_CHECKS.initial_Xd_error_g_l.max()<1e-12
inoc_manifest=dict(observation_policy=GF_ZERO_POLICY,created_at=datetime.now().astimezone().isoformat(),
    source_code_sha256=sha256(LAB2026/'notebooks/diagnostics/nitrogen_consumption_diagnosis.source.ipynb'),
    arrays=inoc_array_manifest,figures=INOC_FIGURES,provenance=INOC_PROVENANCE,
    theta=THETA_SETS,new_simulations=12,reused_references=6,horizon_h=166.,
    gas_data_loaded=False,optimizer_calls=0,kalman_updates=0,
    original_input_hashes=HASH_BEFORE)
(INOC_OUT/'manifest.json').write_text(json.dumps(inoc_manifest,ensure_ascii=False,indent=2),encoding='utf-8')
manifest['common_inoculum_extension']=dict(path=str(INOC_OUT.relative_to(ROOT)),
    reactor_total_million_cells_ml=INOC_TOTAL_MILLION_ML,new_simulations=12,
    reused_references=6,figures=len(INOC_FIGURES))
(OUT/'manifest.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding='utf-8')
print('Extensión verificada: 12 simulaciones nuevas, 6 referencias, 3 figuras; inputs y parámetros intactos.')

### 16.2. Inóculo viable ponderado por los recuentos relativos de cada lab

Se evalúa la propuesta del usuario sin sustituir los datos originales ni reajustar parámetros. Se toman **los valores viables aportados por el usuario**, 25,04; 13,56 y 12,04 millones de células/mL. Son entradas de esta hipótesis; no se presentan como valores recalculados automáticamente de los CSV anteriores.

**Revisión del cálculo y de las unidades:**

1. El promedio aritmético exacto es (25,04+13,56+12,04)/3 = **16,88**, no 16,89.
2. Con 424,19 millones de células viables/mL de inóculo, 30 mL aportan 12 725,7 millones de células viables. Dividiendo por **2 000 mL de volumen final**, la concentración común esperada es **6,36285 millones de células viables/mL**. El paso intermedio por 100 mL da lo mismo: 424,19×100×0,3/2000. Se supone que 2 L es el volumen final, no 2 L más 30 mL; no se aplica de nuevo la dilución analítica.
3. La ponderación propuesta es
\[
C_{v,0}^{(i)}=C_{v,\mathrm{lab}}^{(i)}\,
\frac{6,36285}{16,88}.
\]
Da **9,43873009; 5,11138898 y 4,53843092 millones de células viables/mL**, respectivamente. Se conserva el promedio 6,36285 y las proporciones entre labs. Las simulaciones usan los valores completos, sin redondearlos a 9,43/5,11/4,54.

**Interpretación:** es un reescalado común de los recuentos. Tiene sentido como escenario si se supone que su escala absoluta está sesgada y que sus diferencias relativas sí representan diferencias reales entre reactores. No demuestra que esas diferencias sean fiables. Si todos recibieron 30 mL de una suspensión homogénea en 2 L finales, el aporte esperado sería común; una ponderación desigual requiere esa hipótesis adicional. El reparto del total nominal entre tres reactores se conserva, pero no se prueba una dosis desigual real.

Como los recuentos **ya son viables**, no se multiplican de nuevo por 85,39 %. Se conserva la conversión exploratoria de 0,03 g/L por millón de células/mL. Se comparan:

- **Referencia anterior:** X₀ y Xd₀ previos de cada reactor, como en 16.1.
- **Viable común · volumen:** X₀ = 6,36285×0,03 = **0,1908855 g/L**, usando ahora los volúmenes proporcionados.
- **Viable ponderada:** X₀ específico de cada lab, según la fórmula anterior.

Para aislar la ponderación de X₀, los dos casos nuevos mantienen **el mismo Xd₀ = (496,77−424,19)×30/2000×0,03 = 0,032661 g/L**. Es el aporte nominal de células no viables del inóculo, supuesto común; no es una medición de Xd en cada reactor ni una ponderación de la muerte. Esta convención se muestra explícitamente en la tabla. El caso común calculado difiere sólo un 0,155 % en X₀ del viable común de 16.1, que partía del 7,44 redondeado y del porcentaje de viabilidad redondeado.

Se mantienen los parámetros antiguos, **ambos theta**, química, temperatura, reloj y todos los pulsos hasta **166 h**. La biomasa posterior no se utiliza como validación fiable. NUT conserva su intervalo temporal provisional; no hay EKF ni CO₂.

In [ ]:
W_OUT=OUT/'inoculum_viable_weighted';W_OUT.mkdir(exist_ok=True)
W_USER_VIABLE={'LAB019':25.04,'LAB020':13.56,'LAB021':12.04}
W_MEAN=float(np.mean(list(W_USER_VIABLE.values())))
W_INOC_VIABLE=424.19;W_INOC_TOTAL=496.77
W_DOSE_ML=30.;W_FINAL_ML=2000.;W_MASS=.03
W_COMMON=W_INOC_VIABLE*W_DOSE_ML/W_FINAL_ML
W_SCALE=W_COMMON/W_MEAN
W_WEIGHTED={lab:value*W_SCALE for lab,value in W_USER_VIABLE.items()}
W_DEAD=(W_INOC_TOTAL-W_INOC_VIABLE)*W_DOSE_ML/W_FINAL_ML*W_MASS
assert np.isclose(W_MEAN,16.88,rtol=0,atol=1e-12)
assert np.isclose(W_COMMON,6.36285,rtol=0,atol=1e-12)
assert np.isclose(np.mean(list(W_WEIGHTED.values())),W_COMMON,rtol=0,atol=1e-12)
assert np.isclose(sum(W_WEIGHTED.values())*W_FINAL_ML,3*W_INOC_VIABLE*W_DOSE_ML)
assert np.isclose(W_COMMON,W_INOC_VIABLE*100*.3/W_FINAL_ML)
W_PROVENANCE=dict(user_viable_million_cells_ml=W_USER_VIABLE,mean_viable=W_MEAN,
    viable_inoculum_million_cells_ml=W_INOC_VIABLE,total_inoculum_million_cells_ml=W_INOC_TOTAL,
    inoculum_volume_ml_per_reactor=W_DOSE_ML,assumed_final_volume_ml=W_FINAL_ML,
    common_viable_million_cells_ml=W_COMMON,scale=W_SCALE,weighted_viable_million_cells_ml=W_WEIGHTED,
    mass_factor_g_l_per_million_cells_ml=W_MASS,common_Xd_g_l=W_DEAD,
    source='User-provided viable lab counts and inoculum volumes; normalization hypothesis',
    no_second_viability_correction=True,old_data_changed=False,
    assumptions=['2 L is final volume','Relative viable lab ratios meaningful',
                 'Equal cell mass conversion','Nonviable inoculum contribution common'])
W_CASES={
    'referencia_anterior':('Referencia anterior','#2166ac','-'),
    'viable_comun_volumen':('Viable común','#00897b','--'),
    'viable_ponderada':('Viable ponderada','#e66101','-.'),
}
W_CURVES={};W_INPUT_ROWS=[];W_METRIC_ROWS=[];W_CHECK_ROWS=[];W_FEATURE_ROWS=[];W_NUT_ROWS=[]
for lab in LABS:
    original=BATCHES[lab]
    common=replace(original,initials={**original.initials,'X':W_COMMON*W_MASS,'Xd':W_DEAD})
    weighted=replace(original,initials={**original.initials,'X':W_WEIGHTED[lab]*W_MASS,'Xd':W_DEAD})
    W_INPUT_ROWS.append(dict(lab=lab,user_viable_million_ml=W_USER_VIABLE[lab],mean_viable_million_ml=W_MEAN,
        relative_weight=W_USER_VIABLE[lab]/W_MEAN,common_viable_million_ml=W_COMMON,
        weighted_viable_million_ml=W_WEIGHTED[lab],weighted_X0_g_l=weighted.initials['X'],
        common_X0_g_l=common.initials['X'],new_common_Xd0_g_l=W_DEAD,
        previous_X0_g_l=original.initials['X'],previous_Xd0_g_l=original.initials['Xd'],
        common_X_ratio_to_section_16_1=common.initials['X']/(.2232*.8539),
        N_pulses=repr(original.pulses['N']),end_h=float(original.time[-1])))
    points=POINTS[POINTS.lab.eq(lab)]
    pulse=min(t for t,d in original.pulses['N'])
    alternatives={'referencia_anterior':original,'viable_comun_volumen':common,'viable_ponderada':weighted}
    for case,batch in alternatives.items():
        assert all(batch.initials[s]==original.initials[s] for s in ['N','G','F','E','Gly'])
        assert batch.pulses==original.pulses
        assert np.array_equal(batch.time,original.time)
        assert np.array_equal(batch.temperature_c,original.temperature_c)
        for theta,th in THETA_SETS.items():
            curve=CURVES[(lab,theta,'referencia')] if case=='referencia_anterior' else diagnosis.run(batch,th,'referencia')
            W_CURVES[(lab,theta,case)]=curve
            for row in diagnosis.score(curve,points,pulse):
                W_METRIC_ROWS.append(dict(lab=lab,theta=theta,case=case,**row))
            W_CHECK_ROWS.append(dict(lab=lab,theta=theta,case=case,
                initial_X_error_g_l=abs(curve.X.iloc[0]-batch.initials['X']),
                initial_Xd_error_g_l=abs(curve.Xd.iloc[0]-batch.initials['Xd']),
                inventory_drift_mg_l=1000*curve.attrs['inventory_drift'],jump_error=curve.attrs['jump_error'],
                physically_admissible=curve.attrs['physically_admissible'],
                start_h=float(curve.index[0]),end_h=float(curve.index[-1]),
                **{'raw_min_'+s:v for s,v in curve.attrs['raw_min'].items()},
                **{'raw_max_'+s:v for s,v in curve.attrs['raw_max'].items()}))
            pre=curve[curve.index<=pulse];after=curve[curve.index>pulse]
            W_FEATURE_ROWS.append(dict(lab=lab,theta=theta,case=case,
                first_pre_pulse_N_le_5_h=diagnosis.first_crossing(pre.index,pre.N),
                first_post_pulse_N_le_5_h=diagnosis.first_crossing(after.index,after.N),
                initial_uptake_mg_l_h=float(curve.uptake_mg_l_h.iloc[0]),
                final_YAN_mg_l=1000*curve.N_pred.iloc[-1],**diagnosis.observed_crossing(points,pulse)))
            assert curve.index[0]==0. and curve.index[-1]==166.
            assert curve.attrs['physically_admissible'],(lab,theta,case,curve.attrs['raw_min'])
            for event,dose in batch.pulses['N']:
                assert abs(curve.loc[event+diagnosis.CONFIG['after_event_h'],'N']-curve.loc[event,'N']-dose)<1e-5
            nut=NUT_SAMPLES[NUT_SAMPLES.lab.eq(lab)].iloc[0]
            support=np.unique(np.r_[nut.time_lower_h,nut.time_upper_h,
                curve.index[(curve.index>=nut.time_lower_h)&(curve.index<=nut.time_upper_h)]])
            pred=1000*diagnosis.interpolate_post_action(curve,support,batch.pulses['N'])
            W_NUT_ROWS.append(dict(lab=lab,theta=theta,case=case,observed_mg_l=nut.YAN_mg_l,
                lower_h=nut.time_lower_h,upper_h=nut.time_upper_h,pred_min_mg_l=float(pred.min()),
                pred_max_mg_l=float(pred.max()),time_provisional=True))
W_INPUTS=pd.DataFrame(W_INPUT_ROWS);W_METRICS=pd.DataFrame(W_METRIC_ROWS)
W_CHECKS=pd.DataFrame(W_CHECK_ROWS);W_FEATURES=pd.DataFrame(W_FEATURE_ROWS);W_NUT=pd.DataFrame(W_NUT_ROWS)
display(W_INPUTS.round(6))
print('Promedio viable:',W_MEAN,'; aporte común:',W_COMMON,'; factor común:',W_SCALE)
print('X₀ ponderadas [g/L]:',{lab:value*W_MASS for lab,value in W_WEIGHTED.items()})
print('12 simulaciones nuevas + 6 referencias; sin nueva corrección de viabilidad.')

#### 16.2.1. Las mismas tres figuras con la biomasa viable ponderada

Las dos primeras figuras muestran N, X, GF y E para los tres labs, una por theta. La tercera muestra los residuos de N para ambos theta: predicción menos medición; cero significa coincidencia. **Azul:** referencia anterior; **verde discontinuo:** aporte viable común calculado con los volúmenes; **naranja punto-raya:** viable ponderada. NUT y los pulsos conservan la misma convención que 16.1. Las mediciones huecas de biomasa son un proxy incierto.

Comparar naranja con verde aísla el efecto de conservar las diferencias relativas entre labs. Comparar cualquiera con azul incluye también la corrección de la escala anterior. No se seleccionan pesos para mejorar el ajuste.

Se agregaron los puntos iniciales de biomasa de los escenarios verde y naranja en t=0. Se muestran como círculos normales, sin leyendas adicionales ni recuadros dentro del panel. Los valores de X₀ están en las tablas de esta sección. La curva azul conserva la referencia anterior y los círculos huecos corresponden a las mediciones posteriores de biomasa. Las condiciones iniciales añadidas no se cuentan como mediciones independientes en RMSE.

In [ ]:
W_FIGURES=[]
def save_weighted(fig,name):
    path=W_OUT/(name+'.png')
    fig.savefig(path,dpi=125,bbox_inches='tight',facecolor='white')
    buffer=io.BytesIO();fig.savefig(buffer,format='png',dpi=105,bbox_inches='tight',facecolor='white')
    plt.close(fig);display(Image(data=buffer.getvalue()));buffer.close()
    W_FIGURES.append(dict(name=name,path=str(path.relative_to(ROOT))))
for theta in THETA_SETS:
    fig,axes=plt.subplots(3,4,figsize=(17,9),layout='constrained')
    for i,lab in enumerate(LABS):
        for j,state in enumerate(['N','X','GF','E']):
            ax=axes[i,j];scale=1000. if state=='N' else 1.
            for case,(label,color,style) in W_CASES.items():
                curve=W_CURVES[(lab,theta,case)]
                ax.plot(curve.index,(curve.N_pred if state=='N' else curve[state])*scale,
                    color=color,ls=style,lw=1.7,label=label)
            if state=='X':
                draw_new_biomass_initials(ax,lab,theta,W_CURVES,W_CASES)
            else:
                draw_observations(ax,lab,state,scale)
            mark_events(ax,lab)
            ax.set(title=f'{lab} · {state}\n{theta}',xlabel='t desde inoculación [h]',
                ylabel='mg N/L' if state=='N' else 'g/L',xlim=(-3,166) if state=='X' else (0,166))
    common_legend(fig,axes,ncol=3);save_weighted(fig,'01_quimica_'+theta)
fig,axes=plt.subplots(3,2,figsize=(13,9),layout='constrained')
for i,lab in enumerate(LABS):
    for j,theta in enumerate(THETA_SETS):
        ax=axes[i,j]
        pts=POINTS.loc[POINTS.lab.eq(lab)&POINTS.state.eq('N')&~POINTS.initial_anchor]
        for case,(label,color,style) in W_CASES.items():
            curve=W_CURVES[(lab,theta,case)]
            residual=1000*(np.interp(pts.t_h,curve.index,curve.N_pred)-pts.observed)
            ax.plot(pts.t_h,residual,marker='o',ms=3,color=color,ls=style,label=label)
        ax.axhline(0,color='black',lw=.8);mark_events(ax,lab)
        ax.set(title=f'{lab} · residuo N · {theta}',xlabel='t desde inoculación [h]',
            ylabel='Predicción − YAN [mg N/L]',xlim=(0,166))
common_legend(fig,axes,ncol=2);save_weighted(fig,'02_residuos_N_ambos_theta')

#### 16.2.2. Métricas, agotamiento y lectura por lab

Las métricas usan exactamente las mismas muestras posteriores al ancla inicial en los tres casos. N se presenta en mg N/L, GF/E/Gly en g/L. El resumen no-N usa GF/E/Gly, excluyendo biomasa y sin contar GF junto a sus componentes. Se compara tanto con la referencia anterior como con el aporte viable común: un ratio menor que uno indica menor error. Se muestran resultados individuales, porque una mediana de tres labs puede ocultar un empeoramiento.

El agotamiento operativo corresponde al primer N≤5 mg/L. Las horas son de la malla del modelo, no horas experimentales exactas. NUT se contrasta sobre su intervalo post-pulso provisional, sin incorporarlo a RMSE como un punto de hora conocida. Mejorar el error no convierte la hipótesis de reescalado en una medición de biomasa efectiva.

In [ ]:
W_COMPARE_ROWS=[]
for (lab,theta,case),group in W_METRICS.query('period=="completo"').groupby(['lab','theta','case']):
    row=dict(lab=lab,theta=theta,case=case)
    for state in ['N','GF','E','Gly']:
        item=group[group.state.eq(state)].iloc[0];scale=1000. if state=='N' else 1.
        row[state+'_RMSE']=item.RMSE*scale;row[state+'_bias']=item.bias*scale;row[state+'_n']=int(item.n)
    other=group[group.state.isin(['GF','E','Gly'])&group.eligible]
    row['other_weighted_RMSE']=float(np.sqrt(other.WSSE.sum()/other.n.sum()))
    W_COMPARE_ROWS.append(row)
W_COMPARISON=pd.DataFrame(W_COMPARE_ROWS)
for reference,suffix in [('referencia_anterior','previous'),('viable_comun_volumen','common')]:
    ref=W_COMPARISON[W_COMPARISON.case.eq(reference)][['lab','theta','N_RMSE','other_weighted_RMSE']]
    ref=ref.rename(columns={'N_RMSE':'N_RMSE_'+suffix,'other_weighted_RMSE':'other_weighted_RMSE_'+suffix})
    W_COMPARISON=W_COMPARISON.merge(ref,on=['lab','theta'],validate='many_to_one')
    W_COMPARISON['N_ratio_'+suffix]=W_COMPARISON.N_RMSE/W_COMPARISON['N_RMSE_'+suffix]
    W_COMPARISON['other_ratio_'+suffix]=W_COMPARISON.other_weighted_RMSE/W_COMPARISON['other_weighted_RMSE_'+suffix]
display(W_COMPARISON[['lab','theta','case','N_RMSE','GF_RMSE','E_RMSE','Gly_RMSE',
    'N_ratio_previous','other_ratio_previous','N_ratio_common','other_ratio_common']].round(3))
display(W_FEATURES.round(3));display(W_NUT.round(3))
W_SUMMARY=W_COMPARISON.groupby(['theta','case']).agg(
    median_N_ratio_previous=('N_ratio_previous','median'),median_other_ratio_previous=('other_ratio_previous','median'),
    median_N_ratio_common=('N_ratio_common','median'),median_other_ratio_common=('other_ratio_common','median')).reset_index()
display(W_SUMMARY.round(3))
w_lines=['**Resultados calculados: ponderación frente al aporte viable común.**']
for theta in THETA_SETS:
    w_lines.append('**'+theta+'**')
    group=W_COMPARISON.query('theta==@theta and case=="viable_ponderada"')
    for row in group.itertuples():
        nword='mejora' if row.N_ratio_common<1 else 'empeora'
        oword='mejora' if row.other_ratio_common<1 else 'empeora'
        w_lines.append(f'- {row.lab}: N {nword}, ratio {row.N_ratio_common:.3f}; '
            f'GF/E/Gly {oword}, ratio {row.other_ratio_common:.3f}. '
            f'RMSE N = {row.N_RMSE:.2f} mg N/L.')
for lab in LABS:
    f=W_FEATURES.query('theta=="theta_full" and lab==@lab').set_index('case')
    fmt=lambda v:f'{v:.2f} h' if np.isfinite(v) else 'sin cruce antes del pulso'
    w_lines.append(f'- {lab}, theta_full: N≤5 mg/L antes del pulso: referencia '
        f'{fmt(f.loc["referencia_anterior","first_pre_pulse_N_le_5_h"])}; común '
        f'{fmt(f.loc["viable_comun_volumen","first_pre_pulse_N_le_5_h"])}; ponderada '
        f'{fmt(f.loc["viable_ponderada","first_pre_pulse_N_le_5_h"])}.')
w_lines.append('La ponderación aumenta X₀ de LAB019 respecto al caso común y reduce X₀ de LAB020/021. '
    'Por eso puede acelerar el consumo en LAB019 y retrasarlo en los otros, aunque el promedio de X₀ '
    'se conserve. No cambia qN ni corrige automáticamente el remanente de YAN. '
    'Las diferencias de ajuste describen una hipótesis; no prueban la fiabilidad de los pesos relativos '
    'ni identifican una distribución real del inóculo. No se han ajustado parámetros usando estas muestras.')
display(Markdown('\n\n'.join(w_lines)))

In [ ]:
W_TABLES=dict(initial_conditions=W_INPUTS,metrics=W_METRICS,comparison=W_COMPARISON,
    summary=W_SUMMARY,depletion_features=W_FEATURES,checks=W_CHECKS,
    nut_reserved_validation=W_NUT,figure_index=pd.DataFrame(W_FIGURES))
for name,table in W_TABLES.items(): table.to_csv(W_OUT/(name+'.csv'),index=False)
(W_OUT/'input_provenance.json').write_text(json.dumps(W_PROVENANCE,ensure_ascii=False,indent=2),encoding='utf-8')
w_arrays={};w_array_manifest=[]
for i,(key,curve) in enumerate(W_CURVES.items()):
    name=f'case_{i}';w_arrays[name]=np.column_stack([curve.index,curve.to_numpy()])
    w_array_manifest.append(dict(array=name,key=list(key),columns=['t_h']+list(curve.columns)))
np.savez_compressed(W_OUT/'trajectories.npz',**w_arrays)
assert HASH_BEFORE=={str(p.relative_to(ROOT)):sha256(p) for p in PROTECTED}
assert W_CHECKS.physically_admissible.all()
assert W_CHECKS.inventory_drift_mg_l.max()<.002 and W_CHECKS.jump_error.max()<1e-12
assert W_CHECKS.initial_X_error_g_l.max()<1e-12 and W_CHECKS.initial_Xd_error_g_l.max()<1e-12
assert not W_METRICS.loc[W_METRICS.state.isin(['X','Xd']),'eligible'].any()
w_manifest=dict(observation_policy=GF_ZERO_POLICY,created_at=datetime.now().astimezone().isoformat(),
    source_code_sha256=sha256(LAB2026/'notebooks/diagnostics/nitrogen_consumption_diagnosis.source.ipynb'),
    arrays=w_array_manifest,figures=W_FIGURES,provenance=W_PROVENANCE,theta=THETA_SETS,
    new_simulations=12,reused_references=6,horizon_h=166.,gas_data_loaded=False,
    optimizer_calls=0,kalman_updates=0,original_input_hashes=HASH_BEFORE)
(W_OUT/'manifest.json').write_text(json.dumps(w_manifest,ensure_ascii=False,indent=2),encoding='utf-8')
manifest['weighted_viable_extension']=dict(path=str(W_OUT.relative_to(ROOT)),new_simulations=12,
    reused_references=6,figures=len(W_FIGURES),mean_viable_million_cells_ml=W_COMMON)
(OUT/'manifest.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding='utf-8')
print('Sección 16.2 verificada: 12 simulaciones nuevas + 6 referencias; 3 figuras; normalización y balances conservados.')

## 17. Errores por variable y por tramo: dónde mejora cada inicialización

Se profundiza la comparación de 16.2 para **LAB019–021 y ambos theta**, manteniendo los mismos datos y la simulación completa hasta 166 h. Azul: referencia anterior; verde: viable común calculada con los volúmenes; naranja: viable ponderada. Los X₀ nuevos son entradas, no puntos independientes para validar.

Se muestran residuos **predicción menos medición** de N/YAN, GF y E. Un residuo negativo en N significa subpredicción de YAN; uno positivo en GF significa demasiado azúcar restante; uno negativo en E significa poco etanol. Las líneas unen muestras y no representan un error conocido continuamente. No se puntúa biomasa incierta, no se cuenta GF junto con G y F y se excluye el ancla inicial de los errores. NUT mantiene su contraste por intervalo de las secciones anteriores y no se convierte en una muestra de hora exacta.

Además del horizonte completo se distinguen **0–30 h**, **30 h hasta el pulso (inclusive)** y **después del pulso**. Las 30 h son una división descriptiva común, no una detección del fin del lag. Las figuras conservan toda la curva. RMSE, sesgo, MAE y número de muestras se calculan sobre el mismo soporte en los tres casos. Hay pocas muestras de E; una mejora de RMSE no implica significación estadística.

El GF final de LAB019 se incluye como 0 g/L de trabajo, como un punto normal al final de la serie de residuos. Su procedencia negativa y el flag se conservan; el RMSE incluye esta aproximación con la misma convención de peso de GF usada en el resto del análisis.

In [ ]:
E_OUT=OUT/'error_mechanism_diagnostics';E_OUT.mkdir(exist_ok=True)
E_STATES=('N','GF','E');E_EARLY_H=30.;E_FIGURES=[];E_CURVES={};E_CHECK_ROWS=[];E_SIMULATIONS=0
E_PHASES=('temprana_0_30h','tardia_pre_pulso','post_pulso')
E_LABELS={'referencia_anterior':'Azul · anterior','viable_comun_volumen':'Verde · común','viable_ponderada':'Naranja · ponderada'}
def e_save(fig,name):
    path=E_OUT/(name+'.png');fig.savefig(path,dpi=125,bbox_inches='tight',facecolor='white')
    buffer=io.BytesIO();fig.savefig(buffer,format='png',dpi=105,bbox_inches='tight',facecolor='white')
    plt.close(fig);display(Image(data=buffer.getvalue()));buffer.close()
    E_FIGURES.append(dict(name=name,path=str(path.relative_to(ROOT))))
def e_points(lab,state):
    return POINTS.loc[POINTS.lab.eq(lab)&POINTS.state.eq(state)&~POINTS.initial_anchor].sort_values('t_h')
def e_errors(curve,lab):
    rows=[]
    pulse=min(t for t,d in BATCHES[lab].pulses['N'])
    for state in E_STATES:
        points=e_points(lab,state)
        column='N_pred' if state=='N' else state;scale=1000. if state=='N' else 1.
        pred=np.interp(points.t_h,curve.index,curve[column])
        for t,y,p,sigma,zero in zip(points.t_h,points.observed,pred,points.sigma,points.zero_substitution):
            phase='temprana_0_30h' if t<=E_EARLY_H else 'tardia_pre_pulso' if t<=pulse else 'post_pulso'
            rows.append(dict(state=state,t_h=float(t),observed=float(y*scale),predicted=float(p*scale),
                residual=float((p-y)*scale),phase=phase,unit='mg N/L' if state=='N' else 'g/L',
                sigma_weight=float(sigma*scale),zero_substitution=bool(zero)))
    return pd.DataFrame(rows)
def e_score(curve,lab):
    data=e_errors(curve,lab);result={}
    for state,group in data.groupby('state'):
        r=group.residual.to_numpy();result[state+'_RMSE']=float(np.sqrt(np.mean(r*r)))
        result[state+'_bias']=float(r.mean());result[state+'_n']=len(r)
    return result
def e_register(key,curve,th):
    E_CURVES[key]=curve
    E_CHECK_ROWS.append(dict(key=json.dumps(list(key)),inventory_drift_mg_l=1000*curve.attrs['inventory_drift'],
        jump_error=curve.attrs['jump_error'],physically_admissible=curve.attrs['physically_admissible'],
        start_h=float(curve.index[0]),end_h=float(curve.index[-1]),
        qN=float(th['qN']),mu0=float(th['mu0']),qN_over_mu0=float(th['qN']/th['mu0']),
        **{'raw_min_'+s:v for s,v in curve.attrs['raw_min'].items()},
        **{'raw_max_'+s:v for s,v in curve.attrs['raw_max'].items()}))
    assert curve.index[0]==0. and curve.index[-1]==166.
    assert curve.attrs['inventory_drift']<2e-6 and curve.attrs['jump_error']<1e-12
E_RESIDUAL_ROWS=[]
for (lab,theta,case),curve in W_CURVES.items():
    e_register(('initialization',lab,theta,case),curve,THETA_SETS[theta])
    data=e_errors(curve,lab);data['lab']=lab;data['theta']=theta;data['case']=case
    E_RESIDUAL_ROWS.append(data)
E_RESIDUALS=pd.concat(E_RESIDUAL_ROWS,ignore_index=True)
E_PHASE_ROWS=[]
for (lab,theta,case,state),group in E_RESIDUALS.groupby(['lab','theta','case','state']):
    for phase in ('completo',*E_PHASES):
        g=group if phase=='completo' else group[group.phase.eq(phase)]
        if g.empty: continue
        r=g.residual.to_numpy()
        E_PHASE_ROWS.append(dict(lab=lab,theta=theta,case=case,state=state,phase=phase,n=len(g),
            RMSE=float(np.sqrt(np.mean(r*r))),bias=float(r.mean()),MAE=float(np.mean(abs(r))),
            first_sample_h=float(g.t_h.min()),last_sample_h=float(g.t_h.max()),unit=g.unit.iloc[0]))
E_PHASE_METRICS=pd.DataFrame(E_PHASE_ROWS)
e_ref=E_PHASE_METRICS.query('case=="referencia_anterior"')[['lab','theta','state','phase','RMSE']].rename(columns={'RMSE':'RMSE_reference'})
E_PHASE_METRICS=E_PHASE_METRICS.merge(e_ref,on=['lab','theta','state','phase'],validate='many_to_one')
E_PHASE_METRICS['ratio_to_previous']=E_PHASE_METRICS.RMSE/E_PHASE_METRICS.RMSE_reference.replace(0,np.nan)
for theta in THETA_SETS:
    fig,axes=plt.subplots(3,3,figsize=(15,9),layout='constrained')
    for i,lab in enumerate(LABS):
        for j,state in enumerate(E_STATES):
            ax=axes[i,j]
            for case,(label,color,style) in W_CASES.items():
                g=E_RESIDUALS.query('lab==@lab and theta==@theta and case==@case and state==@state')
                ax.plot(g.t_h,g.residual,color=color,ls=style,marker='o',ms=3,label=label)
            ax.axhline(0,color='black',lw=.8);ax.axvline(E_EARLY_H,color='gray',ls='--',lw=.8,label='División descriptiva 30 h')
            mark_events(ax,lab)
            ax.set(title=f'{lab} · residuo {state}\n{theta}',xlabel='t desde inoculación [h]',
                ylabel='Predicción − medición ['+('mg N/L' if state=='N' else 'g/L')+']',xlim=(0,166))
    common_legend(fig,axes,ncol=3);e_save(fig,'01_residuos_'+theta)
    fig,axes=plt.subplots(1,3,figsize=(15,8),layout='constrained')
    row_keys=[(lab,case) for lab in LABS for case in W_CASES]
    for ax,phase in zip(axes,E_PHASES):
        ratios=np.full((9,3),np.nan)
        for i,(lab,case) in enumerate(row_keys):
            for j,state in enumerate(E_STATES):
                g=E_PHASE_METRICS.query('lab==@lab and theta==@theta and case==@case and state==@state and phase==@phase')
                if len(g): ratios[i,j]=g.ratio_to_previous.iloc[0]
        im=ax.imshow(np.log2(ratios),vmin=-2,vmax=2,cmap='coolwarm',aspect='auto');ax.grid(False)
        for (i,j),value in np.ndenumerate(ratios):
            ax.text(j,i,f'{value:.2f}' if np.isfinite(value) else 'sin datos',ha='center',va='center',fontsize=8,color='white' if np.isfinite(value) and abs(np.log2(value))>1.25 else 'black')
        ax.set_xticks(range(3),E_STATES);ax.set_yticks(range(9),[lab+' · '+E_LABELS[case] for lab,case in row_keys])
        ax.set_title(phase.replace('_',' ')+'\n'+theta)
    cb=fig.colorbar(im,ax=axes,shrink=.65,extend='both');cb.set_ticks([-2,-1,0,1,2],labels=['0,25','0,5','1','2','4'])
    cb.set_label('RMSE / referencia anterior · menor que 1 mejora')
    e_save(fig,'02_errores_por_tramo_'+theta)
display(E_PHASE_METRICS.query('theta=="theta_full"')[['lab','case','state','phase','n','RMSE','bias']].round(3))

## 18. Balances frente al azúcar consumido: separar reloj y proporciones

Se construyen pares de **mediciones químicas con el mismo tiempo**, sin interpolar datos ausentes. Se descuenta el ancla inicial y, para N, se suman los aportes registrados anteriores a cada muestra:
\[
\Delta GF=GF_0-GF(t),\quad N_{cons}=N_0+N_{aportado}(t)-N(t),\quad E_{prod}=E(t)-E_0.
\]
El consumo acumulado de N no es simplemente N₀−N cuando hubo nutrición. Se conserva la convención de muestra antes de la acción si coincide exactamente con el pulso; NUT no tiene pareja de GF a una hora cierta y no entra en estos diagramas. Las líneas del modelo recorren **toda la fermentación**, corrigiendo los saltos de N por el aporte. Los puntos negros son balances observados, no una nueva medición independiente.

Representar N consumido o E producido frente a GF consumido reduce la dependencia explícita del reloj, pero **no elimina la influencia de X, las inhibiciones o la nutrición**. Un desfase temporal y una proporción incorrecta pueden coexistir. La tabla de balances mide el residuo de N/E en los tiempos emparejados; no interpola el modelo a un GF fijo ni mide distancia geométrica a la línea.

Como complemento se calculan tasas **medias entre muestras emparejadas**. Se marcan los intervalos que cruzan un pulso y se excluyen del resumen de tasas los intervalos menores que 6 h para evitar dar demasiado peso a diferencias de muestras muy próximas. El umbral es descriptivo; no es un filtro validado ni una estimación de tasas instantáneas. Se conservan todos los intervalos y su elegibilidad en el CSV.

LAB019 incorpora ahora la pareja final de 166 h usando GF=0 g/L como aproximación explícita. Los balances que dependen de ese valor llevan el flag `GF_zero_substitution` en la tabla y se muestran como los demás puntos en la figura; las tasas del último intervalo también utilizan ese valor de trabajo.

In [ ]:
def e_added(lab,t):
    t=np.asarray(t,float)
    return np.array([sum(d for event,d in BATCHES[lab].pulses['N'] if event<ti-1e-9) for ti in t.ravel()]).reshape(t.shape)
def e_anchor(lab,state):
    return float(POINTS.loc[POINTS.lab.eq(lab)&POINTS.state.eq(state)&POINTS.initial_anchor,'observed'].iloc[0])
E_BALANCE_ROWS=[];E_PAIR_ROWS=[];E_RATE_ROWS=[]
for lab in LABS:
    gf=POINTS.loc[POINTS.lab.eq(lab)&POINTS.state.eq('GF'),['t_h','observed']].rename(columns={'observed':'GF_observed'})
    assert gf.t_h.is_unique
    for state in ('N','E'):
        chem=POINTS.loc[POINTS.lab.eq(lab)&POINTS.state.eq(state),['t_h','observed']].rename(columns={'observed':state+'_observed'})
        paired=gf.merge(chem,on='t_h',validate='one_to_one').sort_values('t_h')
        assert paired.t_h.iloc[0]==0. and paired.t_h.is_unique
        scale=1000. if state=='N' else 1.
        paired['GF_consumed']=e_anchor(lab,'GF')-paired.GF_observed
        paired['balance_observed']=(e_anchor(lab,'N')+e_added(lab,paired.t_h)-paired.N_observed)*scale if state=='N' else paired.E_observed-e_anchor(lab,'E')
        for row in paired.itertuples():
            E_PAIR_ROWS.append(dict(lab=lab,state=state,t_h=row.t_h,GF_consumed_g_l=row.GF_consumed,
                balance_observed=row.balance_observed,unit='mg N/L' if state=='N' else 'g/L',initial_anchor=row.t_h==0.,GF_zero_substitution=bool(lab=='LAB019' and row.t_h==166.)))
        for theta in THETA_SETS:
            for case in W_CASES:
                curve=W_CURVES[(lab,theta,case)]
                pred=np.interp(paired.t_h,curve.index,curve.N_pred if state=='N' else curve.E)
                bp=(curve.N_pred.iloc[0]+e_added(lab,paired.t_h)-pred)*scale if state=='N' else pred-curve.E.iloc[0]
                gp=curve.GF.iloc[0]-np.interp(paired.t_h,curve.index,curve.GF)
                for k,row in enumerate(paired.itertuples()):
                    E_BALANCE_ROWS.append(dict(lab=lab,theta=theta,case=case,state=state,t_h=row.t_h,
                        observed_GF_consumed_g_l=row.GF_consumed,predicted_GF_consumed_g_l=gp[k],
                        observed_balance=row.balance_observed,predicted_balance=bp[k],
                        residual_balance=float(bp[k]-row.balance_observed),initial_anchor=row.t_h==0.,GF_zero_substitution=bool(lab=='LAB019' and row.t_h==166.)))
                for k in range(1,len(paired)):
                    a=paired.iloc[k-1];b=paired.iloc[k];dt=float(b.t_h-a.t_h)
                    dose=float(e_added(lab,np.array([b.t_h]))[0]-e_added(lab,np.array([a.t_h]))[0])
                    gfo=(a.GF_observed-b.GF_observed)/dt
                    gfp=(gp[k]-gp[k-1])/dt
                    observed=(b.balance_observed-a.balance_observed)/dt
                    predicted=(bp[k]-bp[k-1])/dt
                    E_RATE_ROWS.append(dict(lab=lab,theta=theta,case=case,state=state,start_h=a.t_h,end_h=b.t_h,
                        duration_h=dt,crosses_pulse=bool(dose>0),N_added_g_l=dose,eligible_duration=dt>=6.,
                        observed_rate=observed,predicted_rate=predicted,residual_rate=predicted-observed,
                        observed_GF_consumption_rate_g_l_h=gfo,predicted_GF_consumption_rate_g_l_h=gfp,
                        unit='mg N/L/h' if state=='N' else 'g/L/h'))
E_PAIRS=pd.DataFrame(E_PAIR_ROWS);E_BALANCES=pd.DataFrame(E_BALANCE_ROWS);E_RATES=pd.DataFrame(E_RATE_ROWS)
assert len(E_PAIRS)==len(E_PAIRS.drop_duplicates(['lab','state','t_h']))
for theta in THETA_SETS:
    fig,axes=plt.subplots(3,2,figsize=(13,9),layout='constrained')
    for i,lab in enumerate(LABS):
        for j,state in enumerate(('N','E')):
            ax=axes[i,j]
            for case,(label,color,style) in W_CASES.items():
                curve=W_CURVES[(lab,theta,case)]
                gf=curve.GF.iloc[0]-curve.GF
                balance=1000*(curve.N_pred.iloc[0]+e_added(lab,curve.index)-curve.N_pred) if state=='N' else curve.E-curve.E.iloc[0]
                ax.plot(gf,balance,color=color,ls=style,label=label)
                for event,dose in BATCHES[lab].pulses['N']:
                    k=np.flatnonzero(np.isclose(curve.index,event,atol=1e-10,rtol=0))[0]
                    ax.scatter([gf.iloc[k]],[balance.iloc[k]],color=color,marker='|',s=65,label='Posición del pulso')
            pairs=E_PAIRS.query('lab==@lab and state==@state')
            ax.scatter(pairs.GF_consumed_g_l,pairs.balance_observed,color='black',s=25,marker='o',zorder=8,label='Datos')
            for row in pairs.loc[~pairs.initial_anchor].iloc[::2].itertuples():
                ax.annotate(f'{row.t_h:g} h',(row.GF_consumed_g_l,row.balance_observed),xytext=(3,4),textcoords='offset points',fontsize=7)
            ax.set(title=f'{lab} · {state} frente a GF\n{theta}',xlabel='GF consumido desde t₀ [g/L]',
                ylabel='N consumido, corregido por aportes [mg N/L]' if state=='N' else 'E producido desde t₀ [g/L]')
    common_legend(fig,axes,ncol=3);e_save(fig,'03_balances_vs_azucar_'+theta)
E_BALANCE_SUMMARY=E_BALANCES.loc[~E_BALANCES.initial_anchor].groupby(['lab','theta','case','state']).agg(
    n=('residual_balance','size'),bias=('residual_balance','mean'),RMSE=('residual_balance',lambda x:float(np.sqrt(np.mean(x*x))))).reset_index()
E_RATE_SUMMARY=E_RATES.loc[E_RATES.eligible_duration].groupby(['lab','theta','case','state']).agg(
    n_intervals=('residual_rate','size'),bias_rate=('residual_rate','mean'),
    RMSE_rate=('residual_rate',lambda x:float(np.sqrt(np.mean(x*x))))).reset_index()
display(E_BALANCE_SUMMARY.query('theta=="theta_full"').round(3))
display(E_RATE_SUMMARY.query('theta=="theta_full"').round(3))

## 19. ¿Qué biomasa inicial prefiere cada variable con parámetros fijos?

Se barre X₀ entre **0,08 y 0,92 g/L**, incluyendo exactamente los X₀ de referencia, común y ponderados. La malla incluye 22 puntos uniformes más las inicializaciones existentes. Xd₀ permanece en el aporte común del inóculo; se mantienen química, temperatura, pulsos y parámetros antiguos. Las líneas verticales muestran las tres inicializaciones de cada lab.

Se calculan por separado RMSE de N, GF y E sobre las mismas muestras posteriores al inicio. Los mínimos son **mínimos de una malla diagnóstica**, no estimaciones calibradas, intervalos de confianza ni una prueba de identificabilidad. Un mínimo en un extremo pide ampliar o reconsiderar el rango; no se declara un óptimo interior. Los escenarios físicamente inadmisibles se conservan auditados pero se excluyen de la selección del mínimo.

Si distintas variables prefieren X₀ diferentes, cambiar solamente la inicialización no ofrece una corrección simultánea evidente. La conclusión es condicional a estos parámetros, datos y rango; no prueba que ninguna variante del modelo pueda ajustar las variables. No se reemplazan las mediciones por el valor que minimiza el error.

In [ ]:
E_X_GRID=np.unique(np.r_[np.linspace(.08,.92,22),W_COMMON*W_MASS,
    [W_WEIGHTED[lab]*W_MASS for lab in LABS],[BATCHES[lab].initials['X'] for lab in LABS]])
E_X_ROWS=[]
for theta,th in THETA_SETS.items():
    for lab in LABS:
        for x0 in E_X_GRID:
            cached=None
            for case,x in [('viable_comun_volumen',W_COMMON*W_MASS),('viable_ponderada',W_WEIGHTED[lab]*W_MASS)]:
                if np.isclose(x0,x,rtol=0,atol=1e-14): cached=W_CURVES[(lab,theta,case)];break
            batch=replace(BATCHES[lab],initials={**BATCHES[lab].initials,'X':float(x0),'Xd':W_DEAD})
            curve=cached if cached is not None else diagnosis.run(batch,th,'referencia')
            E_SIMULATIONS+=int(cached is None)
            e_register(('X0_scan',lab,theta,float(x0)),curve,th)
            assert abs(curve.X.iloc[0]-x0)<1e-12 and abs(curve.Xd.iloc[0]-W_DEAD)<1e-12
            E_X_ROWS.append(dict(lab=lab,theta=theta,X0_g_l=float(x0),Xd0_g_l=W_DEAD,
                physically_admissible=curve.attrs['physically_admissible'],**e_score(curve,lab)))
        print('Barrido X₀',theta,lab,len(E_X_GRID),'valores; horizonte 166 h',flush=True)
E_X_SCORES=pd.DataFrame(E_X_ROWS);E_X_MIN_ROWS=[]
for theta in THETA_SETS:
    fig,axes=plt.subplots(3,3,figsize=(15,9),layout='constrained')
    for i,lab in enumerate(LABS):
        group=E_X_SCORES.query('lab==@lab and theta==@theta').sort_values('X0_g_l')
        valid=group[group.physically_admissible]
        assert len(valid)>0
        for j,state in enumerate(E_STATES):
            ax=axes[i,j];column=state+'_RMSE';y=group[column].where(group.physically_admissible)
            ax.plot(group.X0_g_l,y,color='black',lw=1.3,marker='.',ms=3,label='RMSE en la malla')
            best=valid.loc[valid[column].idxmin()]
            boundary=bool(np.isclose(best.X0_g_l,E_X_GRID[0]) or np.isclose(best.X0_g_l,E_X_GRID[-1]))
            ax.scatter(best.X0_g_l,best[column],color='#ad1457',marker='*',s=75,zorder=7,label='Mínimo en la malla')
            for case,x in [('referencia_anterior',BATCHES[lab].initials['X']),
                ('viable_comun_volumen',W_COMMON*W_MASS),('viable_ponderada',W_WEIGHTED[lab]*W_MASS)]:
                ax.axvline(x,color=W_CASES[case][1],ls=W_CASES[case][2],lw=1,label=W_CASES[case][0])
            ax.set(title=f'{lab} · error {state}\n{theta}',xlabel='X₀ viable explorado [g/L]',
                ylabel='RMSE ['+('mg N/L' if state=='N' else 'g/L')+']')
            E_X_MIN_ROWS.append(dict(lab=lab,theta=theta,state=state,min_grid_X0_g_l=float(best.X0_g_l),
                min_grid_RMSE=float(best[column]),boundary=boundary,n_admissible=len(valid),
                N_RMSE=float(best.N_RMSE),GF_RMSE=float(best.GF_RMSE),E_RMSE=float(best.E_RMSE)))
    common_legend(fig,axes,ncol=3);e_save(fig,'04_error_vs_X0_'+theta)
E_X_MINIMA=pd.DataFrame(E_X_MIN_ROWS)
display(E_X_MINIMA.round(4))

## 20. Captación frente a capacidad fermentativa: perturbaciones coherentes

Se explora una malla de factores **0,5; 0,75; 1; 1,25; 1,5** para qN y para una capacidad fermentativa λF. Se evalúa tanto la inicialización viable común como la ponderada, para los tres labs y ambos theta. Las simulaciones mantienen toda la curva, química, temperatura, eventos y biomasa inicial del caso correspondiente.

- **qN:** se multiplica sólo qN. Esto cambia qN/μ₀ y el rendimiento biomasa–N; también altera la evolución de N y, por acoplamiento, X y fermentación. No es un control independiente de la curva de N.
- **λF:** se escalan conjuntamente **βG₀, βF₀, sG, sF, qEG, qEF, γG₀ y γF₀**. Así se conservan β/s (saturaciones de azúcar), qE/β y γ/β, mientras se escalan la producción fermentativa de E/Gly y su consumo de azúcares. No se modifican μ₀, sN, consumo asociado al crecimiento, mantenimiento ni inhibiciones. Es un factor hipotético de capacidad, no una nueva medición de actividad.

La malla contiene 25 combinaciones por inicialización/lab/theta. El centro qN=1 y λF=1 reproduce exactamente el caso de 16.2. La identidad N+(qN/μ₀)(X+Xd)−N_aportado se comprueba con **el qN de cada escenario**. Las excursiones internas fuera de límites se marcan; no se presentan como mejoras aceptables.

Cada figura tiene tres filas (labs) y tres columnas (N, GF, E). El color y el número son **RMSE / RMSE del centro de esa misma inicialización y theta**: menor que 1 mejora. El recuadro negro señala el centro. No se comparan colores entre inicializaciones como si tuvieran el mismo denominador. Las tablas también conservan errores absolutos. Una combinación que mejora las tres variables en estos mismos datos es sólo candidata para estudiar; no es una calibración validada, ni se adopta automáticamente por lab.

El resumen distingue **mejora estricta de las tres variables** y **no empeorar ninguna y mejorar al menos una**. Se usa una tolerancia numérica de 10⁻⁶ en los ratios alrededor de 1 para no contar diferencias del integrador como mejoras. Esa tolerancia no representa la incertidumbre experimental. Una opción puede mejorar GF y E conservando N y, por tanto, ser útil aunque no cumpla el primer criterio.

In [ ]:
E_FACTORS=np.array([.5,.75,1.,1.25,1.5])
from matplotlib.patches import Rectangle
E_FERMENT_KEYS=('betaG0','betaF0','sG','sF','qEG','qEF','gammaG0','gammaF0')
E_KIN_ROWS=[];E_PARAMETER_ROWS=[]
for theta,reference_theta in THETA_SETS.items():
    for qfactor in E_FACTORS:
        for ffactor in E_FACTORS:
            th=dict(reference_theta);th['qN']*=float(qfactor)
            for p in E_FERMENT_KEYS: th[p]*=float(ffactor)
            for num,den in [('betaG0','sG'),('betaF0','sF'),('qEG','betaG0'),
                            ('qEF','betaF0'),('gammaG0','betaG0'),('gammaF0','betaF0')]:
                assert np.isclose(th[num]/th[den],reference_theta[num]/reference_theta[den],rtol=1e-12)
            assert np.isclose(th['qN']/th['mu0'],qfactor*reference_theta['qN']/reference_theta['mu0'])
            assert all(th[p]==reference_theta[p] for p in base.FULL17 if p not in (*E_FERMENT_KEYS,'qN'))
            E_PARAMETER_ROWS.append(dict(theta=theta,qN_factor=float(qfactor),fermentation_factor=float(ffactor),**th))
            for lab in LABS:
                for initcase in ('viable_comun_volumen','viable_ponderada'):
                    x0=W_COMMON*W_MASS if initcase=='viable_comun_volumen' else W_WEIGHTED[lab]*W_MASS
                    batch=replace(BATCHES[lab],initials={**BATCHES[lab].initials,'X':x0,'Xd':W_DEAD})
                    center=bool(qfactor==1. and ffactor==1.)
                    curve=W_CURVES[(lab,theta,initcase)] if center else diagnosis.run(batch,th,'referencia')
                    E_SIMULATIONS+=int(not center)
                    e_register(('kinetic',lab,theta,initcase,float(qfactor),float(ffactor)),curve,th)
                    E_KIN_ROWS.append(dict(lab=lab,theta=theta,initialization=initcase,X0_g_l=x0,
                        qN_factor=float(qfactor),fermentation_factor=float(ffactor),
                        physically_admissible=curve.attrs['physically_admissible'],**e_score(curve,lab)))
    print('Malla cinética',theta,'completa; dos inicializaciones y 25 combinaciones por lab',flush=True)
E_KINETIC=pd.DataFrame(E_KIN_ROWS);E_PARAMETERS=pd.DataFrame(E_PARAMETER_ROWS)
center=E_KINETIC.query('qN_factor==1 and fermentation_factor==1')[['lab','theta','initialization',*[s+'_RMSE' for s in E_STATES]]]
center=center.rename(columns={s+'_RMSE':s+'_RMSE_center' for s in E_STATES})
E_KINETIC=E_KINETIC.merge(center,on=['lab','theta','initialization'],validate='many_to_one')
for state in E_STATES: E_KINETIC[state+'_ratio_center']=E_KINETIC[state+'_RMSE']/E_KINETIC[state+'_RMSE_center']
E_KINETIC['improves_all_three']=E_KINETIC.physically_admissible&np.all(
    E_KINETIC[[s+'_ratio_center' for s in E_STATES]].to_numpy()<1.-1e-6,axis=1)
E_KINETIC['nonworse_all_improves_some']=E_KINETIC.physically_admissible&np.all(
    E_KINETIC[[s+'_ratio_center' for s in E_STATES]].to_numpy()<=1.+1e-6,axis=1)&np.any(
    E_KINETIC[[s+'_ratio_center' for s in E_STATES]].to_numpy()<1.-1e-6,axis=1)
E_KINETIC_SUMMARY=E_KINETIC.groupby(['lab','theta','initialization']).agg(
    n_cases=('qN_factor','size'),n_admissible=('physically_admissible','sum'),
    n_improve_all=('improves_all_three','sum'),
    n_nonworse_all=('nonworse_all_improves_some','sum')).reset_index()
E_KINETIC_BEST_ROWS=[]
for theta in THETA_SETS:
    for initcase in ('viable_comun_volumen','viable_ponderada'):
        fig,axes=plt.subplots(3,3,figsize=(14,10),layout='constrained')
        for i,lab in enumerate(LABS):
            group=E_KINETIC.query('lab==@lab and theta==@theta and initialization==@initcase')
            for j,state in enumerate(E_STATES):
                ax=axes[i,j]
                table=group.pivot(index='qN_factor',columns='fermentation_factor',values=state+'_ratio_center').reindex(index=E_FACTORS,columns=E_FACTORS)
                valid=group.pivot(index='qN_factor',columns='fermentation_factor',values='physically_admissible').reindex(index=E_FACTORS,columns=E_FACTORS)
                im=ax.imshow(np.where(valid,np.log2(table),np.nan),vmin=-2,vmax=2,cmap='coolwarm',aspect='auto',origin='lower');ax.grid(False)
                for r,q in enumerate(E_FACTORS):
                    for c,f in enumerate(E_FACTORS):
                        ax.text(c,r,f'{table.loc[q,f]:.2f}' if valid.loc[q,f] else '†',ha='center',va='center',fontsize=8,color='white' if valid.loc[q,f] and abs(np.log2(table.loc[q,f]))>1.25 else 'black')
                ax.add_patch(Rectangle((1.5,1.5),1,1,fill=False,edgecolor='black',linewidth=2))
                ax.set_xticks(range(5),[f'{v:g}' for v in E_FACTORS]);ax.set_yticks(range(5),[f'{v:g}' for v in E_FACTORS])
                ax.set(title=f'{lab} · {state}\n{theta} · '+('común' if initcase=='viable_comun_volumen' else 'ponderada'),
                    xlabel='Factor capacidad fermentativa λF',ylabel='Factor qN')
                acceptable=group[group.physically_admissible]
                assert len(acceptable)>0
                best=acceptable.loc[acceptable[state+'_RMSE'].idxmin()]
                E_KINETIC_BEST_ROWS.append(dict(lab=lab,theta=theta,initialization=initcase,state=state,
                    min_grid_qN_factor=float(best.qN_factor),min_grid_fermentation_factor=float(best.fermentation_factor),
                    boundary=bool(best.qN_factor in (E_FACTORS[0],E_FACTORS[-1]) or best.fermentation_factor in (E_FACTORS[0],E_FACTORS[-1])),
                    **{s+'_ratio_center':float(best[s+'_ratio_center']) for s in E_STATES}))
        cb=fig.colorbar(im,ax=axes,shrink=.65,extend='both');cb.set_ticks([-2,-1,0,1,2],labels=['0,25','0,5','1','2','4'])
        cb.set_label('RMSE / centro de la misma inicialización · menor que 1 mejora')
        e_save(fig,'05_captacion_vs_fermentacion_'+theta+'_'+initcase)
E_KINETIC_BEST=pd.DataFrame(E_KINETIC_BEST_ROWS)
display(E_KINETIC_SUMMARY)
display(E_KINETIC_BEST.query('theta=="theta_full"').round(3))

## 21. Conclusiones calculadas, alcance y trazabilidad

Las conclusiones siguientes se generan a partir de las tablas ejecutadas. Los contrastes de fases y balances no agregan mediciones nuevas. Los barridos reutilizan las mismas muestras que se inspeccionaron; no constituyen validación externa ni seleccionan parámetros finales. Los pesos y multiplicadores no son incertidumbres certificadas.

Se distinguen errores de tiempo, de proporciones y de parámetros **condicionalmente al modelo**. Ningún resultado convierte el recuento celular en una medición de actividad efectiva ni demuestra por sí solo que el YAN residual no sea asimilable. Los pulsos y tiempos NUT siguen sujetos a la calidad del registro experimental. Todos los escenarios conservan el reloj original y el horizonte completo; no se ajusta un lag ni se aplica EKF.

**Lectura conjunta:** los mínimos de X₀ para N, GF y E quedan separados, especialmente en LAB021 con theta_full. Esto explica por qué una misma inicialización no corrige las tres curvas. En los diagramas E frente a GF, las tres trayectorias del modelo casi se superponen, aunque su evolución temporal sea diferente; varios puntos experimentales se apartan de esa relación. Conviene revisar tanto el ritmo como las proporciones de consumo y producción. Esta observación no determina por sí sola qué parámetro, conversión o condición inicial está equivocado.

En las mallas cinéticas, N cambia muy poco al modificar λF para un qN fijo en estos escenarios. Por ello se informa también cuándo GF/E mejoran manteniendo N, sin exigir una mejora estricta de N. El resultado es condicional al régimen simulado y no garantiza desacoplamiento en otros experimentos.

In [ ]:
E_CONCLUSIONS=['**Resultados por lab con theta_full:**']
for lab in LABS:
    best=E_X_MINIMA.query('lab==@lab and theta=="theta_full"').set_index('state')
    desc='; '.join(f'{s}: X₀={best.loc[s,"min_grid_X0_g_l"]:.4f} g/L'+(' (extremo)' if best.loc[s,'boundary'] else '') for s in E_STATES)
    E_CONCLUSIONS.append(f'- {lab}, mínimos diagnósticos de la malla de X₀: '+desc+'.')
    for initcase in ('viable_comun_volumen','viable_ponderada'):
        row=E_KINETIC_SUMMARY.query('lab==@lab and theta=="theta_full" and initialization==@initcase').iloc[0]
        E_CONCLUSIONS.append(f'  {E_LABELS[initcase]}: {int(row.n_improve_all)}/{int(row.n_admissible)} combinaciones admisibles '
            f'de qN/λF mejoran estrictamente N, GF y E; {int(row.n_nonworse_all)} no empeoran ninguna y mejoran al menos una frente al centro. Son candidatos exploratorios, no valores adoptados.')
    post=E_PHASE_METRICS.query('lab==@lab and theta=="theta_full" and state=="N" and phase=="post_pulso"')
    E_CONCLUSIONS.append(f'  Error N después del pulso: '+', '.join(f'{E_LABELS[r.case]} {r.RMSE:.2f} mg N/L' for r in post.itertuples())+'.')
E_CONCLUSIONS.append('En los diagramas frente a GF, los residuos se evalúan en tiempos coincidentes: '
    'no son errores a un mismo GF interpolado. Las tasas entre muestras son promedios sobre intervalos, '
    'no tasas instantáneas. La malla de X₀ y la de parámetros necesitan contrastes independientes '
    'antes de usarse en un filtro o de reemplazar los parámetros antiguos.')
display(Markdown('\n\n'.join(E_CONCLUSIONS)))
E_CHECKS=pd.DataFrame(E_CHECK_ROWS)
assert HASH_BEFORE=={str(p.relative_to(ROOT)):sha256(p) for p in PROTECTED}
assert E_CHECKS.inventory_drift_mg_l.max()<.002 and E_CHECKS.jump_error.max()<1e-12
assert E_CHECKS.start_h.eq(0).all() and E_CHECKS.end_h.eq(166).all()
E_TABLES=dict(residuals=E_RESIDUALS,phase_metrics=E_PHASE_METRICS,paired_observations=E_PAIRS,
    balance_predictions=E_BALANCES,balance_summary=E_BALANCE_SUMMARY,interval_rates=E_RATES,
    interval_rate_summary=E_RATE_SUMMARY,X0_grid_scores=E_X_SCORES,X0_grid_minima=E_X_MINIMA,
    kinetic_grid_scores=E_KINETIC,kinetic_grid_summary=E_KINETIC_SUMMARY,
    kinetic_grid_minima=E_KINETIC_BEST,kinetic_parameters=E_PARAMETERS,checks=E_CHECKS,
    figure_index=pd.DataFrame(E_FIGURES))
for name,table in E_TABLES.items(): table.to_csv(E_OUT/(name+'.csv'),index=False)
arrays={};array_manifest=[]
for i,(key,curve) in enumerate(E_CURVES.items()):
    name=f'case_{i}';arrays[name]=np.column_stack([curve.index,curve.to_numpy()])
    array_manifest.append(dict(array=name,key=list(key),columns=['t_h']+list(curve.columns)))
np.savez_compressed(E_OUT/'trajectories.npz',**arrays)
e_manifest=dict(observation_policy=GF_ZERO_POLICY,created_at=datetime.now().astimezone().isoformat(),python=sys.executable,
    source_code_sha256=sha256(LAB2026/'notebooks/diagnostics/nitrogen_consumption_diagnosis.source.ipynb'),
    original_input_hashes=HASH_BEFORE,theta=THETA_SETS,arrays=array_manifest,figures=E_FIGURES,
    new_simulations=E_SIMULATIONS,total_stored_curves=len(E_CURVES),X0_grid=E_X_GRID.tolist(),
    kinetic_factors=E_FACTORS.tolist(),fermentation_scaled_parameters=list(E_FERMENT_KEYS),
    initialization_comparison='Original, common viable, weighted viable; recent labs only',
    phase_cut_h=E_EARLY_H,minimum_interval_h=6.,final_h=166.,
    gas_data_loaded=False,optimizer_calls=0,kalman_updates=0,biomass_validation=False,
    minima='Descriptive minima on finite grids; no confidence intervals or fitted parameters adopted',
    paired_chemistry='Exact common sample times; no interpolation of missing chemistry')
(E_OUT/'manifest.json').write_text(json.dumps(e_manifest,ensure_ascii=False,indent=2),encoding='utf-8')
manifest['error_mechanism_extension']=dict(path=str(E_OUT.relative_to(ROOT)),
    new_simulations=E_SIMULATIONS,total_stored_curves=len(E_CURVES),figures=len(E_FIGURES))
(OUT/'manifest.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding='utf-8')
print('Análisis 17–21 completo:',E_SIMULATIONS,'simulaciones nuevas;',len(E_CURVES),'curvas guardadas;',len(E_FIGURES),'figuras.')
print('SOURCE limpio; EXECUTED con resultados; inputs, parámetros antiguos y módulos originales intactos; sin commit ni push.')

## 22. Separar captación de N, actividad fermentativa y temperatura

El objetivo es entender por qué una inicialización puede mejorar N y empeorar GF/E. Se cambia **un mecanismo a la vez**, dejando fijos X₀, Xd₀, química inicial, reloj, dosis y hora del pulso. Se comparan LAB019–021 con las inicializaciones **viable común** y **viable ponderada** de 16.2 y los dos conjuntos de parámetros antiguos. La referencia anterior azul se conserva como comparación; no se adopta como biomasa verdadera.

| Prueba | Qué cambia | Qué conserva |
|---|---|---|
| Captación de N | qN × {0,5; 0,75; 1; 1,25; 1,5} | μ₀, sN y parámetros fermentativos |
| Actividad fermentativa | λF × {0,5; 0,75; 1; 1,25; 1,5}, con el escalado coherente de §20 | Crecimiento, captación de N y mantenimiento |
| Temperatura completa | Perfil medido T(t) + {−2; −1; 0; +1; +2} °C | Parámetros e inicialización |

Cambiar qN solo **cambia el rendimiento biomasa/N**: no representa un cambio independiente de la rapidez de crecimiento. Puede cambiar X(t) y, por esa vía, GF y E. El escalado λF conserva β/s, qE/β y γ/β; modifica conjuntamente producción fermentativa de etanol/glicerol y consumo de azúcares. Cambiar T en el modelo completo afecta crecimiento, saturaciones, inhibiciones, mantenimiento y muerte.

Las perturbaciones térmicas son experimentos matemáticos, no correcciones del sensor ni cambios propuestos para operar el proceso. Algunos perfiles perturbados pueden salir del rango histórico. Se simula desde 0 hasta 166 h, **incluyendo el pulso y toda la evolución posterior**, sin CO₂, EKF ni ajuste de parámetros. Los puntos NUT se muestran con su convención temporal ya documentada y se mantienen fuera del RMSE; se excluyen también las condiciones iniciales y la biomasa poco fiable. GF final de LAB019 conserva el cero incorporado por solicitud del usuario.

In [ ]:
S_OUT=OUT/'separated_N_fermentation_temperature';S_OUT.mkdir(exist_ok=True)
S_STATES=('N','GF','E')
S_INITS=('viable_comun_volumen','viable_ponderada')
S_INIT_LABEL={'viable_comun_volumen':'Viable común','viable_ponderada':'Viable ponderada'}
S_FACTORS=(.5,.75,1.,1.25,1.5);S_DELTAS=(-2.,-1.,0.,1.,2.)
S_CURVES={};S_ROWS=[];S_RESIDUALS=[];S_CHECKS=[];S_PARAMETERS=[];S_FIGURES=[]
S_NEW_SIMULATIONS=0;S_CACHE_REUSES=0
S_OLD_RESULT_FILES=[p for p in OUT.rglob('*') if p.suffix in ('.csv','.npz') and S_OUT not in p.parents]
S_OLD_HASHES={str(p.relative_to(ROOT)):sha256(p) for p in S_OLD_RESULT_FILES}

def s_batch(lab,initcase):
    x=W_COMMON*W_MASS if initcase=='viable_comun_volumen' else W_WEIGHTED[lab]*W_MASS
    return replace(BATCHES[lab],initials={**BATCHES[lab].initials,'X':x,'Xd':W_DEAD})

def s_save(fig,name):
    fig.canvas.draw()
    if fig._suptitle is not None:
        title_box=fig._suptitle.get_window_extent(fig.canvas.get_renderer())
        assert all(not title_box.overlaps(legend.get_window_extent(fig.canvas.get_renderer())) for legend in fig.legends)
    path=S_OUT/(name+'.png');fig.savefig(path,dpi=125,bbox_inches='tight',facecolor='white')
    buffer=io.BytesIO();fig.savefig(buffer,format='png',dpi=105,bbox_inches='tight',facecolor='white')
    plt.close(fig);display(Image(data=buffer.getvalue()));buffer.close()
    S_FIGURES.append(dict(name=name,path=str(path.relative_to(ROOT))))

def s_legend(fig,axes,ncol=4):
    unique={}
    for ax in np.asarray(axes).flat:
        handles,labels=ax.get_legend_handles_labels()
        for handle,label in zip(handles,labels): unique.setdefault(label,handle)
    fig.get_layout_engine().set(rect=(0.,0.,1.,.90))
    fig.legend(unique.values(),unique.keys(),loc='upper center',bbox_to_anchor=(.5,.965),
        ncol=ncol,fontsize=8)

def s_score(curve,lab):
    errors=e_errors(curve,lab);rows=[]
    pulse=min(t for t,d in BATCHES[lab].pulses['N'])
    for state in S_STATES:
        g=errors[errors.state.eq(state)]
        for phase,part in [('completo',g),('pre_pulso',g[g.t_h<=pulse]),('post_pulso',g[g.t_h>pulse])]:
            if len(part):
                r=part.residual.to_numpy()
                rows.append(dict(state=state,phase=phase,n=len(part),RMSE=float(np.sqrt(np.mean(r*r))),
                    bias=float(r.mean()),MAE=float(np.mean(abs(r))),unit='mg N/L' if state=='N' else 'g/L'))
    return rows,errors

def s_register(lab,theta,initcase,mechanism,value,curve,th,batch):
    key=(lab,theta,initcase,mechanism,float(value));S_CURVES[key]=curve
    scores,errors=s_score(curve,lab)
    meta=dict(lab=lab,theta=theta,initialization=initcase,mechanism=mechanism,value=float(value))
    S_ROWS.extend({**meta,**r} for r in scores)
    for k,v in meta.items(): errors[k]=v
    S_RESIDUALS.append(errors)
    initial_error=float(np.max(abs(curve[list(base.STATE_NAMES)].iloc[0].to_numpy()-base.initial_vector(batch))))
    assert initial_error<1e-12
    assert curve.index[0]==0 and curve.index[-1]==166
    assert curve.attrs['inventory_drift']<2e-6 and curve.attrs['jump_error']<1e-12
    pulse_error=max(abs(curve.loc[t+diagnosis.CONFIG['after_event_h'],'N']-curve.loc[t,'N']-dose)
        for t,dose in batch.pulses['N'])
    assert pulse_error<1e-5
    S_CHECKS.append({**meta,'X0_g_l':batch.initials['X'],'Xd0_g_l':batch.initials['Xd'],
        'initial_error_g_l':initial_error,'inventory_drift_mg_l':1000*curve.attrs['inventory_drift'],
        'jump_error':curve.attrs['jump_error'],'pulse_sample_error_g_l':pulse_error,
        'physically_admissible':curve.attrs['physically_admissible'],
        'start_h':float(curve.index[0]),'end_h':float(curve.index[-1]),
        **{'raw_min_'+p:v for p,v in curve.attrs['raw_min'].items()},
        **{'raw_max_'+p:v for p,v in curve.attrs['raw_max'].items()}})
    S_PARAMETERS.append({**meta,**th})

for theta,original_theta in THETA_SETS.items():
    for lab in LABS:
        for initcase in S_INITS:
            batch=s_batch(lab,initcase)
            for mechanism,values in [('qN',S_FACTORS),('fermentation',S_FACTORS),('temperature',S_DELTAS)]:
                for value in values:
                    th=dict(original_theta);bt=batch
                    if mechanism=='qN': th['qN']*=value
                    elif mechanism=='fermentation':
                        for p in E_FERMENT_KEYS: th[p]*=value
                    else: bt=replace(batch,temperature_c=batch.temperature_c+value)
                    changed={p for p in base.FULL17 if th[p]!=original_theta[p]}
                    expected=({'qN'} if mechanism=='qN' else set(E_FERMENT_KEYS)) if value!=1 and mechanism!='temperature' else set()
                    assert changed==expected
                    assert all(bt.initials[p]==batch.initials[p] for p in base.STATE_NAMES)
                    assert bt.pulses==batch.pulses
                    if mechanism!='temperature': assert np.array_equal(bt.temperature_c,batch.temperature_c)
                    center=(value==0 if mechanism=='temperature' else value==1)
                    if center: curve=W_CURVES[(lab,theta,initcase)];S_CACHE_REUSES+=1
                    elif mechanism!='temperature':
                        cache_key=('kinetic',lab,theta,initcase,value if mechanism=='qN' else 1.,value if mechanism=='fermentation' else 1.)
                        curve=E_CURVES[cache_key];S_CACHE_REUSES+=1
                    else: curve=diagnosis.run(bt,th,'referencia');S_NEW_SIMULATIONS+=1
                    s_register(lab,theta,initcase,mechanism,value,curve,th,bt)
            print('Efectos separados:',theta,lab,S_INIT_LABEL[initcase],flush=True)
S_METRICS=pd.DataFrame(S_ROWS);S_RESIDUAL_TABLE=pd.concat(S_RESIDUALS,ignore_index=True)
S_CENTER=S_METRICS[S_METRICS.mechanism.eq('qN')&S_METRICS.value.eq(1.)][
    ['lab','theta','initialization','state','phase','RMSE']].rename(columns={'RMSE':'RMSE_base'})
S_METRICS=S_METRICS.merge(S_CENTER,on=['lab','theta','initialization','state','phase'],validate='many_to_one')
S_METRICS['ratio_to_base']=S_METRICS.RMSE/S_METRICS.RMSE_base.replace(0,np.nan)
assert len(S_CURVES)==180 and S_NEW_SIMULATIONS==48 and S_CACHE_REUSES==132
S_INITIALS=pd.DataFrame([dict(lab=lab,initialization=case,X0_g_l=s_batch(lab,case).initials['X'],
    Xd0_g_l=W_DEAD,pulse_h=min(t for t,d in BATCHES[lab].pulses['N']),
    measured_T_min_C=float(BATCHES[lab].temperature_c.min()),measured_T_max_C=float(BATCHES[lab].temperature_c.max()))
    for lab in LABS for case in S_INITS])
display(S_INITIALS.round(4))
print('180 casos: 132 reutilizados y 48 simulaciones térmicas nuevas; parámetros antiguos intactos.')

### 22.1. Comparación de las curvas completas: un mecanismo a la vez

Cada figura tiene **filas = LAB019, LAB020, LAB021** y **columnas = N, GF, E**. Cada bloque mantiene una única inicialización. Azul discontinuo: referencia anterior. Negro: predicción base con la biomasa indicada en el título. Verde/rojo: perturbaciones menores/mayores del mismo mecanismo. Se muestran los extremos para facilitar la lectura; los cinco niveles se conservan en las métricas y curvas de error siguientes.

En N, una curva desplazada hacia la derecha significa que el agotamiento se retrasa. En GF, una curva más alta significa más azúcar residual; en E, una curva más baja significa menos etanol producido. Mejorar N puede empeorar simultáneamente GF y E. Las figuras temporales principales usan theta_full; ambos theta se comparan cuantitativamente en 22.2.

In [ ]:
for initcase in S_INITS:
    for mechanism,title in [('qN','Captación de N'),('fermentation','Actividad fermentativa'),('temperature','Temperatura completa')]:
        fig,axes=plt.subplots(3,3,figsize=(15,9),layout='constrained')
        specs=[(.5,'×0,5','#00897b','--'),(1.,'Base','black','-'),(1.5,'×1,5','#c62828','-.')]
        if mechanism=='temperature': specs=[(-2.,'T − 2 °C','#00897b','--'),(0.,'Base','black','-'),(2.,'T + 2 °C','#c62828','-.')]
        for i,lab in enumerate(LABS):
            for j,state in enumerate(S_STATES):
                ax=axes[i,j];scale=1000 if state=='N' else 1;column='N_pred' if state=='N' else state
                previous=W_CURVES[(lab,'theta_full','referencia_anterior')]
                ax.plot(previous.index,previous[column]*scale,color='#2166ac',ls=':',lw=1.3,label='Referencia anterior')
                for value,label,color,style in specs:
                    curve=S_CURVES[(lab,'theta_full',initcase,mechanism,value)]
                    ax.plot(curve.index,curve[column]*scale,color=color,ls=style,lw=1.5,label=label)
                draw_observations(ax,lab,state,scale);mark_events(ax,lab)
                ax.set(title=f'{lab} · {state}',xlabel='t desde inoculación [h]',
                    ylabel='mg N/L' if state=='N' else 'g/L',xlim=(0,166))
        fig.suptitle(title+' · '+S_INIT_LABEL[initcase]+' · theta_full',y=.995)
        s_legend(fig,axes,ncol=4);s_save(fig,'01_curvas_'+mechanism+'_'+initcase)

### 22.2. Qué variable mejora y cuál empeora

Las figuras siguientes muestran el **RMSE dividido por el RMSE base de la misma inicialización y theta**. Cada fila es un LAB; cada columna es un mecanismo. N, GF y E tienen colores distintos. **Menor que 1 mejora; mayor que 1 empeora**. Los valores absolutos y los errores antes/después del pulso se guardan en las tablas: una reducción relativa grande no garantiza un error absoluto pequeño. No se suman errores de unidades distintas ni se elige una supuesta solución óptima conjunta.

Estos barridos examinan un mecanismo a la vez. No describen las interacciones de una recalibración conjunta. Cambiar toda la temperatura tampoco equivale a identificar o recalibrar las energías de activación.

In [ ]:
for theta in THETA_SETS:
    for initcase in S_INITS:
        fig,axes=plt.subplots(3,3,figsize=(14,9),layout='constrained')
        for i,lab in enumerate(LABS):
            for j,mechanism in enumerate(('qN','fermentation','temperature')):
                ax=axes[i,j]
                g=S_METRICS.query('lab==@lab and theta==@theta and initialization==@initcase and mechanism==@mechanism and phase=="completo"')
                for state,color in [('N','#7b1fa2'),('GF','#00897b'),('E','#e66101')]:
                    part=g[g.state.eq(state)].sort_values('value')
                    ax.plot(part.value,part.ratio_to_base,color=color,marker='o',ms=3,label=state)
                ax.axhline(1,color='black',ls='--',lw=.9)
                ax.set(title=f'{lab} · '+{'qN':'Captación de N','fermentation':'Actividad fermentativa','temperature':'Temperatura'}[mechanism],
                    xlabel='ΔT [°C]' if mechanism=='temperature' else 'Factor qN' if mechanism=='qN' else 'Factor λF',
                    ylabel='RMSE / RMSE base',ylim=(0,None))
        fig.suptitle(S_INIT_LABEL[initcase]+' · '+theta,y=.995)
        s_legend(fig,axes,ncol=3);s_save(fig,'02_errores_'+theta+'_'+initcase)
S_EXTREMES=S_METRICS.query('theta=="theta_full" and phase=="completo"').copy()
display(S_EXTREMES[['lab','initialization','mechanism','value','state','RMSE','ratio_to_base']].round(3))

### 22.3. Qué parte de la respuesta térmica produce el cambio

Para distinguir rutas térmicas se añaden dos **intervenciones matemáticas** con ΔT = ±2 °C y theta_full:

- **T en crecimiento:** sólo el factor aμ·N/(N+KN) y μ se evalúan con T+ΔT. Se actualizan coherentemente crecimiento, captación de N y consumo de azúcar asociado al crecimiento. La fermentación, mantenimiento y muerte usan T medida.
- **T en fermentación:** sólo los factores fermentativos de glucosa/fructosa se evalúan con T+ΔT, incluyendo sus saturaciones e inhibiciones. Se actualizan consumo fermentativo de azúcar y producción de E/glicerol; crecimiento, captación, mantenimiento y muerte usan T medida.

Se comparan con **T completa**, que cambia todas las rutas. Estas intervenciones no representan un reactor con dos temperaturas: sirven para localizar la sensibilidad dentro del modelo. Los efectos no tienen por qué sumarse porque las trayectorias se realimentan de forma no lineal.

La implementación parte del RHS oficial y añade exclusivamente la diferencia de la ruta seleccionada. No modifica el módulo del modelo ni sus constantes globales. Se comprueba que ΔT=0 reproduce el RHS oficial y su trayectoria, y se verifican el balance N–biomasa y los pulsos.

In [ ]:
from scipy.integrate import solve_ivp

def s_route_rhs(t,y,th,batch,delta,route):
    original=np.asarray(base.rhs(t,y,th,batch),float)
    x,xd,n,g,f,e,gly=[max(0.,float(v)) for v in y]
    temp=base.temperature_at(batch,t)
    nominal=base.kinetic_terms(th,temp,x,n,g,f,e)
    shifted=base.kinetic_terms(th,temp+delta,x,n,g,f,e)
    if route=='temperature_growth':
        change=shifted['growth_factor']-nominal['growth_factor']
        original+=x*np.array([th['mu0']*change,0.,-th['qN']*change,-th['qXG']*change,-th['qXF']*change,0.,0.])
    elif route=='temperature_fermentation':
        dg=shifted['glucose_ferm_factor']-nominal['glucose_ferm_factor']
        df=shifted['fructose_ferm_factor']-nominal['fructose_ferm_factor']
        original+=x*np.array([0.,0.,0.,-th['qEG']*dg,-th['qEF']*df,
            th['betaG0']*dg+th['betaF0']*df,th['gammaG0']*dg+th['gammaF0']*df])
    else: raise ValueError(route)
    return original

def s_route_run(batch,th,delta,route):
    time=diagnosis.make_grid(batch);state=base.initial_vector(batch).copy()
    offset_idx=[base.STATE_NAMES.index(p) for p in ('Xd','Gly')]
    offsets=state[offset_idx].copy();state[offset_idx]=0.
    events=base._pulse_events(batch,float(time[0]),float(time[-1]))
    records={float(time[0]):state.copy()};current=float(time[0]);jump_error=0.
    if current in events: state+=events[current]
    for boundary in sorted({float(time[-1]),*[t for t in events if t>current]}):
        requested=time[(time>current)&(time<=boundary)]
        sol=solve_ivp(lambda t,y:s_route_rhs(t,y,th,batch,delta,route),(current,boundary),state,
            t_eval=np.unique(np.r_[requested,boundary]),method='LSODA',rtol=diagnosis.CONFIG['rtol'],
            atol=diagnosis.CONFIG['atol'],max_step=diagnosis.CONFIG['max_step_h'])
        assert sol.success,sol.message
        records.update({float(t):y.copy() for t,y in zip(sol.t,sol.y.T)})
        current=boundary;state=sol.y[:,-1].copy()
        if boundary in events:
            before=state.copy();state+=events[boundary]
            jump_error=max(jump_error,float(np.max(abs(state-before-events[boundary]))))
    values=np.vstack([records[float(t)] for t in time]);values[:,offset_idx]+=offsets
    assert np.isfinite(values).all()
    raw_min={p:float(values[:,i].min()) for i,p in enumerate(base.STATE_NAMES)}
    raw_max={p:float(values[:,i].max()) for i,p in enumerate(base.STATE_NAMES)}
    curve=pd.DataFrame(values,index=time,columns=base.STATE_NAMES)
    for p in base.STATE_NAMES: curve[p]=curve[p].clip(*base.STATE_BOUNDS[p])
    curve['GF']=curve.G+curve.F;curve['N_pred']=curve.N
    curve['T_C']=np.interp(time,batch.time,batch.temperature_c)
    dose=np.array([sum(d for t,d in batch.pulses['N'] if t<ti-1e-9) for ti in time])
    inventory=curve.N+th['qN']/th['mu0']*(curve.X+curve.Xd)-dose
    curve.attrs.update(jump_error=jump_error,raw_min=raw_min,raw_max=raw_max,
        inventory_drift=float(np.max(abs(inventory-inventory.iloc[0]))),
        physically_admissible=all(raw_min[p]>=base.STATE_BOUNDS[p][0]-1e-6 and raw_max[p]<=base.STATE_BOUNDS[p][1]+1e-6 for p in base.STATE_NAMES))
    return curve

S_ROUTE_PARITY=[]
for lab in LABS:
    for initcase in S_INITS:
        batch=s_batch(lab,initcase);th=THETA_SETS['theta_full']
        baseline=S_CURVES[(lab,'theta_full',initcase,'temperature',0.)]
        for route in ('temperature_growth','temperature_fermentation'):
            for t in (0.,20.,40.,70.,100.,150.):
                y=np.array([np.interp(t,baseline.index,baseline[p]) for p in base.STATE_NAMES])
                assert np.array_equal(s_route_rhs(t,y,th,batch,0.,route),np.asarray(base.rhs(t,y,th,batch)))
            # One zero-shift trajectory per route verifies the independent event integrator.
            if lab=='LAB021' and initcase=='viable_ponderada':
                zero=s_route_run(batch,th,0.,route);S_NEW_SIMULATIONS+=1
                difference=float(np.max(abs(zero[list(base.STATE_NAMES)].to_numpy()-baseline[list(base.STATE_NAMES)].to_numpy())))
                assert difference<2e-6
                S_ROUTE_PARITY.append(dict(route=route,max_state_difference_g_l=difference))
            for delta in (-2.,2.):
                curve=s_route_run(batch,th,delta,route);S_NEW_SIMULATIONS+=1
                s_register(lab,'theta_full',initcase,route,delta,curve,th,batch)
        print('Rutas térmicas:',lab,S_INIT_LABEL[initcase],flush=True)
S_METRICS=pd.DataFrame(S_ROWS).merge(S_CENTER,on=['lab','theta','initialization','state','phase'],validate='many_to_one')
S_METRICS['ratio_to_base']=S_METRICS.RMSE/S_METRICS.RMSE_base.replace(0,np.nan)
S_RESIDUAL_TABLE=pd.concat(S_RESIDUALS,ignore_index=True)
display(pd.DataFrame(S_ROUTE_PARITY))
for initcase in S_INITS:
    fig,axes=plt.subplots(3,3,figsize=(15,9),layout='constrained')
    for i,lab in enumerate(LABS):
        for j,state in enumerate(S_STATES):
            ax=axes[i,j];column='N_pred' if state=='N' else state;scale=1000 if state=='N' else 1
            for route,value,label,color,style in [
                ('temperature',0.,'Base','black','-'),('temperature',2.,'T completa','#c62828','-.'),
                ('temperature_growth',2.,'T en crecimiento','#7b1fa2','--'),
                ('temperature_fermentation',2.,'T en fermentación','#00897b',':')]:
                curve=S_CURVES[(lab,'theta_full',initcase,route,value)]
                ax.plot(curve.index,curve[column]*scale,color=color,ls=style,lw=1.5,label=label)
            draw_observations(ax,lab,state,scale);mark_events(ax,lab)
            ax.set(title=f'{lab} · {state}',xlabel='t desde inoculación [h]',
                ylabel='mg N/L' if state=='N' else 'g/L',xlim=(0,166))
    fig.suptitle('Rutas térmicas · +2 °C · '+S_INIT_LABEL[initcase]+' · theta_full',y=.995)
    s_legend(fig,axes,ncol=3);s_save(fig,'03_rutas_termicas_'+initcase)
print('Las pruebas de −2 °C también están calculadas y exportadas; las figuras muestran +2 °C.')

### 22.4. Temperatura observada y lectura cuantitativa por LAB

Se compara el rango térmico de la calibración histórica con los perfiles usados en los tres LAB. Estar dentro del rango no valida automáticamente la ley térmica ni separa su efecto de biomasa, nutrición y protocolo. Las energías de activación son constantes fijadas del modelo, no parámetros reestimados en estas pruebas.

Las conclusiones siguientes se calculan a partir de los errores de **todos los datos posteriores a la inicialización**, con un desglose antes y después del pulso. Se evita interpretar una mejora de N como validación de GF/E, o una mejora en GF/E como validación independiente de X₀. Las comparaciones se condicionan a las hipótesis de biomasa y a los parámetros antiguos; no identifican una causa biológica única.

In [ ]:
S_TRAIN_MIN=min(float(BATCHES[lab].temperature_c.min()) for lab in HIST_LABS)
S_TRAIN_MAX=max(float(BATCHES[lab].temperature_c.max()) for lab in HIST_LABS)
S_THERMAL_ROWS=[]
fig,axes=plt.subplots(1,3,figsize=(15,3.8),layout='constrained')
for ax,lab in zip(axes,LABS):
    batch=BATCHES[lab]
    ax.axhspan(S_TRAIN_MIN,S_TRAIN_MAX,color='#dddddd',alpha=.6,label='Rango histórico')
    ax.plot(batch.time,batch.temperature_c,color='black',lw=1.3,label='T medida')
    mark_events(ax,lab);ax.set(title=lab,xlabel='t desde inoculación [h]',ylabel='T [°C]',xlim=(0,166))
    for delta in S_DELTAS:
        temp=batch.temperature_c+delta
        t=batch.time;outside=(temp<S_TRAIN_MIN)|(temp>S_TRAIN_MAX)
        outside_h=float(np.sum(np.diff(t)*(outside[:-1].astype(float)+outside[1:].astype(float))/2))
        S_THERMAL_ROWS.append(dict(lab=lab,delta_T_C=delta,min_C=float(temp.min()),max_C=float(temp.max()),
            training_min_C=S_TRAIN_MIN,training_max_C=S_TRAIN_MAX,approx_outside_training_h=outside_h))
common_legend(fig,axes,ncol=3);s_save(fig,'04_temperaturas_y_rango_historico')
S_THERMAL=pd.DataFrame(S_THERMAL_ROWS)
display(S_THERMAL.round(3))
S_SUMMARY=[]
def s_rmse(lab,initcase,mechanism,value,theta='theta_full',phase='completo'):
    g=S_METRICS.query('lab==@lab and theta==@theta and initialization==@initcase and mechanism==@mechanism and value==@value and phase==@phase')
    return g.set_index('state').RMSE.reindex(S_STATES).to_numpy()
def s_triplet(values): return f'N {values[0]:.2f} mg N/L; GF {values[1]:.2f} g/L; E {values[2]:.2f} g/L'
for lab in LABS:
    S_SUMMARY.append(f'**{lab}, theta_full:**')
    for initcase in S_INITS:
        base_rmse=s_rmse(lab,initcase,'qN',1.)
        S_SUMMARY.append(f'- {S_INIT_LABEL[initcase]}, RMSE base: {s_triplet(base_rmse)}.')
        for mechanism,value,title in [('qN',.5,'qN ×0,5'),('fermentation',1.5,'λF ×1,5'),('temperature',-2.,'T −2 °C'),('temperature',2.,'T +2 °C')]:
            v=s_rmse(lab,initcase,mechanism,value)
            moves=', '.join(f'{state} '+('mejora' if b<a-1e-5 else 'empeora' if b>a+1e-5 else 'se conserva') for state,a,b in zip(S_STATES,base_rmse,v))
            S_SUMMARY.append(f'  {title}: {s_triplet(v)}. {moves}.')
        for route,title in [('temperature_growth','+2 °C sólo en crecimiento'),('temperature_fermentation','+2 °C sólo en fermentación')]:
            v=s_rmse(lab,initcase,route,2.)
            S_SUMMARY.append(f'  {title}: {s_triplet(v)}.')
    S_SUMMARY.append('')
S_SUMMARY.append(f'Rango histórico usado: {S_TRAIN_MIN:.1f}–{S_TRAIN_MAX:.1f} °C. '
    'LAB021 está dentro de ese rango; estos resultados no apoyan atribuir su discrepancia a una extrapolación fría. '
    'La adecuación de las dependencias térmicas necesita validación entre experimentos, sin confundirla con cambios de biomasa.')
S_SUMMARY.append('Una mejora de GF/E al aumentar λF conservando N demuestra una compensación posible dentro del modelo, '
    'no que λF sea el valor correcto ni que la biomasa ponderada sea verdadera. qN altera el rendimiento de crecimiento; '
    'por eso sus efectos posteriores sobre GF/E no son independientes. Las rutas térmicas aisladas diagnostican la implementación '
    'y sus sensibilidades, no identifican por sí solas un mecanismo biológico.')
display(Markdown('\n\n'.join(S_SUMMARY)))
S_LAB021=S_METRICS.query('lab=="LAB021" and theta=="theta_full" and initialization=="viable_ponderada"')
display(S_LAB021[['mechanism','value','state','phase','n','RMSE','bias','ratio_to_base']].round(4))

### 22.5. Verificación y resultados guardados

Se conserva el contenido ejecutado de las secciones anteriores y se ejecuta esta ampliación usando sus trayectorias verificadas para qN/λF y las condiciones de partida. Las simulaciones nuevas corresponden a temperatura y a la comprobación de paridad del integrador diagnóstico. Un Run All de SOURCE reproduce también las secciones anteriores antes de esta ampliación. No se sustituyen calibraciones ni se modifican datos originales.

Se exportan curvas, parámetros por escenario, residuos, RMSE antes/después del pulso, controles numéricos, cobertura térmica y figuras. Los controles comprueban condiciones iniciales fijas, horizonte completo, salto del pulso, balance de N–biomasa, finitud y límites físicos. La cobertura fuera del rango histórico es una aproximación temporal sobre la malla, no una incertidumbre estadística. Una perturbación con estados fuera de límites se identifica y no se usa para recomendar cambios.

In [ ]:
S_CHECK_TABLE=pd.DataFrame(S_CHECKS);S_PARAMETER_TABLE=pd.DataFrame(S_PARAMETERS)
assert len(S_CURVES)==204 and S_NEW_SIMULATIONS==74
assert S_CHECK_TABLE.start_h.eq(0).all() and S_CHECK_TABLE.end_h.eq(166).all()
assert S_CHECK_TABLE.initial_error_g_l.max()<1e-12
assert S_CHECK_TABLE.inventory_drift_mg_l.max()<.002
assert S_CHECK_TABLE.jump_error.max()<1e-12
assert S_CHECK_TABLE.physically_admissible.all(),S_CHECK_TABLE.loc[~S_CHECK_TABLE.physically_admissible]
# Center errors agree with section 17; no new observations or changed inclusion rules.
S_PREVIOUS=pd.read_csv(OUT/'error_mechanism_diagnostics/phase_metrics.csv')
S_PARITY=S_CENTER.query('phase=="completo"').merge(
    S_PREVIOUS.query('phase=="completo"').rename(columns={'case':'initialization','RMSE':'RMSE_previous'}),
    on=['lab','theta','initialization','state','phase'],validate='one_to_one')
assert len(S_PARITY)==36 and np.allclose(S_PARITY.RMSE_base,S_PARITY.RMSE_previous,rtol=1e-10,atol=1e-8)
assert HASH_BEFORE=={str(p.relative_to(ROOT)):sha256(p) for p in PROTECTED}
assert S_OLD_HASHES=={str(p.relative_to(ROOT)):sha256(p) for p in S_OLD_RESULT_FILES}
S_TABLES=dict(metrics=S_METRICS,residuals=S_RESIDUAL_TABLE,checks=S_CHECK_TABLE,
    parameters=S_PARAMETER_TABLE,initial_conditions=S_INITIALS,temperature_coverage=S_THERMAL,
    route_parity=pd.DataFrame(S_ROUTE_PARITY),figure_index=pd.DataFrame(S_FIGURES))
for name,table in S_TABLES.items(): table.to_csv(S_OUT/(name+'.csv'),index=False)
arrays={};array_manifest=[]
for i,(key,curve) in enumerate(S_CURVES.items()):
    name=f'case_{i}';arrays[name]=np.column_stack([curve.index,curve.to_numpy()])
    array_manifest.append(dict(array=name,key=list(key),columns=['t_h']+list(curve.columns)))
np.savez_compressed(S_OUT/'trajectories.npz',**arrays)
S_MANIFEST=dict(created_at=datetime.now().astimezone().isoformat(),python=sys.executable,
    source_code_sha256=sha256(LAB2026/'notebooks/diagnostics/nitrogen_consumption_diagnosis.source.ipynb'),
    original_input_hashes=HASH_BEFORE,prior_result_hashes=S_OLD_HASHES,theta=THETA_SETS,
    arrays=array_manifest,figures=S_FIGURES,new_simulations=S_NEW_SIMULATIONS,
    reused_cases=S_CACHE_REUSES,total_stored_curves=len(S_CURVES),factor_grid=list(S_FACTORS),
    temperature_delta_grid=list(S_DELTAS),route_delta_grid=[-2.,2.],
    route_validation='Zero-delta RHS and trajectory parity; original model unchanged',
    thermal_routes_theta='theta_full',fermentation_scaled_parameters=list(E_FERMENT_KEYS),
    gas_data_loaded=False,optimizer_calls=0,kalman_updates=0,adopted_parameter_changes=False,
    observation_policy=GF_ZERO_POLICY,exclude_initial_anchors=True,exclude_NUT_from_RMSE=True,
    final_h=166.,config=diagnosis.CONFIG)
(S_OUT/'manifest.json').write_text(json.dumps(S_MANIFEST,ensure_ascii=False,indent=2),encoding='utf-8')
print(f'Ampliación verificada: {len(S_CURVES)} curvas, {S_NEW_SIMULATIONS} simulaciones nuevas '
    f'(incluidas 2 comprobaciones de paridad), {len(S_FIGURES)} figuras.')
print('Balance máximo [mg N/L]:',S_CHECK_TABLE.inventory_drift_mg_l.max())
print('Datos, parámetros antiguos y resultados anteriores intactos. Sin commit ni push.')

## 23. CO₂ desde actividad detectada: viable común y ponderada

**Primera comparación gaseosa, sin EKF ni reajuste:** una figura 1×3 para LAB019–021, sólo con **theta_full** y los **parámetros antiguos de CO₂ del notebook 02**. Se comparan la referencia anterior, viable común y viable ponderada de 16.2. Se conservan exactamente las trayectorias químicas ya calculadas, incluidos los pulsos de N y el horizonte completo de 166 h.

El inicio tₐ se detecta en el **caudal gaseoso medido**, independientemente de la inicialización de biomasa. Se requieren tres bins horarios consecutivos válidos con mediana ≥0,05 g CO₂/L/h. Un bin [t,t+1) sólo está disponible en t+1; se comienza al finalizar el tercero. La persistencia introduce una demora deliberada de confirmación: tₐ identifica emisión sostenida, no el comienzo biológico exacto ni el primer valor positivo.

Se conserva la corrección de cero, unidades y máscara inicial de la auditoría gaseosa previa. El cero usa datos de las primeras 12 h; se exige que todos ellos sean anteriores a tₐ. El campo `flow_filt_sccm` es el ya filtrado por el logger: no se aplica aquí un suavizado centrado, pero su procesamiento interno no se valida como causal en esta prueba. Por ello éste sigue siendo un diagnóstico retrospectivo, no una demostración de un detector online final.

**Qué significa excluir el lag aquí:** no se muestran ni puntúan datos anteriores a tₐ; la capa gaseosa se inicia en tₐ y su gate vale 1 desde ese instante. La química en tₐ procede de las simulaciones desde inoculación, no de una muestra nueva. Sus errores anteriores siguen presentes. No se congelan ni se trasladan las mediciones iniciales, no se reinicia la química y no se convierte caudal nulo en producción biológica nula.

In [ ]:
import ast
from laboratory_2026 import run_co2_matrix_cross_validation_2026 as gas_cross
GAS_HELPER_DIR=LAB2026/'notebooks/EKF'
if str(GAS_HELPER_DIR) not in sys.path: sys.path.insert(0,str(GAS_HELPER_DIR))
import _post_activity_prediction as gas_model
GAS_OUT=OUT/'co2_detected_activity';GAS_OUT.mkdir(exist_ok=True)
GAS_CASES=('referencia_anterior','viable_comun_volumen','viable_ponderada')
GAS_THRESHOLD=float(gas_cross.CO2_DETECTION_LIMIT_G_L_H)
GAS_CONSECUTIVE=3;GAS_FINAL_H=166.;GAS_ZERO_AVAILABLE_H=12.
GAS_FACTOR=44.0095*60.*.74/(1000.*24.16*2.)
assert np.isclose(GAS_FACTOR,.0404391928807947)
GAS_LAYER_PATH=LAB2026/'results/co2_matrix_cross_validation_2026/fit_parameters.csv'
GAS_MASK_PATH=GAS_HELPER_DIR/'post_activity_prediction_results/mask_audit.csv'
GAS_NB02=LAB2026/'notebooks/co2_reformulation/02_nutrition_structure_comparison.source.ipynb'
GAS_INPUT_PATH=DATA_DIR/'backup_global_co2.csv'
GAS_INPUT_FILES=[GAS_LAYER_PATH,GAS_MASK_PATH,GAS_NB02,GAS_INPUT_PATH,
    GAS_HELPER_DIR/'_post_activity_prediction.py',LAB2026/'run_co2_matrix_cross_validation_2026.py']
GAS_INPUT_HASHES={str(p.relative_to(ROOT)):sha256(p) for p in GAS_INPUT_FILES}
GAS_PRIOR_FILES=[p for p in OUT.rglob('*') if p.suffix in ('.csv','.npz') and GAS_OUT not in p.parents]
GAS_PRIOR_HASHES={str(p.relative_to(ROOT)):sha256(p) for p in GAS_PRIOR_FILES}
layer=pd.read_csv(GAS_LAYER_PATH);layer=layer[layer.calibration_matrix.eq('natural')]
GAS_PARAMS=dict(zip(layer.parameter,layer.estimate.astype(float)))
assert set(GAS_PARAMS)==set(gas_cross.PARAMETER_BOUNDS)
nb02=json.loads(GAS_NB02.read_text(encoding='utf-8'))
parameter_code=next(''.join(c['source']) for c in nb02['cells'] if c['cell_type']=='code' and 'PARAM_SETS = {' in ''.join(c['source']))
assignment=next(n for n in ast.parse(parameter_code).body if isinstance(n,ast.Assign)
    and any(isinstance(t,ast.Name) and t.id=='PARAM_SETS' for t in n.targets))
old_node=next(v for k,v in zip(assignment.value.keys,assignment.value.values)
    if isinstance(k,ast.Constant) and k.value=='antiguos')
manual={ast.literal_eval(k):ast.literal_eval(v) for k,v in zip(old_node.keys,old_node.values) if k is not None}
assert manual==dict(A=.30,tau_rise_h=4.,tau_decay_h=12.,kernel_normalization='grid_max')
assert 'HISTORICAL_CO2_LAYER_PATH' in parameter_code
GAS_PARAMS.update(manual);GAS_PARAMS['N_ref_g_l']=.14
assert GAS_PARAMS['kernel_normalization']=='grid_max'
assert THETA_SETS['theta_full']==pd.read_csv(CAL/'theta_natural_full.csv').set_index('parameter').theta.astype(float).to_dict()
GAS_PARAMETER_TABLE=pd.DataFrame([dict(parameter=k,value=v) for k,v in GAS_PARAMS.items()])
display(GAS_PARAMETER_TABLE)

def gas_read_window(path):
    parts=[]
    for chunk in pd.read_csv(path,usecols=['timestamp','fermentador','flow_sccm','flow_filt_sccm','status'],chunksize=500_000,low_memory=False):
        stamp=pd.to_datetime(chunk.timestamp,errors='coerce')
        keep=stamp.between(T0-pd.Timedelta(hours=6),T0+pd.Timedelta(hours=GAS_FINAL_H))
        if keep.any():
            part=chunk.loc[keep].copy();part['timestamp']=stamp.loc[keep];parts.append(part)
    result=pd.concat(parts,ignore_index=True)
    for p in ('flow_sccm','flow_filt_sccm'): result[p]=pd.to_numeric(result[p],errors='coerce')
    return result

GAS_RAW=gas_read_window(GAS_INPUT_PATH)
GAS_MASK=pd.read_csv(GAS_MASK_PATH).set_index('lab')
GAS_HOURLY_PARTS=[];GAS_ONSET_ROWS=[];GAS_CALIBRATION_ROWS=[]
for lab in LABS:
    frame=GAS_RAW[GAS_RAW.fermentador.eq(FERMENTER[lab])].sort_values('timestamp').copy()
    frame['raw_t_h']=(frame.timestamp-T0).dt.total_seconds()/3600.
    mask_end=float(GAS_MASK.loc[lab,'mask_end_h']);zero=float(GAS_MASK.loc[lab,'sensor_zero_sccm'])
    support=frame[frame.raw_t_h.between(mask_end,GAS_ZERO_AVAILABLE_H)].dropna(subset=['flow_filt_sccm'])
    assert len(support)>0 and np.isclose(max(0.,support.flow_filt_sccm.quantile(.10)),zero,rtol=0,atol=1e-12)
    frame['flow_corrected_sccm']=np.maximum(frame.flow_filt_sccm-zero,0.)
    frame['raw_available_h']=frame.raw_t_h
    hourly=frame.set_index('timestamp').resample('1h').agg(
        flow_sccm=('flow_corrected_sccm','median'),n_raw=('flow_corrected_sccm','count'),
        first_raw_h=('raw_available_h','min'),last_raw_h=('raw_available_h','max'))
    hourly=hourly.reset_index();hourly['bin_start_h']=(hourly.timestamp-T0).dt.total_seconds()/3600.
    hourly['bin_end_h']=hourly.bin_start_h+1.;hourly['plot_t_h']=hourly.bin_start_h+.5
    hourly['co2_observed_g_l_h']=hourly.flow_sccm*GAS_FACTOR;hourly['lab']=lab
    hourly=hourly[(hourly.bin_start_h>=0)&(hourly.bin_end_h<=GAS_FINAL_H)].copy()
    hourly['valid']=np.isfinite(hourly.co2_observed_g_l_h)&(hourly.bin_start_h>mask_end)&(hourly.n_raw>=2)&(
        hourly.last_raw_h-hourly.first_raw_h>=.75)
    hourly['quantifiable']=hourly.co2_observed_g_l_h>=GAS_THRESHOLD
    hourly['available_h']=hourly.bin_end_h
    run=[];detected=None;previous=None
    for row in hourly.itertuples():
        contiguous=previous is not None and np.isclose(row.bin_start_h,previous+1.)
        if not contiguous: run=[]
        if row.valid and row.quantifiable: run.append(row)
        else: run=[]
        previous=row.bin_start_h
        if len(run)==GAS_CONSECUTIVE and row.available_h>=GAS_ZERO_AVAILABLE_H:
            detected=run.copy();break
        if len(run)>=GAS_CONSECUTIVE: run=run[-(GAS_CONSECUTIVE-1):]
    assert detected is not None,lab
    start=float(detected[-1].available_h)
    assert start>=float(support.raw_t_h.max()) and start<GAS_FINAL_H
    gas_initial=float(np.median([r.co2_observed_g_l_h for r in detected]))
    GAS_ONSET_ROWS.append(dict(lab=lab,start_h=start,threshold_g_l_h=GAS_THRESHOLD,
        consecutive_bins=GAS_CONSECUTIVE,first_confirmation_bin_h=float(detected[0].bin_start_h),
        last_confirmation_bin_end_h=float(detected[-1].bin_end_h),
        gas_initial_g_l_h=gas_initial,zero_support_last_h=float(support.raw_t_h.max()),
        pulse_h=min(t for t,d in BATCHES[lab].pulses['N']),final_h=GAS_FINAL_H))
    GAS_CALIBRATION_ROWS.append(dict(lab=lab,mask_end_h=mask_end,sensor_zero_sccm=zero,
        zero_support_first_h=float(support.raw_t_h.min()),zero_support_last_h=float(support.raw_t_h.max()),
        zero_support_n=len(support),conversion_g_l_h_per_sccm=GAS_FACTOR,
        logger_status_counts=json.dumps(frame.status.value_counts().to_dict(),ensure_ascii=False)))
    GAS_HOURLY_PARTS.append(hourly)
GAS_HOURLY=pd.concat(GAS_HOURLY_PARTS,ignore_index=True)
GAS_ONSETS=pd.DataFrame(GAS_ONSET_ROWS);GAS_SENSOR_AUDIT=pd.DataFrame(GAS_CALIBRATION_ROWS)
display(GAS_ONSETS.round(5))
display(GAS_SENSOR_AUDIT.drop(columns='logger_status_counts').round(6))
print('Inicio detectado al terminar tres intervalos completos:',dict(zip(GAS_ONSETS.lab,GAS_ONSETS.start_h)))
del GAS_RAW

### 23.1. Inicialización gaseosa y conservación de la química

En tₐ se toman X, Xd, N, G, F, E y glicerol de cada trayectoria química existente. No se vuelven a integrar ni a corregir con muestras posteriores. La referencia anterior permite ver el efecto del cambio de biomasa sobre el mismo período gaseoso.

El caudal inicial es la mediana de los tres bins usados para confirmar actividad, todos anteriores a tₐ. Cd(tₐ) se estima invirtiendo la ecuación de liberación continua y la ganancia antiguas:

\[
 y_a=g\,k\,C_{d,a}\left(0.05+0.95\frac{C_{d,a}}{C_{d,a}+C_{sat,a}}\right).
\]

Cd es una **estimación condicionada al modelo**, no una medición disuelta. Las tres curvas parten del mismo caudal por construcción; esa coincidencia inicial no valida sus estados químicos. O₂(tₐ) se propaga desde la condición inicial antigua usando la biomasa simulada; tampoco es una medición ni se supone cero.

Se usa la estructura simplificada de `co2_local_observability` y del diagnóstico postactividad: pool disuelto con liberación continua, O₂, rampa/gain de nutrición y transitorio multiplicativo en producción, sin término respiratorio separado. Los parámetros antiguos permanecen fijos. Las dos trayectorias con/sin nutrición, la rampa y la normalización antigua `grid_max` conservan el reloj y horizonte originales. `grid_max` sigue siendo una convención retrospectiva, no una normalización online final.

Desde tₐ el gate vale exactamente 1. El pool comienza con Cd estimado y conserva su dinámica de transferencia; no se vuelve a añadir un gate químico ni se cambia a `threshold release`. Se predice libremente desde entonces: el caudal futuro sólo se usa para evaluar.

In [ ]:
GAS_CURVES={};GAS_INITIAL_ROWS=[];GAS_CHECK_ROWS=[];GAS_NO_N_CORES={}
for lab in LABS:
    onset=GAS_ONSETS[GAS_ONSETS.lab.eq(lab)].iloc[0];start=float(onset.start_h)
    for case in GAS_CASES:
        original=W_CURVES[(lab,'theta_full',case)]
        core=original[list(base.STATE_NAMES)].copy()
        time=core.index.to_numpy(float);assert start in time and time[-1]==GAS_FINAL_H
        batch=replace(BATCHES[lab],initials={p:float(core[p].iloc[0]) for p in base.STATE_NAMES})
        no_n=replace(batch,pulses={**batch.pulses,'N':()})
        core0=diagnosis.integrate(no_n,THETA_SETS['theta_full'],time)
        GAS_NO_N_CORES[(lab,case)]=core0
        ramp,multiplier=gas_model.pulse_history(time,batch.pulses['N'],GAS_PARAMS)
        oxygen0=GAS_PARAMS['O2_initial_scale']*gas_cross.co2_model.o2_saturation_mg_l(
            base.temperature_at(batch,time[0]),batch.initials['E'],batch.initials['G'],batch.initials['F'],1.)
        # Only the oxygen history is retained from this full-clock layer evaluation.
        history=gas_model.finish(batch,THETA_SETS['theta_full'],core,core0,GAS_PARAMS,
            np.ones(len(time)),ramp,multiplier,0.,oxygen0)
        keep=time>=start;future_time=time[keep]
        active_core=core.loc[future_time];active_core0=core0.loc[future_time]
        saturation=float(gas_model.csat(batch,active_core.iloc[:1],GAS_PARAMS)[0])
        cd0=gas_model.dissolved_from_flow(onset.gas_initial_g_l_h,saturation,GAS_PARAMS)
        active=gas_model.finish(batch,THETA_SETS['theta_full'],active_core,active_core0,GAS_PARAMS,
            np.ones(len(future_time)),ramp[keep],multiplier[keep],cd0,float(history.loc[start,'O2']))
        GAS_CURVES[(lab,case)]=active
        assert np.array_equal(active[list(base.STATE_NAMES)].to_numpy(),original.loc[future_time,list(base.STATE_NAMES)].to_numpy())
        assert np.array_equal(active.ramp.to_numpy(),ramp[keep])
        assert np.array_equal(active.multiplier.to_numpy(),multiplier[keep])
        assert np.allclose(active.O2,history.loc[future_time,'O2'],rtol=0,atol=1e-12)
        assert active.gate.eq(1.).all()
        initial_flow_error=abs(float(active.predicted.iloc[0])-float(onset.gas_initial_g_l_h))
        assert initial_flow_error<1e-12
        pulse_errors=[]
        for event,dose in batch.pulses['N']:
            if event>=start:
                pulse_errors.append(abs(active.loc[event+diagnosis.CONFIG['after_event_h'],'N']-active.loc[event,'N']-dose))
        assert max(pulse_errors,default=0)<1e-5
        assert active.attrs['mass_error']<1e-10
        assert np.isfinite(active.to_numpy()).all() and (active[['Cd','O2','predicted','qprod','qgas']].to_numpy()>=0).all()
        GAS_INITIAL_ROWS.append(dict(lab=lab,case=case,start_h=start,
            initial_flow_g_l_h=onset.gas_initial_g_l_h,Cd_g_l=cd0,O2_mg_l=float(active.O2.iloc[0]),
            saturation_g_l=saturation,original_X0_g_l=float(core.X.iloc[0]),
            **{p+'_at_start':float(active[p].iloc[0]) for p in base.STATE_NAMES}))
        GAS_CHECK_ROWS.append(dict(lab=lab,case=case,start_h=float(active.index[0]),end_h=float(active.index[-1]),
            chemistry_max_difference_g_l=float(np.max(abs(active[list(base.STATE_NAMES)].to_numpy()-original.loc[future_time,list(base.STATE_NAMES)].to_numpy()))),
            pool_mass_error_g_l=active.attrs['mass_error'],initial_flow_error_g_l_h=initial_flow_error,
            oxygen_continuation_max_error_mg_l=float(np.max(abs(active.O2-history.loc[future_time,'O2']))),
            pulse_sample_error_g_l=max(pulse_errors,default=0),future_N_pulses=len(pulse_errors),
            gate_min=float(active.gate.min()),gate_max=float(active.gate.max()),
            gas_init_last_available_h=float(onset.last_confirmation_bin_end_h),
            no_n_inventory_drift_mg_l=float(np.max(abs(core0.N+THETA_SETS['theta_full']['qN']/THETA_SETS['theta_full']['mu0']*(core0.X+core0.Xd)-
                (core0.N.iloc[0]+THETA_SETS['theta_full']['qN']/THETA_SETS['theta_full']['mu0']*(core0.X.iloc[0]+core0.Xd.iloc[0])))))*1000))
    print(lab,'tₐ =',start,'h; tres inicializaciones gaseosas; química original conservada hasta 166 h',flush=True)
GAS_INITIALS=pd.DataFrame(GAS_INITIAL_ROWS);GAS_CHECKS=pd.DataFrame(GAS_CHECK_ROWS)
assert len(GAS_CURVES)==9 and GAS_CHECKS.future_N_pulses.eq(1).all()
assert GAS_CHECKS.no_n_inventory_drift_mg_l.max()<.002
display(GAS_INITIALS.round(6))
display(GAS_CHECKS.round(12))

### 23.2. Figura 1×3 y errores sobre las mismas mediciones posteriores

Cada panel comienza en su tₐ detectado y llega a 166 h. Se mantiene el reloj desde inoculación para situar correctamente los pulsos. Negro: caudal observado; azul: referencia anterior; verde: viable común; naranja: viable ponderada. No se representa ni puntúa la fase previa, y las mismas mediciones futuras se usan para las tres curvas.

Los puntos son medianas horarias y se dibujan en el centro de cada intervalo. Para calcular los errores se compara cada mediana medida con una mediana aproximada de la predicción dentro del mismo bin, usando ocho puntos interiores uniformes. Sólo se evalúan bins completos que comienzan en tₐ o después; ninguno se utilizó para inicializar Cd. También se informa cuántos puntos superan el umbral y su RMSE separado. El RMSE que incluye valores por debajo del umbral es descriptivo, no una verosimilitud de mediciones exactas en ese rango.

**Soporte de los datos:** en los tres LAB, el último bin válido termina a las 160 h; el siguiente tiene sólo unos minutos registrados y se excluye. La simulación continúa hasta 166 h, pero los errores se calculan únicamente donde existen mediciones válidas. Los bins que contienen la hora del pulso entran en la métrica completa y se excluyen de los subconjuntos estrictamente pre/post pulso.

In [ ]:
GAS_RESIDUAL_PARTS=[];GAS_METRIC_ROWS=[]
for lab in LABS:
    onset=GAS_ONSETS[GAS_ONSETS.lab.eq(lab)].iloc[0];start=float(onset.start_h)
    pulse=min(t for t,d in BATCHES[lab].pulses['N'])
    observed=GAS_HOURLY[GAS_HOURLY.lab.eq(lab)&GAS_HOURLY.valid&(GAS_HOURLY.bin_start_h>=start)].copy()
    assert len(observed)>0 and observed.bin_start_h.min()>=onset.last_confirmation_bin_end_h
    for case in GAS_CASES:
        curve=GAS_CURVES[(lab,case)];evaluation=observed.copy()
        evaluation['predicted_g_l_h']=[float(np.median(np.interp(t+(np.arange(8)+.5)/8.,curve.index,curve.predicted))) for t in observed.bin_start_h]
        evaluation['residual_g_l_h']=evaluation.predicted_g_l_h-evaluation.co2_observed_g_l_h
        evaluation['case']=case;evaluation['theta']='theta_full';GAS_RESIDUAL_PARTS.append(evaluation)
        for window,keep in [('completo',np.ones(len(evaluation),bool)),
            ('pre_pulso',evaluation.bin_end_h<=pulse),('post_pulso',evaluation.bin_start_h>=pulse)]:
            g=evaluation.loc[keep]
            if g.empty: continue
            r=g.residual_g_l_h.to_numpy();q=g[g.quantifiable].residual_g_l_h.to_numpy()
            GAS_METRIC_ROWS.append(dict(lab=lab,case=case,theta='theta_full',window=window,n=len(g),
                RMSE_g_l_h=float(np.sqrt(np.mean(r*r))),bias_g_l_h=float(r.mean()),MAE_g_l_h=float(np.mean(abs(r))),
                n_quantifiable=len(q),RMSE_quantifiable_g_l_h=float(np.sqrt(np.mean(q*q))) if len(q) else np.nan,
                first_bin_start_h=float(g.bin_start_h.min()),last_bin_end_h=float(g.bin_end_h.max())))
GAS_RESIDUALS=pd.concat(GAS_RESIDUAL_PARTS,ignore_index=True)
GAS_METRICS=pd.DataFrame(GAS_METRIC_ROWS)
previous=GAS_METRICS[GAS_METRICS.case.eq('referencia_anterior')][['lab','window','RMSE_g_l_h']].rename(columns={'RMSE_g_l_h':'RMSE_previous_g_l_h'})
GAS_METRICS=GAS_METRICS.merge(previous,on=['lab','window'],validate='many_to_one')
GAS_METRICS['ratio_to_previous']=GAS_METRICS.RMSE_g_l_h/GAS_METRICS.RMSE_previous_g_l_h
fig,axes=plt.subplots(1,3,figsize=(16,4.8),layout='constrained')
for ax,lab in zip(axes,LABS):
    onset=GAS_ONSETS[GAS_ONSETS.lab.eq(lab)].iloc[0];start=float(onset.start_h)
    observed=GAS_HOURLY[GAS_HOURLY.lab.eq(lab)&GAS_HOURLY.valid&(GAS_HOURLY.bin_start_h>=start)]
    for case in GAS_CASES:
        curve=GAS_CURVES[(lab,case)];label,color,style=W_CASES[case]
        ax.plot(curve.index,curve.predicted,color=color,ls=style,lw=1.6,label=label)
    ax.plot(observed.plot_t_h,observed.co2_observed_g_l_h,'o',color='black',ms=2.5,label='CO₂ observado',zorder=6)
    mark_events(ax,lab)
    ax.set(title=f'{lab} · inicio {start:g} h',xlabel='t desde inoculación [h]',ylabel='g CO₂/L/h',xlim=(start,GAS_FINAL_H),ylim=(0,None))
fig.suptitle('CO₂ desde actividad detectada · theta_full · parámetros antiguos',y=.995)
fig.get_layout_engine().set(rect=(0,0,1,.84))
handles,labels=axes[0].get_legend_handles_labels()
fig.legend(handles,labels,loc='upper center',bbox_to_anchor=(.5,.955),ncol=5,fontsize=8)
fig.canvas.draw()
assert all(not fig._suptitle.get_window_extent(fig.canvas.get_renderer()).overlaps(legend.get_window_extent(fig.canvas.get_renderer())) for legend in fig.legends)
GAS_FIGURE_PATH=GAS_OUT/'01_co2_viable_common_weighted_theta_full.png'
fig.savefig(GAS_FIGURE_PATH,dpi=140,bbox_inches='tight',facecolor='white')
buffer=io.BytesIO();fig.savefig(buffer,format='png',dpi=110,bbox_inches='tight',facecolor='white')
plt.close(fig);display(Image(data=buffer.getvalue()));buffer.close()
display(GAS_METRICS[['lab','case','window','n','RMSE_g_l_h','bias_g_l_h','n_quantifiable','RMSE_quantifiable_g_l_h','ratio_to_previous']].round(5))

### 23.3. Lectura y límites de esta primera comparación

El caudal es una medición online del gas liberado, no una balanza ni una medición directa de la producción celular. La capa gaseosa transforma la producción química mediante O₂, la respuesta de nutrición y el reservorio disuelto. Por eso mejorar N/GF/E no obliga a mejorar el caudal, y una mejora de caudal no valida por sí sola todos los estados químicos.

Esta figura evalúa el cambio de biomasa con la capa antigua fija. No identifica parámetros, no elige el inicio por el menor error posterior y no estima estados con el caudal futuro. La temperatura futura y `grid_max` hacen que la predicción sea retrospectiva; además, la química y O₂ heredan hipótesis del período anterior a tₐ. No se presenta como un EKF ni como validación de una inicialización online completa. La amplitud inicial coincidente procede de la inversión de Cd; las diferencias posteriores sí son predicciones libres.

In [ ]:
GAS_CONCLUSIONS=[]
for lab in LABS:
    g=GAS_METRICS.query('lab==@lab and window=="completo"').set_index('case')
    reference=float(g.loc['referencia_anterior','RMSE_g_l_h'])
    details=[]
    for case in GAS_CASES:
        row=g.loc[case]
        change='referencia' if case=='referencia_anterior' else ('mejora' if row.RMSE_g_l_h<reference else 'empeora')+' frente a la referencia anterior'
        details.append(f'{W_CASES[case][0]}: RMSE {row.RMSE_g_l_h:.4f} g CO₂/L/h; sesgo {row.bias_g_l_h:+.4f}; {change}')
    GAS_CONCLUSIONS.append(f'**{lab}:** '+ '; '.join(details)+'.')
GAS_CONCLUSIONS.append('Los errores corresponden a los mismos bins futuros y al horizonte completo, con el pulso incluido. '
    'Un menor RMSE sólo establece mejor concordancia de caudal bajo esta inicialización y esta capa de CO₂. '
    'No confirma cuál biomasa es verdadera ni resuelve los errores químicos heredados antes de la detección.')
display(Markdown('\n\n'.join(GAS_CONCLUSIONS)))
assert GAS_INPUT_HASHES=={str(p.relative_to(ROOT)):sha256(p) for p in GAS_INPUT_FILES}
assert GAS_PRIOR_HASHES=={str(p.relative_to(ROOT)):sha256(p) for p in GAS_PRIOR_FILES}
GAS_TABLES=dict(onset_detection=GAS_ONSETS,sensor_audit=GAS_SENSOR_AUDIT,
    hourly_observations=GAS_HOURLY,initialization=GAS_INITIALS,parameters=GAS_PARAMETER_TABLE,
    checks=GAS_CHECKS,metrics=GAS_METRICS,residuals=GAS_RESIDUALS)
for name,table in GAS_TABLES.items(): table.to_csv(GAS_OUT/(name+'.csv'),index=False)
arrays={};array_manifest=[]
for i,(key,curve) in enumerate(GAS_CURVES.items()):
    name=f'case_{i}';arrays[name]=np.column_stack([curve.index,curve.to_numpy()])
    array_manifest.append(dict(array=name,key=list(key),columns=['t_h']+list(curve.columns)))
for i,(key,curve) in enumerate(GAS_NO_N_CORES.items()):
    arrays[f'no_n_{i}']=np.column_stack([curve.index,curve.to_numpy()])
    array_manifest.append(dict(array=f'no_n_{i}',key=['no_n',*key],columns=['t_h']+list(curve.columns)))
np.savez_compressed(GAS_OUT/'trajectories.npz',**arrays)
GAS_MANIFEST=dict(created_at=datetime.now().astimezone().isoformat(),python=sys.executable,
    source_code_sha256=sha256(LAB2026/'notebooks/diagnostics/nitrogen_consumption_diagnosis.source.ipynb'),
    input_hashes=GAS_INPUT_HASHES,prior_result_hashes=GAS_PRIOR_HASHES,
    theta_name='theta_full',theta=THETA_SETS['theta_full'],co2_parameters=GAS_PARAMS,
    structure='simplified co2_local_observability / _post_activity_prediction; continuous pool release; no separate respiratory source',
    activity_threshold_g_l_h=GAS_THRESHOLD,consecutive_completed_bins=GAS_CONSECUTIVE,
    detector='first three consecutive valid hourly medians above threshold; start at third bin end',
    detector_online_validation=False,logger_filter_causality='not independently verified',
    gate_after_detection=1.,chemistry_reinitialized=False,chemistry_numerically_unchanged=True,
    gas_initialization='past confirmation-bin median inverted through fixed release/gain equation',
    oxygen_initialization='propagated nominal full-clock oxygen history, not measured',
    bin_prediction='median of eight uniformly spaced interior model predictions in the same hourly bin',
    calibration_factor_g_l_h_per_sccm=GAS_FACTOR,arrays=array_manifest,
    new_no_N_integrations=len(GAS_NO_N_CORES),gas_curves=len(GAS_CURVES),
    figures=[str(GAS_FIGURE_PATH.relative_to(ROOT))],simulation_end_h=GAS_FINAL_H,
    optimizer_calls=0,kalman_updates=0,parameters_refitted=False,
    pulse_normalization='historical grid_max on original full grid; retrospective',
    observation_policy=GF_ZERO_POLICY)
(GAS_OUT/'manifest.json').write_text(json.dumps(GAS_MANIFEST,ensure_ascii=False,indent=2),encoding='utf-8')
print('Verificado: 9 curvas gaseosas, figura 1×3, química previa idéntica y pulsos completos hasta 166 h.')
print('Inicio y Cd usan sólo datos disponibles hasta tₐ; predicción libre posterior. Sin reajuste, EKF, commit ni push.')

### 23.4. Figura adicional con theta_defaults

Se conserva la figura anterior con theta_full y se añade **otra figura 1×3**, ahora con **theta_defaults** para las tres inicializaciones: referencia anterior, viable común y viable ponderada. theta_defaults es el vector incompleto completado con los seis valores por defecto descritos en §1; no se ajusta de nuevo.

Se utilizan las trayectorias químicas de theta_defaults ya calculadas en 16.2. La hora detectada y el caudal previo usados para iniciar Cd son exactamente los mismos que en 23.1; se mantienen los parámetros antiguos de la capa gaseosa, el gate=1, el reloj de nutrición y el horizonte de 166 h. O₂ y Cd se calculan con la química propia de theta_defaults, sin copiar los estados de theta_full en tₐ.

Los colores y las convenciones son los de la figura anterior. Se puntúan los mismos bins futuros para poder comparar ambos theta. La tabla final incluye el ratio de RMSE defaults/full **para la misma inicialización de biomasa**, además de la comparación interna con la referencia anterior de theta_defaults. La coincidencia del caudal inicial sigue siendo impuesta por la inversión de Cd, no una validación independiente de la química.

In [ ]:
GAS_DEFAULT_CURVES={};GAS_DEFAULT_INITIAL_ROWS=[];GAS_DEFAULT_CHECK_ROWS=[];GAS_DEFAULT_NO_N_CORES={}
for lab in LABS:
    onset=GAS_ONSETS[GAS_ONSETS.lab.eq(lab)].iloc[0];start=float(onset.start_h)
    for case in GAS_CASES:
        original=W_CURVES[(lab,'theta_defaults',case)]
        core=original[list(base.STATE_NAMES)].copy()
        time=core.index.to_numpy(float);assert start in time and time[-1]==GAS_FINAL_H
        batch=replace(BATCHES[lab],initials={p:float(core[p].iloc[0]) for p in base.STATE_NAMES})
        no_n=replace(batch,pulses={**batch.pulses,'N':()})
        core0=diagnosis.integrate(no_n,THETA_SETS['theta_defaults'],time)
        GAS_DEFAULT_NO_N_CORES[(lab,case)]=core0
        ramp,multiplier=gas_model.pulse_history(time,batch.pulses['N'],GAS_PARAMS)
        oxygen0=GAS_PARAMS['O2_initial_scale']*gas_cross.co2_model.o2_saturation_mg_l(
            base.temperature_at(batch,time[0]),batch.initials['E'],batch.initials['G'],batch.initials['F'],1.)
        # Only the oxygen history is retained from this full-clock layer evaluation.
        history=gas_model.finish(batch,THETA_SETS['theta_defaults'],core,core0,GAS_PARAMS,
            np.ones(len(time)),ramp,multiplier,0.,oxygen0)
        keep=time>=start;future_time=time[keep]
        active_core=core.loc[future_time];active_core0=core0.loc[future_time]
        saturation=float(gas_model.csat(batch,active_core.iloc[:1],GAS_PARAMS)[0])
        cd0=gas_model.dissolved_from_flow(onset.gas_initial_g_l_h,saturation,GAS_PARAMS)
        active=gas_model.finish(batch,THETA_SETS['theta_defaults'],active_core,active_core0,GAS_PARAMS,
            np.ones(len(future_time)),ramp[keep],multiplier[keep],cd0,float(history.loc[start,'O2']))
        GAS_DEFAULT_CURVES[(lab,case)]=active
        assert np.array_equal(active[list(base.STATE_NAMES)].to_numpy(),original.loc[future_time,list(base.STATE_NAMES)].to_numpy())
        assert np.array_equal(active.ramp.to_numpy(),ramp[keep])
        assert np.array_equal(active.multiplier.to_numpy(),multiplier[keep])
        assert np.allclose(active.O2,history.loc[future_time,'O2'],rtol=0,atol=1e-12)
        assert active.gate.eq(1.).all()
        initial_flow_error=abs(float(active.predicted.iloc[0])-float(onset.gas_initial_g_l_h))
        assert initial_flow_error<1e-12
        pulse_errors=[]
        for event,dose in batch.pulses['N']:
            if event>=start:
                pulse_errors.append(abs(active.loc[event+diagnosis.CONFIG['after_event_h'],'N']-active.loc[event,'N']-dose))
        assert max(pulse_errors,default=0)<1e-5
        assert active.attrs['mass_error']<1e-10
        assert np.isfinite(active.to_numpy()).all() and (active[['Cd','O2','predicted','qprod','qgas']].to_numpy()>=0).all()
        GAS_DEFAULT_INITIAL_ROWS.append(dict(lab=lab,case=case,start_h=start,
            initial_flow_g_l_h=onset.gas_initial_g_l_h,Cd_g_l=cd0,O2_mg_l=float(active.O2.iloc[0]),
            saturation_g_l=saturation,original_X0_g_l=float(core.X.iloc[0]),
            **{p+'_at_start':float(active[p].iloc[0]) for p in base.STATE_NAMES}))
        GAS_DEFAULT_CHECK_ROWS.append(dict(lab=lab,case=case,start_h=float(active.index[0]),end_h=float(active.index[-1]),
            chemistry_max_difference_g_l=float(np.max(abs(active[list(base.STATE_NAMES)].to_numpy()-original.loc[future_time,list(base.STATE_NAMES)].to_numpy()))),
            pool_mass_error_g_l=active.attrs['mass_error'],initial_flow_error_g_l_h=initial_flow_error,
            oxygen_continuation_max_error_mg_l=float(np.max(abs(active.O2-history.loc[future_time,'O2']))),
            pulse_sample_error_g_l=max(pulse_errors,default=0),future_N_pulses=len(pulse_errors),
            gate_min=float(active.gate.min()),gate_max=float(active.gate.max()),
            gas_init_last_available_h=float(onset.last_confirmation_bin_end_h),
            no_n_inventory_drift_mg_l=float(np.max(abs(core0.N+THETA_SETS['theta_defaults']['qN']/THETA_SETS['theta_defaults']['mu0']*(core0.X+core0.Xd)-
                (core0.N.iloc[0]+THETA_SETS['theta_defaults']['qN']/THETA_SETS['theta_defaults']['mu0']*(core0.X.iloc[0]+core0.Xd.iloc[0])))))*1000))
    print(lab,'theta_defaults · tₐ =',start,'h; tres inicializaciones gaseosas; química original conservada hasta 166 h',flush=True)
GAS_DEFAULT_INITIALS=pd.DataFrame(GAS_DEFAULT_INITIAL_ROWS);GAS_DEFAULT_CHECKS=pd.DataFrame(GAS_DEFAULT_CHECK_ROWS)
assert len(GAS_DEFAULT_CURVES)==9 and GAS_DEFAULT_CHECKS.future_N_pulses.eq(1).all()
assert GAS_DEFAULT_CHECKS.no_n_inventory_drift_mg_l.max()<.002
display(GAS_DEFAULT_INITIALS.round(6))
display(GAS_DEFAULT_CHECKS.round(12))

In [ ]:
GAS_DEFAULT_RESIDUAL_PARTS=[];GAS_DEFAULT_METRIC_ROWS=[]
for lab in LABS:
    onset=GAS_ONSETS[GAS_ONSETS.lab.eq(lab)].iloc[0];start=float(onset.start_h)
    pulse=min(t for t,d in BATCHES[lab].pulses['N'])
    observed=GAS_HOURLY[GAS_HOURLY.lab.eq(lab)&GAS_HOURLY.valid&(GAS_HOURLY.bin_start_h>=start)].copy()
    assert len(observed)>0 and observed.bin_start_h.min()>=onset.last_confirmation_bin_end_h
    for case in GAS_CASES:
        curve=GAS_DEFAULT_CURVES[(lab,case)];evaluation=observed.copy()
        evaluation['predicted_g_l_h']=[float(np.median(np.interp(t+(np.arange(8)+.5)/8.,curve.index,curve.predicted))) for t in observed.bin_start_h]
        evaluation['residual_g_l_h']=evaluation.predicted_g_l_h-evaluation.co2_observed_g_l_h
        evaluation['case']=case;evaluation['theta']='theta_defaults';GAS_DEFAULT_RESIDUAL_PARTS.append(evaluation)
        for window,keep in [('completo',np.ones(len(evaluation),bool)),
            ('pre_pulso',evaluation.bin_end_h<=pulse),('post_pulso',evaluation.bin_start_h>=pulse)]:
            g=evaluation.loc[keep]
            if g.empty: continue
            r=g.residual_g_l_h.to_numpy();q=g[g.quantifiable].residual_g_l_h.to_numpy()
            GAS_DEFAULT_METRIC_ROWS.append(dict(lab=lab,case=case,theta='theta_defaults',window=window,n=len(g),
                RMSE_g_l_h=float(np.sqrt(np.mean(r*r))),bias_g_l_h=float(r.mean()),MAE_g_l_h=float(np.mean(abs(r))),
                n_quantifiable=len(q),RMSE_quantifiable_g_l_h=float(np.sqrt(np.mean(q*q))) if len(q) else np.nan,
                first_bin_start_h=float(g.bin_start_h.min()),last_bin_end_h=float(g.bin_end_h.max())))
GAS_DEFAULT_RESIDUALS=pd.concat(GAS_DEFAULT_RESIDUAL_PARTS,ignore_index=True)
GAS_DEFAULT_METRICS=pd.DataFrame(GAS_DEFAULT_METRIC_ROWS)
previous=GAS_DEFAULT_METRICS[GAS_DEFAULT_METRICS.case.eq('referencia_anterior')][['lab','window','RMSE_g_l_h']].rename(columns={'RMSE_g_l_h':'RMSE_previous_g_l_h'})
GAS_DEFAULT_METRICS=GAS_DEFAULT_METRICS.merge(previous,on=['lab','window'],validate='many_to_one')
GAS_DEFAULT_METRICS['ratio_to_previous']=GAS_DEFAULT_METRICS.RMSE_g_l_h/GAS_DEFAULT_METRICS.RMSE_previous_g_l_h
fig,axes=plt.subplots(1,3,figsize=(16,4.8),layout='constrained')
for ax,lab in zip(axes,LABS):
    onset=GAS_ONSETS[GAS_ONSETS.lab.eq(lab)].iloc[0];start=float(onset.start_h)
    observed=GAS_HOURLY[GAS_HOURLY.lab.eq(lab)&GAS_HOURLY.valid&(GAS_HOURLY.bin_start_h>=start)]
    for case in GAS_CASES:
        curve=GAS_DEFAULT_CURVES[(lab,case)];label,color,style=W_CASES[case]
        ax.plot(curve.index,curve.predicted,color=color,ls=style,lw=1.6,label=label)
    ax.plot(observed.plot_t_h,observed.co2_observed_g_l_h,'o',color='black',ms=2.5,label='CO₂ observado',zorder=6)
    mark_events(ax,lab)
    ax.set(title=f'{lab} · inicio {start:g} h',xlabel='t desde inoculación [h]',ylabel='g CO₂/L/h',xlim=(start,GAS_FINAL_H),ylim=(0,None))
fig.suptitle('CO₂ desde actividad detectada · theta_defaults · parámetros antiguos',y=.995)
fig.get_layout_engine().set(rect=(0,0,1,.84))
handles,labels=axes[0].get_legend_handles_labels()
fig.legend(handles,labels,loc='upper center',bbox_to_anchor=(.5,.955),ncol=5,fontsize=8)
fig.canvas.draw()
assert all(not fig._suptitle.get_window_extent(fig.canvas.get_renderer()).overlaps(legend.get_window_extent(fig.canvas.get_renderer())) for legend in fig.legends)
GAS_DEFAULT_FIGURE_PATH=GAS_OUT/'02_co2_viable_common_weighted_theta_defaults.png'
fig.savefig(GAS_DEFAULT_FIGURE_PATH,dpi=140,bbox_inches='tight',facecolor='white')
buffer=io.BytesIO();fig.savefig(buffer,format='png',dpi=110,bbox_inches='tight',facecolor='white')
plt.close(fig);display(Image(data=buffer.getvalue()));buffer.close()
display(GAS_DEFAULT_METRICS[['lab','case','window','n','RMSE_g_l_h','bias_g_l_h','n_quantifiable','RMSE_quantifiable_g_l_h','ratio_to_previous']].round(5))

GAS_DEFAULT_COMPARISON=GAS_DEFAULT_METRICS.merge(
    GAS_METRICS[['lab','case','window','n','RMSE_g_l_h','bias_g_l_h']].rename(columns={
        'n':'n_theta_full','RMSE_g_l_h':'RMSE_theta_full_g_l_h','bias_g_l_h':'bias_theta_full_g_l_h'}),
    on=['lab','case','window'],validate='one_to_one')
assert GAS_DEFAULT_COMPARISON.n.eq(GAS_DEFAULT_COMPARISON.n_theta_full).all()
GAS_DEFAULT_COMPARISON['ratio_defaults_full']=GAS_DEFAULT_COMPARISON.RMSE_g_l_h/GAS_DEFAULT_COMPARISON.RMSE_theta_full_g_l_h
display(GAS_DEFAULT_COMPARISON.query('window=="completo"')[
    ['lab','case','n','RMSE_theta_full_g_l_h','RMSE_g_l_h','ratio_defaults_full','bias_g_l_h']].round(5))
GAS_DEFAULT_CONCLUSIONS=[]
for lab in LABS:
    group=GAS_DEFAULT_COMPARISON.query('lab==@lab and window=="completo"')
    desc='; '.join(f'{W_CASES[row.case][0]}: RMSE defaults {row.RMSE_g_l_h:.4f}, '
        f'full {row.RMSE_theta_full_g_l_h:.4f} g CO₂/L/h '
        f'({"mejora" if row.ratio_defaults_full<1 else "empeora"} con defaults)'
        for row in group.itertuples())
    GAS_DEFAULT_CONCLUSIONS.append(f'**{lab}:** '+desc+'.')
GAS_DEFAULT_CONCLUSIONS.append('Los ratios comparan la misma biomasa inicial, los mismos parámetros de CO₂ '
    'y los mismos intervalos futuros. Una mejora de caudal no demuestra una mejora simultánea de N, GF y E; '
    'se deben contrastar con los diagnósticos químicos anteriores.')
display(Markdown('\n\n'.join(GAS_DEFAULT_CONCLUSIONS)))
assert GAS_INPUT_HASHES=={str(p.relative_to(ROOT)):sha256(p) for p in GAS_INPUT_FILES}
assert GAS_PRIOR_HASHES=={str(p.relative_to(ROOT)):sha256(p) for p in GAS_PRIOR_FILES}
GAS_DEFAULT_TABLES=dict(initialization=GAS_DEFAULT_INITIALS,checks=GAS_DEFAULT_CHECKS,
    metrics=GAS_DEFAULT_METRICS,residuals=GAS_DEFAULT_RESIDUALS,comparison_full=GAS_DEFAULT_COMPARISON)
for name,table in GAS_DEFAULT_TABLES.items(): table.to_csv(GAS_OUT/('defaults_'+name+'.csv'),index=False)
arrays={};array_manifest=[]
for i,(key,curve) in enumerate(GAS_DEFAULT_CURVES.items()):
    name=f'case_{i}';arrays[name]=np.column_stack([curve.index,curve.to_numpy()])
    array_manifest.append(dict(array=name,key=list(key),columns=['t_h']+list(curve.columns)))
for i,(key,curve) in enumerate(GAS_DEFAULT_NO_N_CORES.items()):
    name=f'no_n_{i}';arrays[name]=np.column_stack([curve.index,curve.to_numpy()])
    array_manifest.append(dict(array=name,key=['no_n',*key],columns=['t_h']+list(curve.columns)))
np.savez_compressed(GAS_OUT/'defaults_trajectories.npz',**arrays)
GAS_DEFAULT_MANIFEST=dict(created_at=datetime.now().astimezone().isoformat(),python=sys.executable,
    source_code_sha256=sha256(LAB2026/'notebooks/diagnostics/nitrogen_consumption_diagnosis.source.ipynb'),
    theta_name='theta_defaults',theta=THETA_SETS['theta_defaults'],co2_parameters=GAS_PARAMS,
    input_hashes=GAS_INPUT_HASHES,prior_result_hashes=GAS_PRIOR_HASHES,
    arrays=array_manifest,gas_curves=len(GAS_DEFAULT_CURVES),no_N_integrations=len(GAS_DEFAULT_NO_N_CORES),
    figure=str(GAS_DEFAULT_FIGURE_PATH.relative_to(ROOT)),starts_h=dict(zip(GAS_ONSETS.lab,GAS_ONSETS.start_h)),
    gate_after_detection=1.,simulation_end_h=GAS_FINAL_H,optimizer_calls=0,kalman_updates=0,
    full_figure_preserved=True,chemical_trajectories_preserved=True,parameters_refitted=False,
    theta_difference='six inherited defaults: sN,qXG,qXF,sG,sF,m0; same initial biomass per case',
    observation_policy=GF_ZERO_POLICY)
(GAS_OUT/'defaults_manifest.json').write_text(json.dumps(GAS_DEFAULT_MANIFEST,ensure_ascii=False,indent=2),encoding='utf-8')
primary_manifest=json.loads((GAS_OUT/'manifest.json').read_text(encoding='utf-8-sig'))
primary_manifest['theta_defaults_extension']=dict(manifest='defaults_manifest.json',
    figure=str(GAS_DEFAULT_FIGURE_PATH.relative_to(ROOT)),section='23.4',gas_curves=len(GAS_DEFAULT_CURVES))
(GAS_OUT/'manifest.json').write_text(json.dumps(primary_manifest,ensure_ascii=False,indent=2),encoding='utf-8')
print('Figura theta_defaults añadida por separado: 9 curvas, misma actividad detectada y mismos datos futuros.')
print('Figura y resultados theta_full conservados; química previa intacta; sin commit ni push.')


## 24. Activación causal a partir del consumo predicho de G+F

Se prueba la activación propuesta con **theta_full y theta_defaults**, **viable común y ponderada**, parámetros numéricos antiguos de CO₂ y el horizonte completo **0–166 h, con todos los pulsos**. Las figuras anteriores de §23 se conservan. Aquí se simula desde inoculación: no se reinicia el pool en la hora detectada, no se estima Cd a partir del caudal observado y no se desplazan las condiciones iniciales.

En cada instante se calcula únicamente con el estado químico predicho actual:

\[
c(t)=G_0+F_0-[\hat G(t)+\hat F(t)],\qquad
z(t)=\operatorname{clip}\left(\frac{c(t)-2}{6-2},0,1\right),\qquad
a(t)=3z(t)^2-2z(t)^3.
\]

**2 y 6 g/L son umbrales exploratorios fijados previamente**, no ajustados con los resultados. a=0 hasta consumir 2 g/L; a=1 después de 6 g/L. No intervienen un tiempo de cruce futuro, una muestra posterior, la duración final ni un porcentaje del máximo observado. La activación se evalúa sobre el consumo predicho, no sobre G+F interpolado de mediciones. Las trayectorias de este ensayo son monótonas en G+F; para un EKF con correcciones fluctuantes habría que especificar aparte la persistencia o histéresis.

**Control que aísla el efecto:** se simula también a=1 con exactamente la misma química, Cd inicial, O₂, parámetros y transitorio. La diferencia entre estas dos curvas sí corresponde al gate. La comparación con §23 se informa sobre los mismos bins posteriores, pero no aísla sólo el gate: allí el pool se inicializaba con caudal pasado y aquí se inicia desde inoculación.

**Causalidad del transitorio:** en este ensayo se reemplaza `grid_max` por el pico analítico del kernel, conservando A=0,30, τr=4 h y τd=12 h. El pico ocurre en u*=τr·ln(1+τd/τr) y su valor se obtiene de los parámetros, sin recorrer el horizonte futuro. Se aplica también al control a=1. Se guarda además un control con el gate 2–6 y `grid_max` para cuantificar por separado el efecto de esa normalización; no se usa para afirmar causalidad.

**Límite físico:** se conserva la química de §16.2 y el gate modifica sólo la fuente en la capa de CO₂. Es una prueba del criterio de activación, no una modificación coherente de todos los balances fermentativos. Si el modelo produce E antes de activar CO₂, puede alterarse su proporción CO₂/E. Se informa la reducción acumulada de la fuente respecto de a=1; conservar el balance del pool no resuelve esa cuestión estequiométrica. Ningún parámetro se reajusta y no se ejecuta un EKF.

In [ ]:
CA_OUT=OUT/'co2_causal_sugar_activation';CA_OUT.mkdir(exist_ok=True)
CA_CASES=('viable_comun_volumen','viable_ponderada')
CA_VARIANTS=('causal_2_6','sin_gate','causal_2_6_grid_max')
CA_LOW=2.;CA_HIGH=6.;CA_FINAL=166.
CA_PARAMS={**GAS_PARAMS,'kernel_normalization':'analytic_peak'}
assert {k for k in GAS_PARAMS if CA_PARAMS[k]!=GAS_PARAMS[k]}=={'kernel_normalization'}
CA_PRIOR_FILES=[p for p in OUT.rglob('*') if p.suffix in ('.csv','.npz','.png') and CA_OUT not in p.parents and (OUT/'co2_hard_sugar_activation') not in p.parents]
CA_PRIOR_HASHES={str(p.relative_to(ROOT)):sha256(p) for p in CA_PRIOR_FILES}
CA_CURVES={};CA_NO_N={};CA_CHECK_ROWS=[];CA_PREFIX_ROWS=[];CA_FEATURE_ROWS=[]
CA_RESIDUAL_PARTS=[];CA_METRIC_ROWS=[];CA_FIGURES=[]

def ca_activation(gf,gf0):
    gf=np.asarray(gf,float)
    assert np.isfinite(gf).all() and np.isfinite(gf0)
    z=np.clip((float(gf0)-gf-CA_LOW)/(CA_HIGH-CA_LOW),0.,1.)
    return z*z*(3.-2.*z)

def ca_pulse_history(time,schedule):
    time=np.asarray(time,float)
    ramp=np.clip((time-schedule[0][0])/CA_PARAMS['pulse_t_rise_h'],0.,1.) if schedule else np.zeros(len(time))
    multiplier=np.ones(len(time))
    rise,decay=CA_PARAMS['tau_rise_h'],CA_PARAMS['tau_decay_h']
    u_peak=rise*np.log1p(decay/rise)
    peak=-np.expm1(-u_peak/rise)*np.exp(-u_peak/decay)
    assert peak>0
    for event,dose in schedule:
        u=np.maximum(time-event,0.)
        kernel=-np.expm1(-u/rise)*np.exp(-u/decay)/peak
        multiplier+=CA_PARAMS['A']*dose/CA_PARAMS['N_ref_g_l']*kernel
    return ramp,multiplier

def ca_crossing(time,values,level):
    indices=np.flatnonzero(np.asarray(values)>=level)
    if not len(indices): return np.nan
    i=int(indices[0])
    if i==0: return float(time[0])
    before,after=float(values[i-1]),float(values[i])
    return float(time[i-1]+(time[i]-time[i-1])*(level-before)/(after-before)) if after>before else float(time[i])

def ca_bin_prediction(curve,starts):
    return np.array([np.median(np.interp(t+(np.arange(8)+.5)/8.,curve.index,curve.predicted)) for t in starts],float)

def ca_gas_confirmation(curve):
    starts=np.arange(0.,CA_FINAL,1.)
    predicted=ca_bin_prediction(curve,starts)
    for i in range(len(starts)-GAS_CONSECUTIVE+1):
        if (predicted[i:i+GAS_CONSECUTIVE]>=GAS_THRESHOLD).all(): return float(starts[i+GAS_CONSECUTIVE-1]+1.)
    return np.nan

def ca_save(fig,name):
    fig.canvas.draw()
    if fig._suptitle is not None:
        title_box=fig._suptitle.get_window_extent(fig.canvas.get_renderer())
        assert all(not title_box.overlaps(legend.get_window_extent(fig.canvas.get_renderer())) for legend in fig.legends)
    path=CA_OUT/(name+'.png');fig.savefig(path,dpi=140,bbox_inches='tight',facecolor='white')
    buffer=io.BytesIO();fig.savefig(buffer,format='png',dpi=110,bbox_inches='tight',facecolor='white')
    plt.close(fig);display(Image(data=buffer.getvalue()));buffer.close()
    CA_FIGURES.append(dict(name=name,path=str(path.relative_to(ROOT))))

display(pd.DataFrame([dict(parameter=k,old_value=GAS_PARAMS[k],causal_test_value=v) for k,v in CA_PARAMS.items()]))
print('Gate: consumo predicho de 2 a 6 g/L. Cd(0)=0; O₂ inicial según parámetros antiguos. Sin inicialización con caudal.')

### 24.1. Simulaciones y comprobación por prefijos

Se reutilizan los estados químicos y los contrafactuales sin nutrición ya calculados. Para cada theta/LAB/biomasa se evalúan: gate causal 2–6 con pico analítico, control a=1 con pico analítico y auditoría del gate 2–6 con `grid_max`. Todos usan Cd(0)=0 como la convención de la capa original; no se afirma que sea una medición. O₂ inicial conserva su formulación antigua.

La prueba por prefijos vuelve a evaluar la capa usando sólo la trayectoria hasta 12, 30, 60 y 100 h y comprueba que gate, transitorio, O₂, fuente, pool y caudal coinciden con el prefijo de la ejecución hasta 166 h. Comprueba que el cálculo de la capa no necesita el sufijo futuro **cuando los estados y entradas del prefijo son iguales**; no prueba que la cinética sea correcta. La temperatura sigue siendo la medida retrospectiva y la química procede del modelo congelado: una aplicación online necesitaría las entradas disponibles o previstas en cada momento.

Los tiempos de cruce de 2/6 g/L y el inicio gaseoso predicho se calculan **después, para informar resultados**; no se usan para construir a(t). La confirmación gaseosa predicha exige tres bins horarios del modelo sobre 0,05 g/L/h; la observada conserva la máscara de calidad y la disponibilidad de datos de §23.

In [ ]:
for theta,th in THETA_SETS.items():
    no_n_map=GAS_NO_N_CORES if theta=='theta_full' else GAS_DEFAULT_NO_N_CORES
    for lab in LABS:
        onset=GAS_ONSETS[GAS_ONSETS.lab.eq(lab)].iloc[0]
        for case in CA_CASES:
            original=W_CURVES[(lab,theta,case)]
            core=original[list(base.STATE_NAMES)].copy();core0=no_n_map[(lab,case)].copy()
            CA_NO_N[(lab,theta,case)]=core0
            assert np.array_equal(core.index,core0.index)
            time=core.index.to_numpy(float);assert time[0]==0 and time[-1]==CA_FINAL
            gf=(core.G+core.F).to_numpy();gf0=float(gf[0]);consumed=gf0-gf
            gate=ca_activation(gf,gf0)
            assert gate[0]==0 and ((gate>=0)&(gate<=1)).all() and (np.diff(gate)>=-1e-12).all()
            assert np.all(gate[consumed<=CA_LOW]==0) and np.all(gate[consumed>=CA_HIGH]==1)
            batch=replace(BATCHES[lab],initials={p:float(core[p].iloc[0]) for p in base.STATE_NAMES})
            oxygen0=CA_PARAMS['O2_initial_scale']*gas_cross.co2_model.o2_saturation_mg_l(
                base.temperature_at(batch,0.),batch.initials['E'],batch.initials['G'],batch.initials['F'],1.)
            ramp,multiplier=ca_pulse_history(time,batch.pulses['N'])
            old_ramp,old_multiplier=gas_model.pulse_history(time,batch.pulses['N'],GAS_PARAMS)
            assert np.array_equal(ramp,old_ramp)
            frames={}
            for variant in CA_VARIANTS:
                a=np.ones(len(time)) if variant=='sin_gate' else gate
                m=old_multiplier if variant=='causal_2_6_grid_max' else multiplier
                frame=gas_model.finish(batch,th,core,core0,CA_PARAMS,a,ramp,m,0.,oxygen0)
                frame['GF_consumed']=consumed
                assert np.array_equal(frame[list(base.STATE_NAMES)].to_numpy(),original[list(base.STATE_NAMES)].to_numpy())
                assert np.isfinite(frame.to_numpy()).all()
                assert (frame[['Cd','O2','qprod','qgas','predicted']].to_numpy()>=0).all()
                assert frame.attrs['mass_error']<1e-10
                assert frame.Cd.iloc[0]==0 and frame.predicted.iloc[0]==0
                assert frame.index[-1]==166
                if variant!='sin_gate': assert np.all(frame.qprod.to_numpy()[gate==0]==0)
                pulse_error=max(abs(frame.loc[t+diagnosis.CONFIG['after_event_h'],'N']-frame.loc[t,'N']-dose) for t,dose in batch.pulses['N'])
                assert pulse_error<1e-5
                frames[variant]=frame;CA_CURVES[(lab,theta,case,variant)]=frame
                CA_CHECK_ROWS.append(dict(lab=lab,theta=theta,case=case,variant=variant,start_h=0.,end_h=166.,
                    chemistry_max_difference_g_l=0.,pool_mass_error_g_l=frame.attrs['mass_error'],
                    pulse_sample_error_g_l=pulse_error,gate_min=float(a.min()),gate_max=float(a.max()),
                    multiplier_max_difference_from_grid_max=float(np.max(abs(multiplier-old_multiplier))),
                    Cd_initial_g_l=0.,O2_initial_mg_l=oxygen0))
            causal=frames['causal_2_6'];control=frames['sin_gate']
            assert np.allclose(causal.qprod,gate*control.qprod,rtol=1e-12,atol=1e-12)
            # A changed future sugar suffix cannot change the activation already evaluated.
            for cutoff in (12.,30.,60.,100.):
                n=int(np.searchsorted(time,cutoff,side='right'));prefix_time=time[:n]
                prefix_gate=ca_activation(gf[:n],gf0)
                changed=gf.copy();changed[n:]=gf0+100.
                assert np.array_equal(prefix_gate,ca_activation(changed,gf0)[:n])
                prefix_ramp,prefix_multiplier=ca_pulse_history(prefix_time,batch.pulses['N'])
                prefix=gas_model.finish(batch,th,core.iloc[:n],core0.iloc[:n],CA_PARAMS,
                    prefix_gate,prefix_ramp,prefix_multiplier,0.,oxygen0)
                columns=['gate','ramp','multiplier','O2','qprod','Cd','qgas','predicted']
                error=float(np.max(abs(prefix[columns].to_numpy()-causal[columns].iloc[:n].to_numpy())))
                assert error<1e-12,(lab,theta,case,cutoff,error)
                CA_PREFIX_ROWS.append(dict(lab=lab,theta=theta,case=case,cutoff_h=cutoff,max_prefix_error=error))
            source_reduction=float(np.sum(np.diff(time)*(control.qprod.to_numpy()[:-1]-causal.qprod.to_numpy()[:-1])))
            assert source_reduction>=-1e-10
            CA_FEATURE_ROWS.append(dict(lab=lab,theta=theta,case=case,
                activation_start_2g_h=ca_crossing(time,consumed,CA_LOW),
                activation_complete_6g_h=ca_crossing(time,consumed,CA_HIGH),
                observed_gas_confirmation_h=float(onset.start_h),
                predicted_gas_confirmation_h=ca_gas_confirmation(causal),
                control_gas_confirmation_h=ca_gas_confirmation(control),
                source_reduction_vs_a1_g_l=source_reduction,
                gas_max_difference_due_to_normalization_g_l_h=float(np.max(abs(causal.predicted-frames['causal_2_6_grid_max'].predicted)))))
        print('Gate causal y controles:',theta,lab,'; 0–166 h, con pulso',flush=True)
CA_CHECKS=pd.DataFrame(CA_CHECK_ROWS);CA_PREFIX_CHECKS=pd.DataFrame(CA_PREFIX_ROWS)
CA_FEATURES=pd.DataFrame(CA_FEATURE_ROWS)
assert len(CA_CURVES)==36 and len(CA_PREFIX_CHECKS)==48
display(CA_FEATURES.round(5))
display(CA_PREFIX_CHECKS.groupby('theta').max(numeric_only=True).round(14))

### 24.2. Figuras 1×3: gate causal y control sin gate

Hay una figura completa para cada theta. Verde: viable común; naranja: viable ponderada. Las curvas principales usan a(G+F); las curvas claras punteadas usan a=1 con la misma inicialización desde inoculación y el mismo pico analítico. Negro: caudal observado. Se mantienen los pulsos registrados. No se recorta el inicio: ahora interesa comprobar **cuándo se activa la predicción**.

Las curvas tenues del fondo muestran **G+F y etanol predichos**, para viable común y ponderada, con la misma theta que el CO₂. Se leen en el **eje derecho, en g/L**: G+F usa línea continua y etanol línea punteada. Verde corresponde a viable común y naranja a viable ponderada; el eje izquierdo conserva el caudal en g CO₂/L/h. Son las trayectorias químicas ya calculadas, idénticas con gate y con a=1, por lo que se dibujan una sola vez por biomasa. No se recalcula ni se ajusta la química.

La figura adicional de a(t) amplía sólo el intervalo de activación para leer los umbrales; todas las simulaciones y las figuras de caudal llegan hasta 166 h. Las líneas grises señalan la hora de confirmación observada y no intervienen en el cálculo.

Los errores usan la misma mediana aproximada de ocho predicciones interiores por bin que §23. Se informan el soporte válido completo, el período anterior y posterior a la detección observada y los períodos pre/post pulso. Para comparar contra §23 se usa únicamente el mismo soporte posterior; se advierte que cambia también la inicialización gaseosa. El último bin observado válido termina a 160 h y las simulaciones continúan hasta 166 h.

In [ ]:
for (lab,theta,case,variant),curve in CA_CURVES.items():
    onset=GAS_ONSETS[GAS_ONSETS.lab.eq(lab)].iloc[0];detected=float(onset.start_h)
    pulse=min(t for t,d in BATCHES[lab].pulses['N'])
    observed=GAS_HOURLY[GAS_HOURLY.lab.eq(lab)&GAS_HOURLY.valid].copy()
    observed['predicted_g_l_h']=ca_bin_prediction(curve,observed.bin_start_h)
    observed['residual_g_l_h']=observed.predicted_g_l_h-observed.co2_observed_g_l_h
    observed['theta']=theta;observed['case']=case;observed['variant']=variant
    CA_RESIDUAL_PARTS.append(observed)
    windows=[('completo',np.ones(len(observed),bool)),
        ('anterior_detectado',observed.bin_end_h<=detected),('posterior_detectado',observed.bin_start_h>=detected),
        ('pre_pulso',observed.bin_end_h<=pulse),('post_pulso',observed.bin_start_h>=pulse)]
    for window,keep in windows:
        part=observed.loc[keep]
        if part.empty: continue
        r=part.residual_g_l_h.to_numpy();q=part[part.quantifiable].residual_g_l_h.to_numpy()
        CA_METRIC_ROWS.append(dict(lab=lab,theta=theta,case=case,variant=variant,window=window,n=len(part),
            RMSE_g_l_h=float(np.sqrt(np.mean(r*r))),bias_g_l_h=float(r.mean()),MAE_g_l_h=float(np.mean(abs(r))),
            n_quantifiable=len(q),RMSE_quantifiable_g_l_h=float(np.sqrt(np.mean(q*q))) if len(q) else np.nan,
            first_bin_start_h=float(part.bin_start_h.min()),last_bin_end_h=float(part.bin_end_h.max())))
CA_RESIDUALS=pd.concat(CA_RESIDUAL_PARTS,ignore_index=True);CA_METRICS=pd.DataFrame(CA_METRIC_ROWS)
control=CA_METRICS[CA_METRICS.variant.eq('sin_gate')][['lab','theta','case','window','RMSE_g_l_h']].rename(columns={'RMSE_g_l_h':'RMSE_control_a1_g_l_h'})
CA_METRICS=CA_METRICS.merge(control,on=['lab','theta','case','window'],validate='many_to_one')
CA_METRICS['ratio_to_control_a1']=CA_METRICS.RMSE_g_l_h/CA_METRICS.RMSE_control_a1_g_l_h

for theta in THETA_SETS:
    fig,axes=plt.subplots(1,3,figsize=(18,5.2),layout='constrained')
    chemical_axes=[]
    for ax,lab in zip(axes,LABS):
        chem=ax.twinx();chemical_axes.append(chem)
        chem.set_zorder(1);ax.set_zorder(2);ax.patch.set_visible(False)
        chem.grid(False)
        for case in CA_CASES:
            label,color,style=W_CASES[case]
            chemical=CA_CURVES[(lab,theta,case,'causal_2_6')]
            short_label='común' if case=='viable_comun_volumen' else 'ponderada'
            chem.plot(chemical.index,chemical.G+chemical.F,color=color,ls='-',lw=1.3,alpha=.30,
                label='G+F · '+short_label)
            chem.plot(chemical.index,chemical.E,color=color,ls=':',lw=1.5,alpha=.35,
                label='Etanol · '+short_label)
            control=CA_CURVES[(lab,theta,case,'sin_gate')]
            causal=CA_CURVES[(lab,theta,case,'causal_2_6')]
            ax.plot(control.index,control.predicted,color=color,ls=':',lw=1.2,alpha=.45,label=label+' · a=1')
            ax.plot(causal.index,causal.predicted,color=color,ls=style,lw=1.7,label=label+' · causal')
        chem.set(ylabel='G+F / etanol [g/L]',ylim=(0,180))
        chem.tick_params(axis='y',colors='#777777',labelsize=8)
        chem.yaxis.label.set_color('#777777');chem.spines['right'].set_color('#aaaaaa')
        observed=GAS_HOURLY[GAS_HOURLY.lab.eq(lab)&GAS_HOURLY.valid]
        ax.plot(observed.plot_t_h,observed.co2_observed_g_l_h,'o',color='black',ms=2.5,label='CO₂ observado',zorder=6)
        mark_events(ax,lab)
        ax.set(title=lab,xlabel='t desde inoculación [h]',ylabel='g CO₂/L/h',xlim=(0,166),ylim=(0,None))
    fig.suptitle('Activación por G+F predicho · 2–6 g/L · '+theta,y=.995)
    fig.get_layout_engine().set(rect=(0,0,1,.79))
    handles,labels=axes[0].get_legend_handles_labels()
    chem_handles,chem_labels=chemical_axes[0].get_legend_handles_labels()
    fig.legend(handles+chem_handles,labels+chem_labels,loc='upper center',bbox_to_anchor=(.5,.955),ncol=5,fontsize=8)
    ca_save(fig,'01_co2_causal_vs_a1_'+theta)
fig,axes=plt.subplots(1,3,figsize=(16,4.3),layout='constrained')
zoom_end=min(166.,max(45.,float(np.ceil(CA_FEATURES.activation_complete_6g_h.max()/5.)*5.+5.)))
for ax,lab in zip(axes,LABS):
    for theta in THETA_SETS:
        for case in CA_CASES:
            curve=CA_CURVES[(lab,theta,case,'causal_2_6')];label,color,_=W_CASES[case]
            ax.plot(curve.index,curve.gate,color=color,ls='-' if theta=='theta_full' else '--',lw=1.5,label=label+' · '+theta)
    detected=float(GAS_ONSETS[GAS_ONSETS.lab.eq(lab)].start_h.iloc[0])
    ax.axvline(detected,color='gray',ls=':',lw=1.,label='CO₂ detectado')
    ax.set(title=lab,xlabel='t desde inoculación [h]',ylabel='a(t)',xlim=(0,zoom_end),ylim=(-.03,1.03))
fig.suptitle('Activación calculada · detalle del inicio · umbrales 2–6 g/L',y=.995)
fig.get_layout_engine().set(rect=(0,0,1,.79))
handles,labels=axes[0].get_legend_handles_labels()
fig.legend(handles,labels,loc='upper center',bbox_to_anchor=(.5,.955),ncol=3,fontsize=8)
ca_save(fig,'02_activacion_por_estado')


### 24.2.1. Activación completa al consumir 2 g/L frente a 6 g/L

Esta figura adicional queda inmediatamente debajo del detalle de activación anterior. Ahora se comparan **dos reglas independientes de encendido**, para LAB019–021, ambas theta y ambas biomasas:
\[
c(t)=G_0+F_0-[\hat G(t)+\hat F(t)],\qquad
a_2(t)=\begin{cases}0&c(t)<2\\1&c(t)\ge2\end{cases},\qquad
a_6(t)=\begin{cases}0&c(t)<6\\1&c(t)\ge6\end{cases}.
\]

No hay transición gradual entre 2 y 6: cada curva pasa directamente de 0 a 1 al alcanzar **su propio umbral**. El consumo es el predicho por la química actual y no usa mediciones futuras. Se evalúa el umbral en cada nodo de la malla química existente (paso habitual 0,25 h); la tabla informa el primer nodo que lo cumple, sin interpolar un cruce futuro para activar antes. En estas trayectorias el consumo es monótono. Para un EKF con correcciones no monótonas habría que especificar persistencia o histéresis por separado.

**Filas 1–2:** a(t) para theta_full y theta_defaults, con detalle 0–45 h. **Filas 3–4:** el caudal correspondiente para theta_full y theta_defaults, hasta **166 h y con todos los pulsos**. Columnas: LAB019, LAB020 y LAB021. Verde: viable común; naranja: viable ponderada. Línea continua: encendido a 2 g/L; discontinua: encendido a 6 g/L. La confirmación de CO₂ observado es sólo una referencia en los paneles de activación.

Activar completamente la **fuente** no obliga al **caudal** a saltar: siguen interviniendo O₂, el reservorio disuelto y la ecuación de liberación. Se conservan los parámetros numéricos antiguos, el kernel analítico, Cd(0)=0, las condiciones iniciales y toda la química. Sólo se cambia a(t). Las cuatro curvas de caudal pueden superponerse por parejas después del arranque. Como en §24, queda pendiente la consistencia estequiométrica de aplicar este encendido únicamente a la fuente gaseosa.

In [ ]:
HARD_OUT=OUT/'co2_hard_sugar_activation';HARD_OUT.mkdir(exist_ok=True)
HARD_PRIOR_FILES=[p for p in OUT.rglob('*') if p.is_file() and HARD_OUT not in p.parents]
HARD_PRIOR_HASHES={str(p.relative_to(ROOT)):sha256(p) for p in HARD_PRIOR_FILES}
HARD_THRESHOLDS=(2.,6.);HARD_CURVES={};HARD_FEATURE_ROWS=[];HARD_CHECK_ROWS=[];HARD_PREFIX_ROWS=[]
HARD_RESIDUAL_PARTS=[];HARD_FIGURES=[]

def hard_activation(consumed,threshold):
    return (np.asarray(consumed,float)>=threshold).astype(float)

for theta,th in THETA_SETS.items():
    no_n_map=GAS_NO_N_CORES if theta=='theta_full' else GAS_DEFAULT_NO_N_CORES
    for lab in LABS:
        observed=GAS_HOURLY[GAS_HOURLY.lab.eq(lab)&GAS_HOURLY.valid].sort_values('bin_start_h').copy()
        detected=float(GAS_ONSETS[GAS_ONSETS.lab.eq(lab)].start_h.iloc[0])
        for case in CA_CASES:
            original=W_CURVES[(lab,theta,case)]
            core=original[list(base.STATE_NAMES)].copy();core0=no_n_map[(lab,case)].copy()
            t=core.index.to_numpy(float);consumed=float(core.G.iloc[0]+core.F.iloc[0])-(core.G+core.F).to_numpy()
            assert np.array_equal(core.index,core0.index) and t[0]==0 and t[-1]==166
            assert np.all(np.diff(consumed)>=-1e-10)
            batch=replace(BATCHES[lab],initials={p:float(core[p].iloc[0]) for p in base.STATE_NAMES})
            oxygen0=CA_PARAMS['O2_initial_scale']*gas_cross.co2_model.o2_saturation_mg_l(
                base.temperature_at(batch,0.),batch.initials['E'],batch.initials['G'],batch.initials['F'],1.)
            ramp,multiplier=ca_pulse_history(t,batch.pulses['N'])
            control=CA_CURVES[(lab,theta,case,'sin_gate')]
            for threshold in HARD_THRESHOLDS:
                gate=hard_activation(consumed,threshold)
                curve=gas_model.finish(batch,th,core,core0,CA_PARAMS,gate,ramp,multiplier,0.,oxygen0)
                curve['GF_consumed']=consumed
                assert set(np.unique(gate))=={0.,1.} and (np.diff(gate)>=0).all()
                first=int(np.flatnonzero(gate)[0]);assert consumed[first]>=threshold and consumed[first-1]<threshold
                assert np.array_equal(curve[list(base.STATE_NAMES)].to_numpy(),original[list(base.STATE_NAMES)].to_numpy())
                assert np.isfinite(curve.to_numpy()).all() and curve.attrs['mass_error']<1e-10
                assert (curve[['Cd','O2','qprod','qgas','predicted']].to_numpy()>=0).all()
                assert np.allclose(curve.qprod,gate*control.qprod,rtol=1e-12,atol=1e-12)
                pulse_error=max(abs(curve.loc[event+diagnosis.CONFIG['after_event_h'],'N']-curve.loc[event,'N']-dose)
                    for event,dose in batch.pulses['N'])
                assert pulse_error<1e-5
                HARD_CURVES[(lab,theta,case,threshold)]=curve
                HARD_CHECK_ROWS.append(dict(lab=lab,theta=theta,case=case,threshold_g_l=threshold,
                    pool_mass_error_g_l=curve.attrs['mass_error'],chemistry_max_difference_g_l=0.,
                    pulse_error_g_l=pulse_error,start_h=0.,end_h=166.))
                for cutoff in (12.,30.,60.,100.):
                    n=int(np.searchsorted(t,cutoff,side='right'))
                    changed=consumed.copy();changed[n:]=-100.
                    prefix_gate=hard_activation(consumed[:n],threshold)
                    assert np.array_equal(prefix_gate,hard_activation(changed,threshold)[:n])
                    pr,pm=ca_pulse_history(t[:n],batch.pulses['N'])
                    prefix=gas_model.finish(batch,th,core.iloc[:n],core0.iloc[:n],CA_PARAMS,prefix_gate,pr,pm,0.,oxygen0)
                    cols=['gate','ramp','multiplier','O2','qprod','Cd','qgas','predicted']
                    error=float(np.max(abs(prefix[cols].to_numpy()-curve[cols].iloc[:n].to_numpy())))
                    assert error<1e-12
                    HARD_PREFIX_ROWS.append(dict(lab=lab,theta=theta,case=case,threshold_g_l=threshold,cutoff_h=cutoff,max_prefix_error=error))
                binned=ca_bin_prediction(curve,observed.bin_start_h)
                residual=binned-observed.co2_observed_g_l_h.to_numpy()
                peak=int(np.argmax(binned));onset=ca_gas_confirmation(curve)
                HARD_FEATURE_ROWS.append(dict(lab=lab,theta=theta,case=case,threshold_g_l=threshold,
                    full_activation_grid_h=float(t[first]),previous_grid_consumption_g_l=float(consumed[first-1]),
                    activation_grid_consumption_g_l=float(consumed[first]),grid_bracket_width_h=float(t[first]-t[first-1]),
                    observed_onset_h=detected,predicted_onset_h=onset,onset_error_h=onset-detected,
                    peak_g_l_h=float(binned[peak]),peak_h=float(observed.plot_t_h.iloc[peak]),
                    n=len(residual),RMSE_g_l_h=float(np.sqrt(np.mean(residual**2))),bias_g_l_h=float(residual.mean())))
                part=observed.copy();part['predicted_g_l_h']=binned;part['residual_g_l_h']=residual
                part['theta']=theta;part['case']=case;part['threshold_g_l']=threshold
                HARD_RESIDUAL_PARTS.append(part)
            early=HARD_CURVES[(lab,theta,case,2.)];late=HARD_CURVES[(lab,theta,case,6.)]
            assert (late.gate<=early.gate).all() and (late.predicted<=early.predicted+1e-10).all()
        print('Encendidos completos a 2 y 6 g/L:',theta,lab,flush=True)
HARD_FEATURES=pd.DataFrame(HARD_FEATURE_ROWS);HARD_CHECKS=pd.DataFrame(HARD_CHECK_ROWS)
HARD_PREFIX_CHECKS=pd.DataFrame(HARD_PREFIX_ROWS);HARD_RESIDUALS=pd.concat(HARD_RESIDUAL_PARTS,ignore_index=True)
assert len(HARD_CURVES)==24 and len(HARD_PREFIX_CHECKS)==96

fig,axes=plt.subplots(4,3,figsize=(16,14.8),layout='constrained')
for ti,theta in enumerate(THETA_SETS):
    for j,lab in enumerate(LABS):
        activation_ax=axes[ti,j];gas_ax=axes[ti+2,j]
        for case in CA_CASES:
            color=W_CASES[case][1];label='Común' if case=='viable_comun_volumen' else 'Ponderada'
            for threshold in HARD_THRESHOLDS:
                curve=HARD_CURVES[(lab,theta,case,threshold)]
                style='-' if threshold==2 else '--';legend=label+f' · {threshold:g} g/L'
                activation_ax.step(curve.index,curve.gate,where='post',color=color,ls=style,lw=1.6,label=legend)
                gas_ax.plot(curve.index,curve.predicted,color=color,ls=style,lw=1.6,label=legend)
        detected=float(GAS_ONSETS[GAS_ONSETS.lab.eq(lab)].start_h.iloc[0])
        activation_ax.axvline(detected,color='gray',ls=':',lw=1.,label='CO₂ detectado')
        activation_ax.set(title=lab+' · activación · '+theta,xlabel='t desde inoculación [h]',ylabel='a(t)',xlim=(0,45),ylim=(-.03,1.03))
        obs=GAS_HOURLY[GAS_HOURLY.lab.eq(lab)&GAS_HOURLY.valid]
        gas_ax.plot(obs.plot_t_h,obs.co2_observed_g_l_h,'o',color='black',ms=2.5,label='CO₂ observado',zorder=6)
        mark_events(gas_ax,lab)
        gas_ax.set(title=lab+' · CO₂ · '+theta,xlabel='t desde inoculación [h]',ylabel='g CO₂/L/h',xlim=(0,166),ylim=(0,None))
fig.suptitle('Activación completa al consumir 2 g/L frente a 6 g/L · ambas biomasas',y=.995)
fig.get_layout_engine().set(rect=(0,0,1,.90))
handles,labels=axes[0,0].get_legend_handles_labels();gh,gl=axes[2,0].get_legend_handles_labels()
unique=dict(zip(labels,handles))
for handle,label in zip(gh,gl):unique.setdefault(label,handle)
fig.legend(unique.values(),unique.keys(),loc='upper center',bbox_to_anchor=(.5,.973),ncol=4,fontsize=8)
fig.canvas.draw();renderer=fig.canvas.get_renderer();legend_box=fig.legends[0].get_window_extent(renderer)
assert not fig._suptitle.get_window_extent(renderer).overlaps(legend_box)
assert all(not ax.get_tightbbox(renderer).overlaps(legend_box) for ax in axes.flat)
figure_path=HARD_OUT/'01_activacion_completa_2_vs_6.png'
fig.savefig(figure_path,dpi=140,bbox_inches='tight',facecolor='white')
buffer=io.BytesIO();fig.savefig(buffer,format='png',dpi=110,bbox_inches='tight',facecolor='white')
plt.close(fig);display(Image(data=buffer.getvalue()));buffer.close()
HARD_FIGURES.append(dict(name=figure_path.stem,path=str(figure_path.relative_to(ROOT))))
display(Markdown('**Tiempos calculados.** El encendido es el salto de a(t); la confirmación gaseosa '
    'requiere tres bins horarios sobre 0,05 g/L/h. No son el mismo instante.'))
display(HARD_FEATURES[['lab','theta','case','threshold_g_l','full_activation_grid_h',
    'predicted_onset_h','observed_onset_h','onset_error_h','RMSE_g_l_h']].round(5))
sentences=[]
for lab in LABS:
    text=[]
    for theta in THETA_SETS:
        for case in CA_CASES:
            sub=HARD_FEATURES[(HARD_FEATURES.lab==lab)&(HARD_FEATURES.theta==theta)&(HARD_FEATURES.case==case)].set_index('threshold_g_l')
            text.append(f'{theta}, {W_CASES[case][0]}: confirmación a {sub.loc[2.,"predicted_onset_h"]:.0f} h '
                f'(2 g/L) y {sub.loc[6.,"predicted_onset_h"]:.0f} h (6 g/L), '
                f'frente a {sub.loc[2.,"observed_onset_h"]:.0f} h observadas')
    sentences.append('**'+lab+':** '+'; '.join(text)+'.')
sentences.append('El encendido a 6 g/L retrasa la fuente respecto de 2 g/L bajo la misma química. '
    'No desplaza la química ni reinicia el reservorio; las diferencias de caudal pueden desaparecer tras el arranque. '
    'Esta prueba no modifica N, GF ni E y no determina un umbral óptimo ni valida el modelo físico completo.')
display(Markdown('\n\n'.join(sentences)))
assert HARD_PRIOR_HASHES=={str(p.relative_to(ROOT)):sha256(p) for p in HARD_PRIOR_FILES}
assert GAS_INPUT_HASHES=={str(p.relative_to(ROOT)):sha256(p) for p in GAS_INPUT_FILES}
for name,table in dict(features=HARD_FEATURES,residuals=HARD_RESIDUALS,checks=HARD_CHECKS,
    prefix_checks=HARD_PREFIX_CHECKS,figure_index=pd.DataFrame(HARD_FIGURES)).items():
    table.to_csv(HARD_OUT/(name+'.csv'),index=False)
arrays={};array_manifest=[]
for i,(key,curve) in enumerate(HARD_CURVES.items()):
    name=f'case_{i}';arrays[name]=np.column_stack([curve.index,curve.to_numpy()])
    array_manifest.append(dict(array=name,key=list(key),columns=['t_h']+list(curve.columns)))
np.savez_compressed(HARD_OUT/'trajectories.npz',**arrays)
manifest=dict(created_at=datetime.now().astimezone().isoformat(),python=sys.executable,
    source_code_sha256=sha256(LAB2026/'notebooks/diagnostics/nitrogen_consumption_diagnosis.source.ipynb'),
    activation='pointwise hard switch: 0 below threshold; 1 at or above threshold',thresholds_g_l=list(HARD_THRESHOLDS),
    crossing_policy='first existing grid node at or above threshold; no anticipatory interpolation',
    theta=THETA_SETS,co2_parameters=CA_PARAMS,input_hashes=GAS_INPUT_HASHES,prior_result_hashes=HARD_PRIOR_HASHES,
    simulation_start_h=0.,simulation_end_h=166.,initial_dissolved_CO2_g_l=0.,
    chemical_trajectories_unchanged=True,activation_uses_observed_CO2=False,
    arrays=array_manifest,curves=len(HARD_CURVES),prefix_checks=len(HARD_PREFIX_CHECKS),figures=HARD_FIGURES,
    optimizer_calls=0,kalman_updates=0,whole_process_online_validation=False,
    physical_scope='gas source activation only; chemical consistency remains unresolved')
(HARD_OUT/'manifest.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding='utf-8')
print('Verificados 24 escenarios y 96 prefijos; figura añadida debajo de la activación suave. Sin commit ni push.')

In [ ]:
prior=pd.concat([GAS_METRICS,GAS_DEFAULT_METRICS],ignore_index=True)
prior=prior[prior.window.eq('completo')&prior.case.isin(CA_CASES)][['lab','theta','case','n','RMSE_g_l_h']].rename(columns={
    'n':'n_previous','RMSE_g_l_h':'RMSE_previous_detected_start_g_l_h'})
CA_PREVIOUS_COMPARISON=CA_METRICS.query('variant=="causal_2_6" and window=="posterior_detectado"').merge(prior,on=['lab','theta','case'],validate='one_to_one')
assert CA_PREVIOUS_COMPARISON.n.eq(CA_PREVIOUS_COMPARISON.n_previous).all()
CA_PREVIOUS_COMPARISON['ratio_to_previous_detected_start']=CA_PREVIOUS_COMPARISON.RMSE_g_l_h/CA_PREVIOUS_COMPARISON.RMSE_previous_detected_start_g_l_h
display(CA_METRICS.query('variant in ["causal_2_6","sin_gate"] and window in ["completo","posterior_detectado"]')[
    ['lab','theta','case','variant','window','n','RMSE_g_l_h','bias_g_l_h','ratio_to_control_a1']].round(5))
display(CA_PREVIOUS_COMPARISON[['lab','theta','case','n','RMSE_g_l_h','RMSE_previous_detected_start_g_l_h','ratio_to_previous_detected_start']].round(5))

### 24.3. Resultados calculados, alcance y trazabilidad

El gate causal puede activarse antes que el caudal observado porque depende del consumo que predice la cinética; después todavía intervienen O₂ y el pool disuelto. Una formulación causal puede producir una predicción incorrecta. Las siguientes conclusiones distinguen el efecto del gate frente a su control y el desfase de inicio gaseoso. El período posterior no basta para validar una mejora del arranque: también se revisa el soporte completo.

No se han cambiado N, GF ni E ni se ha solucionado automáticamente su discrepancia. La prueba mantiene deliberadamente la fuente gaseosa como capa separada; antes de usarla como modelo físico completo habría que revisar el acoplamiento estequiométrico y, si representa actividad fermentativa, hacer coherentes sus efectos sobre azúcar, etanol y CO₂.

In [ ]:
CA_CONCLUSIONS=[]
for theta in THETA_SETS:
    CA_CONCLUSIONS.append('**'+theta+':**')
    for lab in LABS:
        details=[]
        for case in CA_CASES:
            feature=CA_FEATURES.query('lab==@lab and theta==@theta and case==@case').iloc[0]
            row=CA_METRICS.query('lab==@lab and theta==@theta and case==@case and variant=="causal_2_6" and window=="completo"').iloc[0]
            direction='mejora' if row.ratio_to_control_a1<1 else 'empeora'
            details.append(f'{W_CASES[case][0]}: a empieza a {feature.activation_start_2g_h:.2f} h y llega a 1 a {feature.activation_complete_6g_h:.2f} h; '
                f'CO₂ confirmado predicho {feature.predicted_gas_confirmation_h:.0f} h frente a observado {feature.observed_gas_confirmation_h:.0f} h; '
                f'RMSE {row.RMSE_g_l_h:.4f} frente a a=1 {row.RMSE_control_a1_g_l_h:.4f} g/L/h ({direction})')
        CA_CONCLUSIONS.append('- '+lab+': '+'; '.join(details)+'.')
CA_CONCLUSIONS.append(f'Comprobación por prefijos: error máximo {CA_PREFIX_CHECKS.max_prefix_error.max():.3g}. '
    f'Cambiar grid_max por el pico analítico con el mismo gate produce una diferencia máxima de caudal de '
    f'{CA_FEATURES.gas_max_difference_due_to_normalization_g_l_h.max():.6g} g/L/h en estos ensayos. '
    'Esa auditoría separa el pequeño o grande efecto numérico de la normalización del efecto de a(t).')
CA_CONCLUSIONS.append('No se eligen umbrales óptimos ni se declara validación online. La regla a(G+F) y el kernel analítico '
    'no necesitan el sufijo futuro del estado, pero el ensayo usa temperatura medida retrospectiva y cinética congelada. '
    'Una coincidencia del inicio gaseoso no demuestra que la biomasa, N o la actividad estén correctamente estimados.')
display(Markdown('\n\n'.join(CA_CONCLUSIONS)))
assert CA_PRIOR_HASHES=={str(p.relative_to(ROOT)):sha256(p) for p in CA_PRIOR_FILES}
assert GAS_INPUT_HASHES=={str(p.relative_to(ROOT)):sha256(p) for p in GAS_INPUT_FILES}
CA_TABLES=dict(metrics=CA_METRICS,residuals=CA_RESIDUALS,checks=CA_CHECKS,
    prefix_checks=CA_PREFIX_CHECKS,activation_features=CA_FEATURES,
    comparison_detected_start=CA_PREVIOUS_COMPARISON,figure_index=pd.DataFrame(CA_FIGURES))
for name,table in CA_TABLES.items(): table.to_csv(CA_OUT/(name+'.csv'),index=False)
arrays={};array_manifest=[]
for i,(key,curve) in enumerate(CA_CURVES.items()):
    name=f'case_{i}';arrays[name]=np.column_stack([curve.index,curve.to_numpy()])
    array_manifest.append(dict(array=name,key=list(key),columns=['t_h']+list(curve.columns)))
np.savez_compressed(CA_OUT/'trajectories.npz',**arrays)
CA_MANIFEST=dict(created_at=datetime.now().astimezone().isoformat(),python=sys.executable,
    source_code_sha256=sha256(LAB2026/'notebooks/diagnostics/nitrogen_consumption_diagnosis.source.ipynb'),
    input_hashes=GAS_INPUT_HASHES,prior_result_hashes=CA_PRIOR_HASHES,
    theta=THETA_SETS,numeric_CO2_parameters_unchanged=True,co2_parameters=CA_PARAMS,
    previous_pulse_normalization=GAS_PARAMS['kernel_normalization'],
    activation='pointwise smoothstep of predicted GF0-GF(t); fixed 2 and 6 g/L thresholds',
    activation_thresholds_g_l=[CA_LOW,CA_HIGH],activation_uses_observed_CO2=False,
    initial_dissolved_CO2_g_l=0.,gas_initialized_from_measurement=False,
    chemical_trajectories_unchanged=True,variants=list(CA_VARIANTS),arrays=array_manifest,
    curves=len(CA_CURVES),prefix_checks=len(CA_PREFIX_CHECKS),figures=CA_FIGURES,
    simulation_start_h=0.,simulation_end_h=166.,optimizer_calls=0,kalman_updates=0,
    whole_process_causal_validation=False,temperature='retrospective measured input',
    physical_scope='CO2-layer activation diagnostic; unchanged chemistry; CO2/ethanol source consistency unresolved',
    observation_policy=GF_ZERO_POLICY)
(CA_OUT/'manifest.json').write_text(json.dumps(CA_MANIFEST,ensure_ascii=False,indent=2),encoding='utf-8')
print('Verificado: 36 curvas, 48 controles por prefijos, 3 figuras 1×3, química y resultados previos intactos.')
print('SOURCE limpio; EXECUTED con resultados. Sin reajuste, EKF, commit ni push.')

## 25. Comparación adicional de umbrales: 2–6, 5–10 y 8–15 g/L

Esta comparación se añade **debajo de las figuras anteriores, que se conservan**. Se prueban las tres parejas solicitadas en LAB019–021, con theta_full/theta_defaults y biomasa viable común/ponderada: **36 curvas**. Ningún umbral se ajusta individualmente a un LAB.

La regla mantiene la forma de §24, reemplazando solamente sus límites:
\[
c(t)=G_0+F_0-[\hat G(t)+\hat F(t)],\quad
z=\operatorname{clip}\!\left(\frac{c(t)-c_{\rm inicio}}{c_{\rm completo}-c_{\rm inicio}},0,1\right),\quad
a(t)=z^2(3-2z).
\]

Se conservan las mismas trayectorias químicas, nutriciones, parámetros numéricos antiguos de CO₂, Cd(0)=0, O₂ inicial y normalización analítica del transitorio de §24. Se simula **desde inoculación hasta 166 h**, sin reiniciar con la señal observada ni desplazar las muestras. El caudalímetro se usa para evaluar las predicciones; no para activar la fuente. Los umbrales no son gramos de CO₂: son **g/L de azúcar consumida predicha**.

El gate sigue afectando solamente a la capa gaseosa. Aumentarlo puede retrasar o suprimir parte de la fuente sin mejorar la química; no garantiza consistencia entre producción de etanol y CO₂. Las parejas varían tanto el comienzo como el ancho de la transición: se comparan como escenarios, sin atribuir el cambio a un único límite.

In [ ]:
TS_OUT=OUT/'co2_activation_threshold_comparison';TS_OUT.mkdir(exist_ok=True)
TS_THRESHOLDS={'2_6':(2.,6.),'5_10':(5.,10.),'8_15':(8.,15.)}
TS_COLORS={'2_6':'#2166ac','5_10':'#00897b','8_15':'#e66101'}
TS_STYLES={'2_6':'-','5_10':'--','8_15':'-.'}
TS_LABELS={key:f'{low:g}–{high:g} g/L' for key,(low,high) in TS_THRESHOLDS.items()}
TS_PRIOR_FILES=[p for p in OUT.rglob('*') if p.suffix in ('.csv','.npz','.png','.json') and TS_OUT not in p.parents]
TS_PRIOR_HASHES={str(p.relative_to(ROOT)):sha256(p) for p in TS_PRIOR_FILES}
TS_CURVES={};TS_CHECK_ROWS=[];TS_PREFIX_ROWS=[];TS_FEATURE_ROWS=[]
TS_METRIC_ROWS=[];TS_RESIDUAL_PARTS=[];TS_FIGURES=[];TS_OBS_ROWS=[]

def ts_gate(consumed,low,high):
    assert high>low>=0
    z=np.clip((np.asarray(consumed,float)-low)/(high-low),0.,1.)
    return z*z*(3.-2.*z)

def ts_onset(starts,ends,values):
    """First confirmation using consecutive, valid, hourly bins."""
    starts=np.asarray(starts,float);ends=np.asarray(ends,float);values=np.asarray(values,float)
    for i in range(len(values)-GAS_CONSECUTIVE+1):
        j=i+GAS_CONSECUTIVE
        if np.allclose(np.diff(starts[i:j]),1.) and (values[i:j]>=GAS_THRESHOLD).all():
            if ends[j-1]>=GAS_ZERO_AVAILABLE_H: return float(ends[j-1])
    return np.nan

def ts_peak_fall(observed,values):
    """Peak and final confirmed fall on the same observed support; no extrapolation."""
    values=np.asarray(values,float);starts=observed.bin_start_h.to_numpy(float)
    ends=observed.bin_end_h.to_numpy(float);times=observed.plot_t_h.to_numpy(float)
    peak=int(np.argmax(values));fall=np.nan
    for i in range(peak+1,len(values)-GAS_CONSECUTIVE+1):
        j=i+GAS_CONSECUTIVE
        if np.allclose(np.diff(starts[i:j]),1.) and (values[i:]<GAS_THRESHOLD).all():
            fall=float(ends[j-1]);break
    return dict(peak_h=float(times[peak]),peak_g_l_h=float(values[peak]),
                fall_confirmation_h=fall,fall_not_confirmed=bool(np.isnan(fall)),
                last_bin_end_h=float(ends[-1]))

def ts_save(fig,name):
    fig.canvas.draw();renderer=fig.canvas.get_renderer()
    title_box=fig._suptitle.get_window_extent(renderer)
    assert all(not title_box.overlaps(l.get_window_extent(renderer)) for l in fig.legends)
    for ax in fig.axes:
        assert all(not ax.get_tightbbox(renderer).overlaps(l.get_window_extent(renderer)) for l in fig.legends)
    path=TS_OUT/(name+'.png');fig.savefig(path,dpi=140,bbox_inches='tight',facecolor='white')
    buffer=io.BytesIO();fig.savefig(buffer,format='png',dpi=110,bbox_inches='tight',facecolor='white')
    plt.close(fig);display(Image(data=buffer.getvalue()));buffer.close()
    TS_FIGURES.append(dict(name=name,path=str(path.relative_to(ROOT))))

for lab in LABS:
    obs=GAS_HOURLY[GAS_HOURLY.lab.eq(lab)&GAS_HOURLY.valid].sort_values('bin_start_h')
    feature=ts_peak_fall(obs,obs.co2_observed_g_l_h)
    onset=ts_onset(obs.bin_start_h,obs.bin_end_h,obs.co2_observed_g_l_h)
    assert onset==float(GAS_ONSETS[GAS_ONSETS.lab.eq(lab)].start_h.iloc[0])
    TS_OBS_ROWS.append(dict(lab=lab,onset_confirmation_h=onset,n=len(obs),**feature))
TS_OBS_FEATURES=pd.DataFrame(TS_OBS_ROWS).set_index('lab')
display(TS_OBS_FEATURES.round(4))

### 25.1. Qué se compara y cómo se comprueba

Los tiempos y máximos se calculan sobre **el mismo soporte de mediciones válidas**, con la misma aproximación de mediana horaria de §24. No se confunde el primer CO₂ producido con su detección:

- **Inicio:** confirmación al terminar tres bins consecutivos con caudal ≥0,05 g/L/h. Se informa predicho menos observado; negativo significa adelanto.
- **Máximo:** mayor mediana horaria y su hora, en todo el soporte válido. No se suaviza adicionalmente ni se ajusta una curva al máximo observado. El panel correspondiente amplía ±20 h alrededor del máximo observado.
- **Caída final:** confirmación de tres bins consecutivos por debajo de 0,05 g/L/h después del máximo de cada curva, sin nuevos cruces por encima del límite en los datos válidos posteriores. Es un diagnóstico retrospectivo de la caída definitiva, no una regla online. Si no se confirma antes del fin del soporte observado (160 h), se indica «no confirmada hasta 160 h»; no se extrapolan mediciones.
- **Errores:** RMSE y sesgo del soporte completo, del período anterior a la confirmación observada, del entorno ±20 h del máximo observado y de la caída desde ese máximo hasta el final observado. Son ventanas descriptivas que pueden solaparse.

Se comprueban identidad de la química, balances del pool, pulsos, reproducción del caso 2–6 de §24 y causalidad por prefijos a 12, 30, 60 y 100 h para cada escenario. El mayor umbral debe producir una activación y un caudal no mayores que los de los umbrales menores bajo esta capa y estas entradas iguales. La temperatura sigue siendo la medida retrospectiva; no se afirma validación online ni se ejecuta un EKF.

In [ ]:
for theta,th in THETA_SETS.items():
    no_n_map=GAS_NO_N_CORES if theta=='theta_full' else GAS_DEFAULT_NO_N_CORES
    for lab in LABS:
        obs=GAS_HOURLY[GAS_HOURLY.lab.eq(lab)&GAS_HOURLY.valid].sort_values('bin_start_h').copy()
        reference=TS_OBS_FEATURES.loc[lab]
        for case in CA_CASES:
            original=W_CURVES[(lab,theta,case)]
            core=original[list(base.STATE_NAMES)].copy();core0=no_n_map[(lab,case)].copy()
            time=core.index.to_numpy(float);consumed=float(core.G.iloc[0]+core.F.iloc[0])-(core.G+core.F).to_numpy()
            assert np.array_equal(core.index,core0.index) and time[0]==0 and time[-1]==166
            batch=replace(BATCHES[lab],initials={p:float(core[p].iloc[0]) for p in base.STATE_NAMES})
            oxygen0=CA_PARAMS['O2_initial_scale']*gas_cross.co2_model.o2_saturation_mg_l(
                base.temperature_at(batch,0.),batch.initials['E'],batch.initials['G'],batch.initials['F'],1.)
            ramp,multiplier=ca_pulse_history(time,batch.pulses['N'])
            previous=None
            for threshold,(low,high) in TS_THRESHOLDS.items():
                gate=ts_gate(consumed,low,high)
                curve=gas_model.finish(batch,th,core,core0,CA_PARAMS,gate,ramp,multiplier,0.,oxygen0)
                curve['GF_consumed']=consumed
                assert np.isfinite(curve.to_numpy()).all() and curve.attrs['mass_error']<1e-10
                assert np.array_equal(curve[list(base.STATE_NAMES)].to_numpy(),original[list(base.STATE_NAMES)].to_numpy())
                assert (curve[['Cd','O2','qprod','qgas','predicted']].to_numpy()>=0).all()
                assert np.all(curve.qprod.to_numpy()[gate==0]==0) and gate[0]==0
                assert (np.diff(gate)>=-1e-12).all() and ((gate>=0)&(gate<=1)).all()
                baseline_error=np.nan
                if threshold=='2_6':
                    old=CA_CURVES[(lab,theta,case,'causal_2_6')]
                    baseline_error=float(np.max(abs(curve.to_numpy()-old[curve.columns].to_numpy())))
                    assert baseline_error<1e-12
                if previous is not None:
                    assert (gate<=previous.gate.to_numpy()+1e-12).all()
                    assert (curve.predicted<=previous.predicted+1e-10).all()
                previous=curve
                pulse_error=max(abs(curve.loc[t+diagnosis.CONFIG['after_event_h'],'N']-curve.loc[t,'N']-dose) for t,dose in batch.pulses['N'])
                assert pulse_error<1e-5
                TS_CURVES[(lab,theta,case,threshold)]=curve
                TS_CHECK_ROWS.append(dict(lab=lab,theta=theta,case=case,threshold=threshold,
                    start_h=0.,end_h=166.,pool_mass_error_g_l=curve.attrs['mass_error'],
                    chemistry_max_difference_g_l=0.,pulse_error_g_l=pulse_error,
                    baseline_2_6_max_difference=baseline_error))
                for cutoff in (12.,30.,60.,100.):
                    n=int(np.searchsorted(time,cutoff,side='right'));pg=ts_gate(consumed[:n],low,high)
                    changed=consumed.copy();changed[n:]=-100.
                    assert np.array_equal(pg,ts_gate(changed,low,high)[:n])
                    pr,pm=ca_pulse_history(time[:n],batch.pulses['N'])
                    prefix=gas_model.finish(batch,th,core.iloc[:n],core0.iloc[:n],CA_PARAMS,pg,pr,pm,0.,oxygen0)
                    columns=['gate','ramp','multiplier','O2','qprod','Cd','qgas','predicted']
                    err=float(np.max(abs(prefix[columns].to_numpy()-curve[columns].iloc[:n].to_numpy())))
                    assert err<1e-12
                    TS_PREFIX_ROWS.append(dict(lab=lab,theta=theta,case=case,threshold=threshold,cutoff_h=cutoff,max_prefix_error=err))
                predicted=ca_bin_prediction(curve,obs.bin_start_h)
                feature=ts_peak_fall(obs,predicted)
                onset=ts_onset(obs.bin_start_h,obs.bin_end_h,predicted)
                assert onset==ca_gas_confirmation(curve)
                control=CA_CURVES[(lab,theta,case,'sin_gate')]
                source_reduction=float(np.sum(np.diff(time)*(control.qprod.to_numpy()[:-1]-curve.qprod.to_numpy()[:-1])))
                TS_FEATURE_ROWS.append(dict(lab=lab,theta=theta,case=case,threshold=threshold,
                    low_g_l=low,high_g_l=high,gate_start_h=ca_crossing(time,consumed,low),
                    gate_complete_h=ca_crossing(time,consumed,high),
                    predicted_onset_h=onset,observed_onset_h=reference.onset_confirmation_h,
                    onset_error_h=onset-reference.onset_confirmation_h,
                    predicted_peak_h=feature['peak_h'],observed_peak_h=reference.peak_h,
                    peak_time_error_h=feature['peak_h']-reference.peak_h,
                    predicted_peak_g_l_h=feature['peak_g_l_h'],observed_peak_g_l_h=reference.peak_g_l_h,
                    peak_height_error_g_l_h=feature['peak_g_l_h']-reference.peak_g_l_h,
                    predicted_fall_h=feature['fall_confirmation_h'],observed_fall_h=reference.fall_confirmation_h,
                    fall_error_h=feature['fall_confirmation_h']-reference.fall_confirmation_h,
                    fall_not_confirmed=feature['fall_not_confirmed'],last_bin_end_h=feature['last_bin_end_h'],
                    source_reduction_vs_a1_g_l=source_reduction))
                residual=obs.copy();residual['predicted_g_l_h']=predicted
                residual['residual_g_l_h']=predicted-residual.co2_observed_g_l_h
                for name,value in dict(theta=theta,case=case,threshold=threshold).items(): residual[name]=value
                TS_RESIDUAL_PARTS.append(residual)
                peak_low=max(0.,reference.peak_h-20.);peak_high=min(166.,reference.peak_h+20.)
                windows=[('completo',np.ones(len(obs),bool)),
                    ('inicio',obs.bin_end_h<=reference.onset_confirmation_h),
                    ('maximo',obs.plot_t_h.between(peak_low,peak_high)),
                    ('caida',obs.plot_t_h>=reference.peak_h)]
                for window,keep in windows:
                    part=residual.loc[keep];r=part.residual_g_l_h.to_numpy()
                    assert len(r)>0
                    TS_METRIC_ROWS.append(dict(lab=lab,theta=theta,case=case,threshold=threshold,window=window,
                        n=len(r),RMSE_g_l_h=float(np.sqrt(np.mean(r*r))),MAE_g_l_h=float(np.mean(abs(r))),
                        bias_g_l_h=float(np.mean(r)),first_bin_start_h=float(part.bin_start_h.min()),
                        last_bin_end_h=float(part.bin_end_h.max())))
        print('Tres umbrales y ambas biomasas:',theta,lab,'; horizonte 0–166 h',flush=True)
TS_CHECKS=pd.DataFrame(TS_CHECK_ROWS);TS_PREFIX_CHECKS=pd.DataFrame(TS_PREFIX_ROWS)
TS_FEATURES=pd.DataFrame(TS_FEATURE_ROWS);TS_RESIDUALS=pd.concat(TS_RESIDUAL_PARTS,ignore_index=True)
TS_METRICS=pd.DataFrame(TS_METRIC_ROWS)
baseline=TS_METRICS[TS_METRICS.threshold.eq('2_6')][['lab','theta','case','window','RMSE_g_l_h']].rename(columns={'RMSE_g_l_h':'RMSE_2_6_g_l_h'})
TS_METRICS=TS_METRICS.merge(baseline,on=['lab','theta','case','window'],validate='many_to_one')
TS_METRICS['RMSE_change_percent']=100*(TS_METRICS.RMSE_g_l_h/TS_METRICS.RMSE_2_6_g_l_h-1)
assert len(TS_CURVES)==36 and len(TS_PREFIX_CHECKS)==144 and len(TS_METRICS)==144
print('Comprobados 36 escenarios y 144 prefijos; caso 2–6 idéntico a §24.')

### 25.2. Figuras nuevas: inicio, máximo y caída

Se añaden cuatro figuras, una por combinación de theta y biomasa, para no mezclar seis curvas en cada panel. **Columnas:** LAB019, LAB020 y LAB021. **Filas:** inicio (0–60 h), entorno del máximo observado (±20 h) y caída (desde ese máximo hasta 166 h). Las ampliaciones se usan para leer detalles; las simulaciones conservan siempre el proceso completo y los pulsos.

Los colores identifican únicamente las parejas de umbrales y se repiten en todas las figuras. Los puntos negros son el caudal medido. No se añaden marcadores especiales ni leyendas para cada estadístico. La tabla posterior cuantifica los desfases de inicio y de máximo, la caída final y los errores por ventana. «No confirmada» en la caída indica que la predicción no terminó de caer en el soporte observado; no significa que ese tiempo sea cero.

**En el máximo y en buena parte de la caída las tres curvas casi coinciden.** Se usan trazos sólido, discontinuo y punto-raya para hacer visible su superposición: no faltan escenarios en esos paneles.

In [ ]:
for theta in THETA_SETS:
    for case in CA_CASES:
        fig,axes=plt.subplots(3,3,figsize=(16,11),layout='constrained')
        for j,lab in enumerate(LABS):
            obs=GAS_HOURLY[GAS_HOURLY.lab.eq(lab)&GAS_HOURLY.valid]
            peak=float(TS_OBS_FEATURES.loc[lab,'peak_h'])
            windows=[('Inicio',(0.,60.)),('Máximo',(max(0.,peak-20.),min(166.,peak+20.))),('Caída',(peak,166.))]
            for i,(row_label,limits) in enumerate(windows):
                ax=axes[i,j]
                for threshold in TS_THRESHOLDS:
                    curve=TS_CURVES[(lab,theta,case,threshold)]
                    ax.plot(curve.index,curve.predicted,color=TS_COLORS[threshold],ls=TS_STYLES[threshold],lw=1.7,label=TS_LABELS[threshold])
                ax.plot(obs.plot_t_h,obs.co2_observed_g_l_h,'o',color='black',ms=2.5,label='CO₂ observado',zorder=6)
                mark_events(ax,lab)
                ax.set(title=lab+' · '+row_label,xlabel='t desde inoculación [h]',ylabel='g CO₂/L/h',xlim=limits,ylim=(0,None))
        fig.suptitle('Umbrales por G+F predicho · '+theta+' · '+W_CASES[case][0],y=.995)
        fig.get_layout_engine().set(rect=(0,0,1,.92))
        handles,labels=axes[0,0].get_legend_handles_labels()
        fig.legend(handles,labels,loc='upper center',bbox_to_anchor=(.5,.963),ncol=5,fontsize=9)
        ts_save(fig,'01_umbrales_'+theta+'_'+case)

for theta in THETA_SETS:
    display(Markdown('**'+theta+' — desfases y máximos calculados:**'))
    table=TS_FEATURES[TS_FEATURES.theta.eq(theta)][['lab','case','threshold','onset_error_h',
        'predicted_peak_g_l_h','observed_peak_g_l_h','peak_time_error_h','predicted_fall_h','observed_fall_h']].copy()
    table['case']=table['case'].map({k:v[0] for k,v in W_CASES.items()});table['threshold']=table['threshold'].map(TS_LABELS)
    table=table.round(4).rename(columns={'case':'Biomasa','threshold':'Umbrales','onset_error_h':'Δ inicio [h]',
        'predicted_peak_g_l_h':'Máximo pred. [g/L/h]','observed_peak_g_l_h':'Máximo obs. [g/L/h]',
        'peak_time_error_h':'Δ hora máximo [h]','predicted_fall_h':'Caída pred. [h]','observed_fall_h':'Caída obs. [h]'})
    table['Caída pred. [h]']=table['Caída pred. [h]'].astype(object).where(table['Caída pred. [h]'].notna(),'No confirmada hasta 160 h')
    display(table)
    display(TS_METRICS[TS_METRICS.theta.eq(theta)].pivot(index=['lab','case','threshold'],columns='window',values='RMSE_g_l_h').round(5))

### 25.3. Interpretación y resultados guardados

Los resultados siguientes comparan cada pareja contra 2–6 bajo la misma theta y biomasa. Un mejor inicio no basta para declarar mejor modelo: se comprueba también la altura, la hora del máximo y la caída. Un máximo parecido puede coexistir con una cola demasiado larga.

**LAB019:** con theta_full y viable común, 8–15 lleva la confirmación a 31 h frente a 30 h observadas; con theta_defaults y común, 5–10 la lleva a 29 h. Con ponderada todos los escenarios siguen adelantados, aunque 8–15 reduce el adelanto. El máximo y la caída apenas cambian: mejorar la confirmación de inicio no corrige la sobrepredicción posterior de full ni la cola prolongada de defaults.

**LAB020:** con full, común + 8–15 confirma a 35 h y ponderada + 5–10 a 33 h, frente a 34 h observadas. Con defaults, común + 5–10 confirma a 32 h; ponderada + 2–6 a 31 h y + 5–10 a 36 h. Por tanto, no hay una pareja que reproduzca igual el inicio bajo todas las inicializaciones. Ninguna pareja consigue confirmar la caída final antes de 160 h en este LAB.

**LAB021:** full + ponderada ya confirma correctamente a 21 h con 2–6; subir los umbrales la retrasa a 26/31 h. Full + común queda más cerca con 5–10 (22 h frente a 21 h); defaults + común queda más cerca con 2–6 (20 h). Defaults + ponderada ya empieza tarde con 2–6 y empeora al aumentar los límites. La caída tardía de full y el máximo bajo de defaults permanecen prácticamente intactos.

**Conclusión de la comparación:** los umbrales modifican principalmente el arranque, sin desplazar toda la curva. En estas pruebas la fuente y el reservorio hacen que las predicciones vuelvan a aproximarse después de completar la activación. La elección depende de theta y biomasa y no resuelve el error de la dinámica posterior. N, GF y E son exactamente los mismos para las tres parejas.

Si las curvas se superponen tras la activación, el umbral sólo modifica el arranque en estos ensayos. No se atribuye a la temperatura o a una biomasa «verdadera» un resultado que esta comparación no identifica. Tampoco se elige automáticamente un umbral óptimo: estas son pruebas sobre los mismos tres LAB, sin validación independiente. La eventual consistencia estequiométrica de la fuente gaseosa debe revisarse antes de usar el gate como mecanismo físico completo.

In [ ]:
TS_CONCLUSIONS=[]
for lab in LABS:
    TS_CONCLUSIONS.append('**'+lab+'** (confirmación observada '+f'{TS_OBS_FEATURES.loc[lab,"onset_confirmation_h"]:.0f} h):')
    for theta in THETA_SETS:
        for case in CA_CASES:
            sub=TS_FEATURES[(TS_FEATURES.lab==lab)&(TS_FEATURES.theta==theta)&(TS_FEATURES.case==case)]
            sub=sub.set_index('threshold').loc[list(TS_THRESHOLDS)]
            onset=' / '.join(f'{v:.0f}' for v in sub.predicted_onset_h)
            peaks=' / '.join(f'{v:.3f}' for v in sub.predicted_peak_g_l_h)
            falls=' / '.join('no confirmada ≤160 h' if pd.isna(v) else f'{v:.0f} h' for v in sub.predicted_fall_h)
            TS_CONCLUSIONS.append(f'- {theta}, {W_CASES[case][0]}: para 2–6 / 5–10 / 8–15, '
                f'inicio {onset} h; máximo {peaks} g/L/h (observado {sub.observed_peak_g_l_h.iloc[0]:.3f}); caída {falls}.')
            small=TS_CURVES[(lab,theta,case,'2_6')];large=TS_CURVES[(lab,theta,case,'8_15')]
            after=max(float(sub.gate_complete_h.max())+20.,float(TS_OBS_FEATURES.loc[lab,'peak_h']))
            keep=small.index>=after
            difference=float(np.max(abs(small.predicted[keep]-large.predicted[keep])))
            TS_CONCLUSIONS.append(f'  Diferencia máxima de caudal entre 2–6 y 8–15 desde {after:.1f} h: {difference:.5g} g/L/h.')
TS_CONCLUSIONS.append('La caída se compara sólo hasta 160 h porque ése es el soporte observado válido. '
    'Las curvas completas llegan a 166 h. Una caída no confirmada queda como NaN en CSV y no se reemplaza por cero. '
    'La fuente acumulada eliminada por cada gate frente a a=1 se guarda para revisar el alcance físico de la activación.')
display(Markdown('\n\n'.join(TS_CONCLUSIONS)))
assert TS_PRIOR_HASHES=={str(p.relative_to(ROOT)):sha256(p) for p in TS_PRIOR_FILES}
assert GAS_INPUT_HASHES=={str(p.relative_to(ROOT)):sha256(p) for p in GAS_INPUT_FILES}
tables=dict(observed_features=TS_OBS_FEATURES.reset_index(),features=TS_FEATURES,metrics=TS_METRICS,
    residuals=TS_RESIDUALS,checks=TS_CHECKS,prefix_checks=TS_PREFIX_CHECKS,figure_index=pd.DataFrame(TS_FIGURES))
for name,table in tables.items(): table.to_csv(TS_OUT/(name+'.csv'),index=False)
arrays={};array_manifest=[]
for i,(key,curve) in enumerate(TS_CURVES.items()):
    name=f'case_{i}';arrays[name]=np.column_stack([curve.index,curve.to_numpy()])
    array_manifest.append(dict(array=name,key=list(key),columns=['t_h']+list(curve.columns)))
np.savez_compressed(TS_OUT/'trajectories.npz',**arrays)
manifest=dict(created_at=datetime.now().astimezone().isoformat(),python=sys.executable,
    source_code_sha256=sha256(LAB2026/'notebooks/diagnostics/nitrogen_consumption_diagnosis.source.ipynb'),
    thresholds_g_l=TS_THRESHOLDS,co2_parameters=CA_PARAMS,theta=THETA_SETS,curves=len(TS_CURVES),
    arrays=array_manifest,figures=TS_FIGURES,input_hashes=GAS_INPUT_HASHES,prior_result_hashes=TS_PRIOR_HASHES,
    prefix_checks=len(TS_PREFIX_CHECKS),simulation_start_h=0.,simulation_end_h=166.,
    observed_end_h=float(TS_OBS_FEATURES.last_bin_end_h.max()),chemical_trajectories_unchanged=True,
    gas_initialized_from_measurement=False,thresholds_fitted=False,optimizer_calls=0,kalman_updates=0,
    physical_scope='CO2 gate only; chemical balances unchanged; CO2/ethanol source consistency unresolved',
    fall_definition='3 consecutive valid bins below detection after own peak, with no later valid re-crossing; retrospective',
    online_validation=False,temperature='retrospective measured input')
(TS_OUT/'manifest.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding='utf-8')
print('Añadidas cuatro figuras 3×3 debajo de las anteriores: 36 curvas, 144 prefijos y comparación inicio/máximo/caída.')
print('Química, parámetros numéricos antiguos y figuras previas conservados. Sin reajuste, EKF, commit ni push.')